# Global Market Signal Dashboard v5 — Trade Signals, Risk Scoring & Actionable Outlook

**Purpose**: Surface concrete, actionable trade signals from mechanical market data — no subjective regime labels, no opinions. Everything is auditable with sample sizes and confidence.

---

## Dashboard Architecture

| # | Section | What It Tells You | Trade Action |
|---|---------|-------------------|--------------|
| 1 | **Signal Scorecard** | Single-screen summary of all signals firing today | Prioritize which markets to trade and which direction |
| 2 | **Trend Signals** | Multi-timeframe momentum alignment + breakout proximity | Trend-following entries, mean-reversion setups |
| 3 | **Risk & Volatility Signals** | Drawdown severity, vol regime, RSI extremes | Position sizing, stop placement, vol-selling setups |
| 4 | **Breadth & Internals** | Market participation, thrust, McClellan, dispersion | Confirm/deny index-level moves, rotation trades |
| 5 | **Sector Rotation (RRG)** | Relative strength + momentum by sector | Sector pair trades, rotation timing |
| 6 | **Conditional Outlook** | Forward return distributions under current conditions | Expected value, tail risk, optimal horizon |
| 7 | **Cross-Market Heatmap** | All indices ranked on every metric | Global macro positioning, relative value |
| 8 | **Trade Idea Generator** | Automated signal combinations → trade suggestions | Direct trade ideas with entry/stop/target logic |
| 9 | **Data Quality** | Coverage, staleness, diagnostics | Trust your signals |

> **How to use**: Run all cells. Select a market from the dropdown. Every chart has an explanation box above it describing exactly what to look for and what constitutes a signal.


In [1]:
# ============================================================
# 1) IMPORTS
# ============================================================
import os, time, random, warnings
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional
from datetime import datetime

import numpy as np
import pandas as pd
import yfinance as yf

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import ipywidgets as widgets
from IPython.display import display, HTML

warnings.filterwarnings("ignore")
pd.options.display.max_columns = 200
pd.options.display.float_format = '{:.4f}'.format


## 2) Configuration

Edit `DEFAULT_INDICES` to add/remove markets. Member universes (S&P 500, NASDAQ-100) enable breadth, dispersion, and sector rotation tabs.


In [2]:
# ============================================================
# 2) CONFIGURATION
# ============================================================
@dataclass
class CFG:
    # --- Markets ---
    indices: Dict[str, str] = None
    sp500_constituents_url: str = "https://raw.githubusercontent.com/datasets/s-and-p-500-companies/master/data/constituents.csv"
    nasdaq100_constituents_url: str = "https://raw.githubusercontent.com/datasets/nasdaq-100-companies/master/data/constituents.csv"
    lookback: str = "5y"

    # --- Forward horizons for outlook engine ---
    forward_horizons: Tuple[int, ...] = (5, 10, 20, 60, 120)

    # --- Data fetching ---
    cache_dir: str = "./cache_signal_dashboard_v5"
    batch_size: int = 120
    pause_seconds: float = 0.6
    max_retries: int = 3

    # --- Indicator windows ---
    ma_windows: Tuple[int, ...] = (10, 20, 50, 100, 200)
    rsi_period: int = 14
    rv_window: int = 20
    rv_long_window: int = 60
    range_window: int = 252
    atr_window: int = 14

    # --- Breadth ---
    breadth_mom_window: int = 20
    dispersion_window: int = 20
    thrust_ema_span: int = 10
    mcclellan_fast: int = 19
    mcclellan_slow: int = 39

    # --- RRG ---
    rrg_ma: int = 14
    rrg_tail: int = 15

    # --- Signal thresholds (all mechanical, auditable) ---
    trend_stack_bullish: int = 4      # >= this many MAs above = strong uptrend
    trend_stack_bearish: int = 1      # <= this many MAs above = strong downtrend
    range_pos_breakout: float = 0.92  # near 52w high = breakout territory
    range_pos_breakdown: float = 0.10 # near 52w low = breakdown territory
    rsi_overbought: float = 70.0
    rsi_oversold: float = 30.0
    rv_high_pct: float = 0.80         # vol in top 20% of 2y = elevated
    rv_low_pct: float = 0.20          # vol in bottom 20% = compressed
    breadth_strong: float = 0.70      # >70% above 200DMA = healthy
    breadth_weak: float = 0.40        # <40% above 200DMA = deteriorating
    thrust_signal: float = 0.61       # Zweig breadth thrust level
    mcc_bullish: float = 50.0         # McClellan oscillator strongly positive
    mcc_bearish: float = -50.0
    max_missing_pct: float = 0.08

    # --- Signal scoring weights ---
    score_weights: Dict[str, float] = None

CFG = CFG()
CFG.indices = {
    "S&P 500": "^GSPC",
    "NASDAQ Composite": "^IXIC",
    "NASDAQ 100": "^NDX",
    "Dow 30": "^DJI",
    "Russell 2000": "^RUT",
    "KOSPI": "^KS11",
    "Nikkei 225": "^N225",
    "MSCI World": "^990100-USD-STRD",
}
CFG.score_weights = {
    "trend_stack": 2.0,
    "ma200_position": 1.5,
    "range_position": 1.0,
    "rsi_signal": 1.0,
    "vol_regime": 1.0,
    "drawdown": 1.5,
    "breadth": 2.0,
    "breadth_momentum": 1.5,
    "thrust": 2.0,
    "mcclellan": 1.0,
    "dispersion": 0.8,
}

os.makedirs(CFG.cache_dir, exist_ok=True)
print(f"Cache dir: {os.path.abspath(CFG.cache_dir)}")
print(f"Markets: {list(CFG.indices.keys())}")
print(f"Forward horizons: {CFG.forward_horizons}")


Cache dir: c:\Users\vigne\OneDrive - Singapore University of Technology and Design\Documents\QCP\Signal Dashboard - Equity Breadth - Copy\cache_signal_dashboard_v5
Markets: ['S&P 500', 'NASDAQ Composite', 'NASDAQ 100', 'Dow 30', 'Russell 2000', 'KOSPI', 'Nikkei 225', 'MSCI World']
Forward horizons: (5, 10, 20, 60, 120)


## 3) Data Manager (Batched, Cached, Incremental)

Downloads price data from Yahoo Finance with caching, batching, and retry logic. Parquet-based local cache keyed by ticker + lookback.


In [3]:
# ============================================================
# 3) DATA MANAGER
# ============================================================
def _safe_name(s: str) -> str:
    return "".join([c if c.isalnum() or c in ("-", "_") else "_" for c in str(s)])


def _parquet_engine_available() -> bool:
    try:
        import pyarrow  # noqa: F401
        return True
    except Exception:
        try:
            import fastparquet  # noqa: F401
            return True
        except Exception:
            return False


PARQUET_AVAILABLE = _parquet_engine_available()
CACHE_EXT = ".parquet" if PARQUET_AVAILABLE else ".pkl"


def _cache_file_candidates(key: str, lookback: str) -> List[str]:
    base = f"prices__{_safe_name(key)}__{_safe_name(lookback)}"
    return [
        os.path.join(CFG.cache_dir, base + ".parquet"),
        os.path.join(CFG.cache_dir, base + ".pkl"),
    ]


def cache_path_prices(key: str, lookback: str) -> str:
    return os.path.join(CFG.cache_dir, f"prices__{_safe_name(key)}__{_safe_name(lookback)}{CACHE_EXT}")


def _read_cached_prices(key: str, lookback: str) -> pd.DataFrame:
    for path in _cache_file_candidates(key, lookback):
        if not os.path.exists(path):
            continue
        try:
            if path.endswith(".parquet"):
                return pd.read_parquet(path)
            return pd.read_pickle(path)
        except Exception:
            continue
    return pd.DataFrame()


def _write_cached_prices(df: pd.DataFrame, key: str, lookback: str) -> None:
    path = cache_path_prices(key, lookback)
    try:
        if path.endswith(".parquet"):
            df.to_parquet(path)
        else:
            df.to_pickle(path)
    except Exception as exc:
        fallback_path = os.path.join(CFG.cache_dir, f"prices__{_safe_name(key)}__{_safe_name(lookback)}.pkl")
        df.to_pickle(fallback_path)
        print(f"Cache write fallback -> {fallback_path}: {exc}")


def _yf_download_close(tickers: List[str], lookback: str) -> pd.DataFrame:
    df = yf.download(
        tickers=tickers,
        period=lookback,
        progress=False,
        group_by="column",
        threads=True,
        auto_adjust=False,
    )
    if df is None or len(df) == 0:
        return pd.DataFrame()
    if isinstance(df.columns, pd.MultiIndex):
        close = df["Close"].copy()
    else:
        close = df[["Close"]].rename(columns={"Close": tickers[0]})
    close = close.sort_index()
    close.columns = [str(c).strip() for c in close.columns]
    return close


def download_close_batched(tickers: List[str], lookback: str) -> pd.DataFrame:
    tickers = list(dict.fromkeys([t for t in tickers if isinstance(t, str) and t.strip()]))
    if not tickers:
        return pd.DataFrame()
    chunks = [tickers[i:i + CFG.batch_size] for i in range(0, len(tickers), CFG.batch_size)]
    out = []
    for i, batch in enumerate(chunks, 1):
        for attempt in range(CFG.max_retries):
            try:
                part = _yf_download_close(batch, lookback)
                out.append(part)
                break
            except Exception as e:
                wait = CFG.pause_seconds * (2 ** attempt) + random.random()
                print(f"Batch {i}/{len(chunks)} failed ({attempt + 1}/{CFG.max_retries}): {e} -> sleep {wait:.1f}s")
                time.sleep(wait)
        time.sleep(CFG.pause_seconds)
    if not out:
        return pd.DataFrame()
    prices = pd.concat(out, axis=1).sort_index()
    prices = prices.loc[:, ~prices.columns.duplicated()]
    return prices


def load_or_download_index_close(ticker: str, lookback: str) -> pd.Series:
    df = _read_cached_prices(ticker, lookback)
    if not df.empty and ticker in df.columns:
        return df[ticker].ffill()

    prices = download_close_batched([ticker], lookback)
    if prices.empty or ticker not in prices.columns:
        empty = pd.DataFrame({ticker: pd.Series(dtype=float)})
        _write_cached_prices(empty, ticker, lookback)
        return pd.Series(dtype=float, name=ticker)

    _write_cached_prices(prices[[ticker]], ticker, lookback)
    return prices[ticker].ffill()


def load_or_download_universe_close(tickers: List[str], lookback: str, universe_key: str) -> pd.DataFrame:
    tickers = list(dict.fromkeys(tickers))
    df = _read_cached_prices(universe_key, lookback)
    if not df.empty:
        missing = [t for t in tickers if t not in df.columns]
        if missing:
            print(f"[cache top-up] {universe_key}: {len(missing)} missing tickers -> downloading")
            newp = download_close_batched(missing, lookback)
            df = pd.concat([df, newp], axis=1).sort_index()
            df = df.loc[:, ~df.columns.duplicated()]
            _write_cached_prices(df, universe_key, lookback)
        return df.ffill()

    prices = download_close_batched(tickers, lookback)
    _write_cached_prices(prices, universe_key, lookback)
    return prices.ffill()

## 4) Universe Loaders (S&P 500, NASDAQ-100)


In [4]:
# ============================================================



# 4) UNIVERSE LOADERS



# ============================================================



def _normalize_cols(df: pd.DataFrame) -> pd.DataFrame:



    df = df.copy()



    df.columns = [str(c).strip() for c in df.columns]



    return df











def _load_cached_csv(path: str) -> Optional[pd.DataFrame]:



    if os.path.exists(path):



        try:



            return _normalize_cols(pd.read_csv(path))



        except Exception:



            return None



    return None











def load_sp500_constituents(cache_name: str = "sp500_constituents.csv", force_refresh: bool = False) -> pd.DataFrame:



    local = os.path.join(CFG.cache_dir, cache_name)







    def ok(df):



        cols = set(df.columns)



        return ("Symbol" in cols) and (("Sector" in cols) or ("GICS Sector" in cols))







    if not force_refresh:



        df = _load_cached_csv(local)



        if df is not None and ok(df):



            df["Symbol"] = df["Symbol"].astype(str).str.replace(".", "-", regex=False).str.strip()



            if "Sector" not in df.columns and "GICS Sector" in df.columns:



                df["Sector"] = df["GICS Sector"]



            return df







    try:



        df = _normalize_cols(pd.read_csv(CFG.sp500_constituents_url))



        if "Symbol" in df.columns:



            df["Symbol"] = df["Symbol"].astype(str).str.replace(".", "-", regex=False).str.strip()



        if "Sector" not in df.columns and "GICS Sector" in df.columns:



            df["Sector"] = df["GICS Sector"]



        if ok(df):



            df.to_csv(local, index=False)



            return df



    except Exception as e:



        print("S&P 500 constituents fetch failed:", e)







    return pd.DataFrame(columns=["Symbol", "Name", "Sector"])











def load_nasdaq100_constituents(cache_name: str = "nasdaq100_constituents.csv", force_refresh: bool = False) -> pd.DataFrame:



    local = os.path.join(CFG.cache_dir, cache_name)



    urls = [



        CFG.nasdaq100_constituents_url,



        "https://en.wikipedia.org/wiki/Nasdaq-100",



    ]







    def ok(df):



        return "Symbol" in set(df.columns)







    if not force_refresh:



        df = _load_cached_csv(local)



        if df is not None and ok(df):



            df["Symbol"] = df["Symbol"].astype(str).str.replace(".", "-", regex=False).str.strip()



            return df







    for url in urls:



        try:



            if "wikipedia" in url.lower():



                import io



                import urllib.request







                req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})



                with urllib.request.urlopen(req, timeout=20) as resp:



                    html = resp.read().decode("utf-8", errors="ignore")



                tables = pd.read_html(io.StringIO(html))



                df = None



                for table in tables:



                    table = _normalize_cols(table)



                    rename_map = {}



                    if "Ticker" in table.columns and "Symbol" not in table.columns:



                        rename_map["Ticker"] = "Symbol"



                    if rename_map:



                        table = table.rename(columns=rename_map)



                    if ok(table):



                        df = table



                        break



                if df is None:



                    continue



            else:



                df = _normalize_cols(pd.read_csv(url))







            if "Symbol" in df.columns:



                df["Symbol"] = df["Symbol"].astype(str).str.replace(".", "-", regex=False).str.strip()



            if ok(df):



                df.to_csv(local, index=False)



                return df



        except Exception as e:



            print(f"NASDAQ-100 constituents fetch failed from {url}: {e}")







    return pd.DataFrame(columns=["Symbol", "Name"])











sp500_df = load_sp500_constituents(force_refresh=False)



nasdaq100_df = load_nasdaq100_constituents(force_refresh=False)



print(f"S&P 500 members: {len(sp500_df)} | NASDAQ-100 members: {len(nasdaq100_df)}")





def _read_html_tables(url: str) -> List[pd.DataFrame]:

    import io

    import urllib.request



    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})

    with urllib.request.urlopen(req, timeout=30) as resp:

        html = resp.read().decode("utf-8", errors="ignore")

    return [

        _normalize_cols(t) for t in pd.read_html(io.StringIO(html))

    ]





def _standardize_constituent_frame(df: pd.DataFrame, market_name: str, universe_type: str, source: str, note: str) -> pd.DataFrame:

    out = df.copy()

    rename_map = {

        "Ticker": "Symbol",

        "Company": "Name",

        "% Weight": "Weight",

        "Weight %": "Weight",

    }

    out = out.rename(columns={k: v for k, v in rename_map.items() if k in out.columns})

    if "Symbol" not in out.columns:

        out["Symbol"] = ""

    if "Name" not in out.columns:

        out["Name"] = ""

    if "Weight" not in out.columns:

        out["Weight"] = np.nan

    out["Symbol"] = out["Symbol"].astype(str).str.replace("KRX:", "", regex=False).str.replace("TYO:", "", regex=False).str.strip()

    out["Name"] = out["Name"].astype(str).str.strip()

    out["Market"] = market_name

    out["UniverseType"] = universe_type

    out["Source"] = source

    out["Note"] = note

    keep = [c for c in ["Symbol", "Name", "Weight", "Sector", "Industry", "Market", "UniverseType", "Source", "Note"] if c in out.columns]

    return out[keep].drop_duplicates().reset_index(drop=True)





def _load_cached_constituent_table(cache_name: str) -> Optional[pd.DataFrame]:

    local = os.path.join(CFG.cache_dir, cache_name)

    if os.path.exists(local):

        try:

            return _normalize_cols(pd.read_csv(local))

        except Exception:

            return None

    return None





def _save_constituent_table(df: pd.DataFrame, cache_name: str) -> pd.DataFrame:

    local = os.path.join(CFG.cache_dir, cache_name)

    df.to_csv(local, index=False)

    return df





def load_dow30_constituents(cache_name: str = "dow30_constituents.csv", force_refresh: bool = False) -> pd.DataFrame:

    cached = None if force_refresh else _load_cached_constituent_table(cache_name)

    if cached is not None and len(cached):

        return cached

    for table in _read_html_tables("https://en.wikipedia.org/wiki/Dow_Jones_Industrial_Average"):

        cols = set(table.columns)

        if {"Company", "Symbol"}.issubset(cols):

            df = _standardize_constituent_frame(

                table,

                market_name="Dow 30",

                universe_type="Exact constituents",

                source="Wikipedia DJIA components",

                note="Exact 30 current Dow Jones Industrial Average members.",

            )

            return _save_constituent_table(df, cache_name)

    return pd.DataFrame(columns=["Symbol", "Name", "Market", "UniverseType", "Source", "Note"])





def load_companiesmarketcap_holdings(url: str, market_name: str, cache_name: str, note: str, force_refresh: bool = False) -> pd.DataFrame:

    cached = None if force_refresh else _load_cached_constituent_table(cache_name)

    if cached is not None and len(cached):

        return cached

    for table in _read_html_tables(url):

        cols = set(table.columns)

        if ("Ticker" in cols or "Symbol" in cols) and ("Name" in cols or "Company" in cols):

            df = _standardize_constituent_frame(

                table,

                market_name=market_name,

                universe_type="Proxy holdings",

                source=url,

                note=note,

            )

            return _save_constituent_table(df, cache_name)

    return pd.DataFrame(columns=["Symbol", "Name", "Weight", "Market", "UniverseType", "Source", "Note"])





def load_market_constituents(market_name: str, force_refresh: bool = False) -> pd.DataFrame:

    if market_name == "S&P 500":

        return _standardize_constituent_frame(

            sp500_df,

            market_name=market_name,

            universe_type="Exact constituents",

            source=CFG.sp500_constituents_url,

            note="Exact S&P 500 constituent list.",

        )

    if market_name == "NASDAQ 100":

        return _standardize_constituent_frame(

            nasdaq100_df,

            market_name=market_name,

            universe_type="Exact constituents",

            source="NASDAQ-100 public constituent tables",

            note="Exact NASDAQ-100 constituent list.",

        )

    if market_name == "Dow 30":

        return load_dow30_constituents(force_refresh=force_refresh)

    proxy_map = {

        "NASDAQ Composite": {

            "url": "https://companiesmarketcap.com/fidelity-nasdaq-composite-index-etf/holdings/",

            "cache": "nasdaq_composite_proxy_holdings.csv",

            "note": "Proxy holdings using Fidelity Nasdaq Composite Index ETF (ONEQ), not the full Nasdaq Composite membership.",

        },

        "Russell 2000": {

            "url": "https://companiesmarketcap.com/ishares-russell-2000-etf/holdings/",

            "cache": "russell2000_proxy_holdings.csv",

            "note": "Proxy holdings using iShares Russell 2000 ETF (IWM).",

        },

        "KOSPI": {

            "url": "https://companiesmarketcap.com/ishares-msci-south-korea-etf/holdings/",

            "cache": "kospi_proxy_holdings.csv",

            "note": "Proxy holdings using iShares MSCI South Korea ETF (EWY), not the full KOSPI composite membership.",

        },

        "Nikkei 225": {

            "url": "https://companiesmarketcap.com/ishares-msci-japan-etf/holdings/",

            "cache": "nikkei225_proxy_holdings.csv",

            "note": "Proxy holdings using iShares MSCI Japan ETF (EWJ), not the exact Nikkei 225 membership.",

        },

        "MSCI World": {

            "url": "https://companiesmarketcap.com/ishares-msci-world-etf/holdings/",

            "cache": "msci_world_proxy_holdings.csv",

            "note": "Proxy holdings using iShares MSCI World ETF (URTH).",

        },

    }

    if market_name in proxy_map:

        meta = proxy_map[market_name]

        return load_companiesmarketcap_holdings(meta["url"], market_name, meta["cache"], meta["note"], force_refresh=force_refresh)

    return pd.DataFrame(columns=["Symbol", "Name", "Weight", "Market", "UniverseType", "Source", "Note"])


def yahoo_symbol_for_market(symbol: str, market_name: str) -> str:
    sym = str(symbol).strip().replace("/", "-")
    if not sym:
        return sym
    if market_name == "KOSPI":
        return sym if ".KS" in sym else f"{sym}.KS"
    if market_name == "Nikkei 225":
        return sym if ".T" in sym else f"{sym}.T"
    return sym


def prepare_market_universe_symbols(constituents_df: pd.DataFrame, market_name: str) -> List[str]:
    if constituents_df is None or constituents_df.empty or "Symbol" not in constituents_df.columns:
        return []
    symbols = [yahoo_symbol_for_market(s, market_name) for s in constituents_df["Symbol"].tolist()]
    symbols = [s for s in symbols if isinstance(s, str) and s.strip()]
    return list(dict.fromkeys(symbols))


def market_universe_download_enabled(market_name: str) -> bool:
    return market_name in {
        "S&P 500",
        "NASDAQ 100",
        "NASDAQ Composite",
        "Dow 30",
        "Russell 2000",
        "KOSPI",
        "Nikkei 225",
    }


CFG.breadth_member_limits = getattr(CFG, "breadth_member_limits", {
    "S&P 500": 503,
    "NASDAQ 100": 101,
    "Dow 30": 30,
    "NASDAQ Composite": 300,
    "Russell 2000": 400,
    "KOSPI": 120,
    "Nikkei 225": 225,
})
CFG.member_idea_limits = getattr(CFG, "member_idea_limits", {
    "S&P 500": 120,
    "NASDAQ 100": 100,
    "Dow 30": 30,
    "NASDAQ Composite": 120,
    "Russell 2000": 150,
    "KOSPI": 85,
    "Nikkei 225": 120,
})


def parse_weight_value(value) -> float:
    if pd.isna(value):
        return np.nan
    s = str(value).replace('%', '').replace(',', '').strip()
    try:
        return float(s)
    except Exception:
        return np.nan


def select_constituent_subset(df: pd.DataFrame, market_name: str, limit_map: Dict[str, int]) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame()
    limit = int(limit_map.get(market_name, len(df)))
    out = df.copy()
    if 'Weight' in out.columns:
        out['_weight_num'] = out['Weight'].apply(parse_weight_value)
        out = out.sort_values(['_weight_num', 'Symbol'], ascending=[False, True], na_position='last')
    return out.head(limit).drop(columns=[c for c in ['_weight_num'] if c in out.columns]).reset_index(drop=True)

S&P 500 members: 503 | NASDAQ-100 members: 101


## 5) Indicators & Signal Engine

**Every indicator produces a raw value AND a signal interpretation:**
- Raw value = the metric itself (e.g., RSI = 72.3)
- Percentile = where this value sits in the last 2 years (cross-market comparable)
- Signal = mechanical label (bullish/bearish/neutral) based on thresholds in CFG

This section also computes **ATR** (for stop-loss placement), **rate of change** (momentum confirmation), and **vol term structure** (short vs long vol).


In [5]:
# ============================================================
# 5) INDICATORS & SIGNAL ENGINE
# ============================================================

# --- Core indicator functions ---
def rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    up = delta.clip(lower=0)
    down = -delta.clip(upper=0)
    ma_up = up.ewm(alpha=1/period, adjust=False).mean()
    ma_down = down.ewm(alpha=1/period, adjust=False).mean()
    rs = ma_up / ma_down.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

def realized_vol(ret: pd.Series, window: int = 20) -> pd.Series:
    return ret.rolling(window).std() * np.sqrt(252)

def atr(close: pd.Series, window: int = 14) -> pd.Series:
    """Average True Range — used for stop placement and position sizing."""
    high = close.rolling(2).max()  # approximate with close-only data
    low = close.rolling(2).min()
    tr = pd.concat([high - low, (close - close.shift(1)).abs()], axis=1).max(axis=1)
    return tr.rolling(window).mean()

def rate_of_change(close: pd.Series, periods: List[int] = [5, 20, 60]) -> Dict[str, pd.Series]:
    """Multi-period rate of change for momentum confirmation."""
    out = {}
    for p in periods:
        out[f"roc_{p}d"] = close.pct_change(p)
    return out

def rolling_percentile(s: pd.Series, window: int = 252*2) -> pd.Series:
    s = s.astype(float)
    def pct(x):
        x = pd.Series(x).dropna()
        if len(x) < 30:
            return np.nan
        last = x.iloc[-1]
        return float((x <= last).mean())
    return s.rolling(window, min_periods=60).apply(pct, raw=False)

def ma_slope(close: pd.Series, window: int, slope_period: int = 10) -> pd.Series:
    """Slope of a moving average — positive = trending up, negative = trending down."""
    ma = close.rolling(window, min_periods=window).mean()
    return ma.pct_change(slope_period)

# --- Master feature builder ---
def compute_index_features(close: pd.Series) -> pd.DataFrame:
    close = close.dropna().copy()
    if close.empty:
        return pd.DataFrame()

    ret = close.pct_change()
    dd = close / close.cummax() - 1.0
    rv_short = realized_vol(ret, CFG.rv_window)
    rv_long = realized_vol(ret, CFG.rv_long_window)
    rsi14 = rsi(close, CFG.rsi_period)
    atr14 = atr(close, CFG.atr_window)

    # Moving averages and distances
    mas = {}
    dist = {}
    slopes = {}
    for w in CFG.ma_windows:
        ma = close.rolling(w, min_periods=w).mean()
        mas[f"ma{w}"] = ma
        dist[f"dist_ma{w}"] = close / ma - 1.0
        slopes[f"slope_ma{w}"] = ma_slope(close, w)

    # 52-week range
    hi = close.rolling(CFG.range_window, min_periods=CFG.range_window).max()
    lo = close.rolling(CFG.range_window, min_periods=CFG.range_window).min()
    range_pos = (close - lo) / (hi - lo).replace(0, np.nan)
    dist_to_hi = close / hi - 1.0
    dist_to_lo = close / lo - 1.0

    # Rate of change
    rocs = rate_of_change(close, [5, 20, 60])

    # Vol term structure: short vol / long vol
    vol_term = rv_short / rv_long.replace(0, np.nan)

    # Build DataFrame
    df = pd.DataFrame({
        "close": close,
        "ret1d": ret,
        "drawdown": dd,
        "rv20_ann": rv_short,
        "rv60_ann": rv_long,
        "vol_term_structure": vol_term,
        "rsi14": rsi14,
        "atr14": atr14,
        "range_pos": range_pos,
        "dist_52w_high": dist_to_hi,
        "dist_52w_low": dist_to_lo,
    })
    for k, v in mas.items():
        df[k] = v
    for k, v in dist.items():
        df[k] = v
    for k, v in slopes.items():
        df[k] = v
    for k, v in rocs.items():
        df[k] = v

    # Percentiles (2y rolling)
    df["rv20_pct"] = rolling_percentile(df["rv20_ann"], window=252*2)
    df["dd_pct"] = rolling_percentile(df["drawdown"], window=252*2)
    df["range_pos_pct"] = rolling_percentile(df["range_pos"], window=252*2)
    df["rsi_pct"] = rolling_percentile(df["rsi14"], window=252*2)
    df["vol_term_pct"] = rolling_percentile(df["vol_term_structure"], window=252*2)
    for w in CFG.ma_windows:
        df[f"dist_ma{w}_pct"] = rolling_percentile(df[f"dist_ma{w}"], window=252*2)

    return df

# --- Breadth & Dispersion ---
def compute_universe_breadth_dispersion(stocks_close: pd.DataFrame) -> Dict[str, pd.Series]:
    stocks = stocks_close.loc[:, ~stocks_close.columns.duplicated()].copy()
    if stocks.empty:
        return {}
    ret_1d = stocks.pct_change()

    ma50 = stocks.rolling(50, min_periods=50).mean()
    ma200 = stocks.rolling(200, min_periods=200).mean()
    v50 = ma50.notna()
    v200 = ma200.notna()

    pct_above_50 = ((stocks > ma50) & v50).sum(axis=1) / v50.sum(axis=1).replace(0, np.nan)
    pct_above_200 = ((stocks > ma200) & v200).sum(axis=1) / v200.sum(axis=1).replace(0, np.nan)
    breadth_mom_200 = pct_above_200 - pct_above_200.shift(CFG.breadth_mom_window)

    # New 52w high/low counts
    hi_252 = stocks.rolling(252, min_periods=252).max()
    lo_252 = stocks.rolling(252, min_periods=252).min()
    at_high = ((stocks >= hi_252 * 0.98) & hi_252.notna()).sum(axis=1)
    at_low = ((stocks <= lo_252 * 1.02) & lo_252.notna()).sum(axis=1)
    n_valid = hi_252.notna().sum(axis=1).replace(0, np.nan)
    pct_near_52w_high = at_high / n_valid
    pct_near_52w_low = at_low / n_valid

    disp = ret_1d.std(axis=1)
    disp_roll = disp.rolling(CFG.dispersion_window).mean()

    # Correlation (avg pairwise) — high = herding, low = dispersion
    corr_roll = ret_1d.rolling(60).corr()
    # Mean pairwise corr per day (expensive but informative)
    avg_corr = pd.Series(index=stocks.index, dtype=float)
    # Use a faster approach: average of cross-sectional correlations
    for dt in stocks.index[-252:]:
        window = ret_1d.loc[:dt].tail(60)
        if len(window) >= 40 and window.shape[1] >= 10:
            cm = window.corr()
            mask = np.triu(np.ones(cm.shape, dtype=bool), k=1)
            avg_corr[dt] = cm.values[mask].mean()

    pct_above_200_pct = rolling_percentile(pct_above_200, window=252*2)
    breadth_mom_pct = rolling_percentile(breadth_mom_200, window=252*2)
    disp_pct = rolling_percentile(disp_roll, window=252*2)

    return {
        "pct_above_50": pct_above_50,
        "pct_above_200": pct_above_200,
        "breadth_mom_200": breadth_mom_200,
        "pct_near_52w_high": pct_near_52w_high,
        "pct_near_52w_low": pct_near_52w_low,
        "dispersion": disp,
        "dispersion_roll": disp_roll,
        "avg_corr": avg_corr,
        "pct_above_200_pct": pct_above_200_pct,
        "breadth_mom_pct": breadth_mom_pct,
        "dispersion_pct": disp_pct,
    }


## 6) S&P 500 Breadth Thrust, McClellan & Sector Rotation (RRG)

- **Breadth Thrust**: Zweig-style breadth thrust (10D EMA of advance ratio). Readings above 0.615 are historically rare and bullish.
- **McClellan Oscillator**: Fast EMA(19) minus Slow EMA(39) of net advances. Positive = breadth expanding, negative = contracting.
- **RRG**: Relative Rotation Graph — plots each sector's relative strength and momentum vs the index. Sectors rotate clockwise through Leading → Weakening → Lagging → Improving.


In [6]:
# ============================================================
# 6) S&P 500 EXTRAS
# ============================================================
def compute_sp500_breadth_thrust_mcclellan(stocks_close: pd.DataFrame) -> pd.DataFrame:
    stocks = stocks_close.copy()
    if stocks.empty:
        return pd.DataFrame()
    ret_1d = stocks.pct_change(1)
    adv = (ret_1d > 0).sum(axis=1)
    dec = (ret_1d < 0).sum(axis=1)
    tot = (adv + dec).replace(0, np.nan)
    ad_ratio = adv / tot
    thrust = ad_ratio.ewm(span=CFG.thrust_ema_span, adjust=False).mean()

    net_adv = (adv - dec).astype(float)
    mcc_fast = net_adv.ewm(span=CFG.mcclellan_fast, adjust=False).mean()
    mcc_slow = net_adv.ewm(span=CFG.mcclellan_slow, adjust=False).mean()
    mcc = mcc_fast - mcc_slow

    # Advance-Decline Line (cumulative)
    ad_line = net_adv.cumsum()

    # Up Volume proxy (count-based since we don't have volume)
    up_pct = adv / tot

    out = pd.DataFrame({
        "breadth_thrust": thrust,
        "mcclellan": mcc,
        "ad_line": ad_line,
        "adv_ratio": ad_ratio,
        "up_pct_ema": up_pct.ewm(span=10).mean(),
    })
    out["thrust_pct"] = rolling_percentile(out["breadth_thrust"], window=252*2)
    out["mcc_pct"] = rolling_percentile(out["mcclellan"], window=252*2)
    return out

def compute_sp500_rrg(stocks_close: pd.DataFrame, sector_map: Dict[str, str],
                       bench_close: pd.Series) -> pd.DataFrame:
    stocks = stocks_close.copy()
    if stocks.empty:
        return pd.DataFrame()
    ret_1d = stocks.pct_change(1)
    sector_members: Dict[str, List[str]] = {}
    for t in stocks.columns:
        sec = sector_map.get(t, "Unknown")
        if sec == "Unknown":
            continue
        sector_members.setdefault(sec, []).append(t)
    sector_ret = {}
    for sec, mem in sector_members.items():
        sector_ret[sec] = ret_1d[mem].mean(axis=1)
    sector_ret = pd.DataFrame(sector_ret).dropna(how="all")
    bench_ret = bench_close.pct_change(1).reindex(sector_ret.index).fillna(0.0)
    rows = []
    for sec in sector_ret.columns:
        rel_perf = (1 + sector_ret[sec].fillna(0.0)).cumprod() / (1 + bench_ret).cumprod()
        rs_ratio = 100 * (rel_perf / rel_perf.rolling(CFG.rrg_ma, min_periods=CFG.rrg_ma).mean())
        rs_mom = 100 * (rs_ratio / rs_ratio.rolling(CFG.rrg_ma, min_periods=CFG.rrg_ma).mean())
        rs_ratio = rs_ratio.dropna()
        rs_mom = rs_mom.reindex(rs_ratio.index).dropna()
        if len(rs_ratio) < CFG.rrg_tail or len(rs_mom) < CFG.rrg_tail:
            continue
        x_tail = rs_ratio.tail(CFG.rrg_tail).tolist()
        y_tail = rs_mom.tail(CFG.rrg_tail).tolist()
        x = float(rs_ratio.iloc[-1])
        y = float(rs_mom.iloc[-1])
        quad = ("Leading" if (x>=100 and y>=100)
                else "Weakening" if (x>=100 and y<100)
                else "Improving" if (x<100 and y>=100)
                else "Lagging")

        # Compute recent return for context
        sec_cum = (1 + sector_ret[sec].fillna(0)).cumprod()
        ret_20d = float(sec_cum.iloc[-1] / sec_cum.iloc[-min(20, len(sec_cum))] - 1) if len(sec_cum) > 1 else np.nan

        rows.append({
            "Sector": sec,
            "Members": len(sector_members.get(sec, [])),
            "RS_Ratio": x,
            "RS_Momentum": y,
            "Quadrant": quad,
            "Ret_20D": ret_20d,
            "Tail_X": x_tail,
            "Tail_Y": y_tail,
        })
    return pd.DataFrame(rows)


## 7) Outlook Engine — Conditional Distributions, Tail Risk, MAE/MFE

**How this works**: For any condition (e.g., "trend stack >= 4/5 AND low vol"), the engine:
1. Finds all historical dates where that condition was true
2. Computes the forward return distribution at each horizon (5d, 10d, 20d, 60d, 120d)
3. Reports quantiles (p10–p90), win rate, tail probabilities, Expected Shortfall, and path extremes (MAE = max adverse excursion, MFE = max favorable excursion)

**Trade use**: If the median 20d forward return under your condition is +1.8% with a win rate of 68% and ES(5%) of -4.2%, that's a concrete edge with quantified downside.


In [7]:
# ============================================================
# 7) OUTLOOK ENGINE
# ============================================================
def forward_returns(close: pd.Series, horizon: int) -> pd.Series:
    close = close.dropna()
    return close.shift(-horizon) / close - 1.0

def rolling_path_extremes(close: pd.Series, horizon: int) -> Tuple[pd.Series, pd.Series]:
    c = close.dropna()
    idx = c.index
    arr = c.values
    mae = np.full(len(arr), np.nan)
    mfe = np.full(len(arr), np.nan)
    for i in range(len(arr) - horizon):
        future = arr[i+1:i+horizon+1]
        mae[i] = (np.min(future) / arr[i]) - 1.0
        mfe[i] = (np.max(future) / arr[i]) - 1.0
    return pd.Series(mae, index=idx), pd.Series(mfe, index=idx)

def expected_shortfall(x: pd.Series, alpha: float = 0.05) -> float:
    x = x.dropna()
    if len(x) == 0:
        return np.nan
    q = x.quantile(alpha)
    tail = x[x <= q]
    return float(tail.mean()) if len(tail) else np.nan

def summarize_distribution(x: pd.Series) -> Dict[str, float]:
    x = x.dropna()
    if len(x) == 0:
        return {k: np.nan for k in ["n","mean","median","std","p5","p10","p25","p75","p90","p95","p_gt0","p_lt_5","p_lt_10","es5","skew"]}
    return {
        "n": float(len(x)),
        "mean": float(x.mean()),
        "median": float(x.median()),
        "std": float(x.std()),
        "p5": float(x.quantile(0.05)),
        "p10": float(x.quantile(0.10)),
        "p25": float(x.quantile(0.25)),
        "p75": float(x.quantile(0.75)),
        "p90": float(x.quantile(0.90)),
        "p95": float(x.quantile(0.95)),
        "p_gt0": float((x > 0).mean()),
        "p_lt_5": float((x < -0.05).mean()),
        "p_lt_10": float((x < -0.10).mean()),
        "es5": float(expected_shortfall(x, 0.05)),
        "skew": float(x.skew()),
    }

def outlook_table(close: pd.Series, mask: pd.Series, horizons: Tuple[int, ...]) -> pd.DataFrame:
    close = close.dropna()
    mask = mask.reindex(close.index).fillna(False)
    rows = []
    for h in horizons:
        fwd = forward_returns(close, h)
        cond = fwd[mask].dropna()
        uncond = fwd.dropna()
        s1 = summarize_distribution(cond)
        s0 = summarize_distribution(uncond)
        mae, mfe = rolling_path_extremes(close, h)
        mae_c = mae[mask].dropna()
        mfe_c = mfe[mask].dropna()
        rows.append({
            "Horizon": f"{h}D",
            "N": int(s1["n"]),
            "Median%": s1["median"]*100,
            "Mean%": s1["mean"]*100,
            "Std%": s1["std"]*100,
            "Win%": s1["p_gt0"]*100,
            "P10%": s1["p10"]*100,
            "P25%": s1["p25"]*100,
            "P75%": s1["p75"]*100,
            "P90%": s1["p90"]*100,
            "P(<-5%)": s1["p_lt_5"]*100,
            "P(<-10%)": s1["p_lt_10"]*100,
            "ES(5%)": s1["es5"]*100,
            "Skew": s1["skew"],
            "MAE_Med%": float(mae_c.median())*100 if len(mae_c) else np.nan,
            "MFE_Med%": float(mfe_c.median())*100 if len(mfe_c) else np.nan,
            "Uncond_Mean%": s0["mean"]*100,
            "Edge_Mean%": (s1["mean"] - s0["mean"])*100,
            "Edge_Win%": (s1["p_gt0"] - s0["p_gt0"])*100,
        })
    return pd.DataFrame(rows)

def quantile_fan(close: pd.Series, mask: pd.Series, horizons: Tuple[int, ...]) -> pd.DataFrame:
    close = close.dropna()
    mask = mask.reindex(close.index).fillna(False)
    rows = []
    for h in horizons:
        fwd = forward_returns(close, h)[mask].dropna()
        if len(fwd) == 0:
            rows.append({"horizon": h, "p5": np.nan, "p10": np.nan, "p25": np.nan, "p50": np.nan, "p75": np.nan, "p90": np.nan, "p95": np.nan})
            continue
        rows.append({
            "horizon": h,
            "p5": float(fwd.quantile(0.05)),
            "p10": float(fwd.quantile(0.10)),
            "p25": float(fwd.quantile(0.25)),
            "p50": float(fwd.quantile(0.50)),
            "p75": float(fwd.quantile(0.75)),
            "p90": float(fwd.quantile(0.90)),
            "p95": float(fwd.quantile(0.95)),
        })
    return pd.DataFrame(rows)


## 8) Signal Scoring Engine

Each signal is scored from **-1 (strongly bearish)** to **+1 (strongly bullish)** with 0 = neutral. Signals are then weighted and aggregated into a composite score. The composite score ranges from -10 to +10.

**Signal interpretation guide:**
| Score Range | Interpretation | Suggested Posture |
|-------------|---------------|-------------------|
| +6 to +10 | Strong bullish alignment | Full long, add on dips |
| +3 to +6 | Moderate bullish | Lean long, tighter stops |
| -3 to +3 | Mixed/neutral | Reduce size, wait for clarity |
| -6 to -3 | Moderate bearish | Lean short or defensive |
| -10 to -6 | Strong bearish alignment | Shorts, hedges, raise cash |


In [8]:
# ============================================================
# 8) SIGNAL SCORING ENGINE
# ============================================================
def score_signal(value: float, bullish_above: float, bearish_below: float,
                 max_bull: float = None, max_bear: float = None) -> float:
    """Score a directional signal from -1 to +1 with linear interpolation."""
    if np.isnan(value):
        return 0.0

    if bullish_above <= bearish_below:
        mid = (bullish_above + bearish_below) / 2
        width = max((bearish_below - bullish_above) / 2, 1e-9)
        if value >= bearish_below:
            if max_bull is None or max_bull == bearish_below:
                return 1.0
            return min(1.0, 0.5 + 0.5 * (value - bearish_below) / (max_bull - bearish_below))
        if value <= bullish_above:
            if max_bear is None or max_bear == bullish_above:
                return -1.0
            return max(-1.0, -0.5 - 0.5 * (bullish_above - value) / (bullish_above - max_bear))
        return (value - mid) / width * 0.5

    if value >= bullish_above:
        if max_bull is None or max_bull == bullish_above:
            return 1.0
        return min(1.0, 0.5 + 0.5 * (value - bullish_above) / (max_bull - bullish_above))
    if value <= bearish_below:
        if max_bear is None or max_bear == bearish_below:
            return -1.0
        return max(-1.0, -0.5 - 0.5 * (bearish_below - value) / (bearish_below - max_bear))

    mid = (bullish_above + bearish_below) / 2
    half_range = max((bullish_above - bearish_below) / 2, 1e-9)
    return (value - mid) / half_range * 0.5


def score_signal_inverted(value: float, bullish_below: float, bearish_above: float,
                          floor_value: float = None, ceiling_value: float = None) -> float:
    """Lower values are bullish, higher values are bearish."""
    return -score_signal(
        value,
        bullish_above=bearish_above,
        bearish_below=bullish_below,
        max_bull=ceiling_value,
        max_bear=floor_value,
    )


def compute_signal_scores(state: Dict) -> pd.DataFrame:
    """Compute all individual signal scores and the composite."""
    df = state["index_feat"]
    if df.empty:
        return pd.DataFrame()

    last = df.iloc[-1]
    n_ma = len(CFG.ma_windows)
    stack = int(sum(1 for w in CFG.ma_windows if last.get(f"dist_ma{w}", -1) > 0))

    signals = {}

    signals["trend_stack"] = {
        "raw": f"{stack}/{n_ma}",
        "score": score_signal(stack, CFG.trend_stack_bullish, CFG.trend_stack_bearish, n_ma, 0),
        "label": "BULLISH" if stack >= CFG.trend_stack_bullish else "BEARISH" if stack <= CFG.trend_stack_bearish else "MIXED",
        "detail": f"Price above {stack} of {n_ma} MAs (10/20/50/100/200)",
    }

    d200 = last.get("dist_ma200", 0) * 100
    signals["ma200_position"] = {
        "raw": f"{d200:+.1f}%",
        "score": score_signal(d200, 2.0, -2.0, 10.0, -10.0),
        "label": "ABOVE" if d200 > 2 else "BELOW" if d200 < -2 else "AT",
        "detail": f"Close is {d200:+.1f}% from 200DMA",
    }

    slope200 = last.get("slope_ma200", 0)
    signals["ma200_slope"] = {
        "raw": f"{slope200 * 100:+.2f}%",
        "score": score_signal(slope200, 0.005, -0.005, 0.03, -0.03),
        "label": "RISING" if slope200 > 0.005 else "FALLING" if slope200 < -0.005 else "FLAT",
        "detail": f"200DMA slope over 10 periods: {slope200 * 100:+.2f}%",
    }

    rp = last.get("range_pos", 0.5)
    signals["range_position"] = {
        "raw": f"{rp:.2f}",
        "score": score_signal(rp, 0.7, 0.3, 1.0, 0.0),
        "label": "BREAKOUT" if rp >= CFG.range_pos_breakout else "BREAKDOWN" if rp <= CFG.range_pos_breakdown else "MID-RANGE",
        "detail": f"52w range position: {rp:.2f} (1=high, 0=low)",
    }

    r = last.get("rsi14", 50)
    if r >= CFG.rsi_overbought:
        rsi_score = -0.5 * min(1.0, (r - CFG.rsi_overbought) / 15)
    elif r <= CFG.rsi_oversold:
        rsi_score = 0.5 * min(1.0, (CFG.rsi_oversold - r) / 15)
    else:
        rsi_score = score_signal(r, 55, 45, 70, 30) * 0.3
    signals["rsi_signal"] = {
        "raw": f"{r:.1f}",
        "score": rsi_score,
        "label": "OVERBOUGHT" if r >= CFG.rsi_overbought else "OVERSOLD" if r <= CFG.rsi_oversold else "NEUTRAL",
        "detail": f"RSI(14): {r:.1f}",
    }

    rv_pct = last.get("rv20_pct", 0.5)
    rv_ann = last.get("rv20_ann", 0) * 100
    signals["vol_regime"] = {
        "raw": f"{rv_ann:.1f}% (p{rv_pct * 100:.0f})",
        "score": score_signal_inverted(rv_pct, bullish_below=0.20, bearish_above=0.80, floor_value=0.0, ceiling_value=1.0),
        "label": "ELEVATED" if rv_pct >= CFG.rv_high_pct else "COMPRESSED" if rv_pct <= CFG.rv_low_pct else "NORMAL",
        "detail": f"RV20 annualized: {rv_ann:.1f}%, percentile: {rv_pct * 100:.0f}th",
    }

    vt = last.get("vol_term_structure", 1.0)
    signals["vol_term"] = {
        "raw": f"{vt:.2f}",
        "score": score_signal_inverted(vt, bullish_below=0.85, bearish_above=1.20, floor_value=0.50, ceiling_value=2.00),
        "label": "BACKWARDATION" if vt > 1.2 else "CONTANGO" if vt < 0.85 else "NORMAL",
        "detail": f"Short/long vol ratio: {vt:.2f} (>1.2 = stress spike, <0.85 = calm)",
    }

    dd = last.get("drawdown", 0) * 100
    dd_pct = last.get("dd_pct", 0.5)
    signals["drawdown"] = {
        "raw": f"{dd:+.1f}%",
        "score": score_signal(dd, -1.0, -10.0, 0.0, -25.0),
        "label": "SEVERE" if dd < -10 else "MODERATE" if dd < -5 else "SHALLOW" if dd < -1 else "NONE",
        "detail": f"Drawdown: {dd:+.1f}% from peak (pctile: {dd_pct * 100:.0f}th)",
    }

    roc20 = last.get("roc_20d", 0) * 100
    signals["momentum_20d"] = {
        "raw": f"{roc20:+.1f}%",
        "score": score_signal(roc20, 2.0, -2.0, 8.0, -8.0),
        "label": "STRONG" if roc20 > 3 else "WEAK" if roc20 < -3 else "FLAT",
        "detail": f"20-day rate of change: {roc20:+.1f}%",
    }

    memb = state.get("members_feat", {})
    if memb:
        pct200 = memb.get("pct_above_200")
        if pct200 is not None and len(pct200) > 0:
            b = float(pct200.iloc[-1])
            signals["breadth"] = {
                "raw": f"{b * 100:.1f}%",
                "score": score_signal(b, CFG.breadth_strong, CFG.breadth_weak, 0.90, 0.20),
                "label": "STRONG" if b >= CFG.breadth_strong else "WEAK" if b <= CFG.breadth_weak else "MODERATE",
                "detail": f"{b * 100:.1f}% of members above 200DMA",
            }
        bm = memb.get("breadth_mom_200")
        if bm is not None and len(bm) > 0:
            bv = float(bm.iloc[-1]) * 100
            signals["breadth_momentum"] = {
                "raw": f"{bv:+.1f}pp",
                "score": score_signal(bv, 3.0, -3.0, 10.0, -10.0),
                "label": "IMPROVING" if bv > 3 else "DETERIORATING" if bv < -3 else "STABLE",
                "detail": f"Change in %>200DMA over 20 days: {bv:+.1f}pp",
            }

    thrust_df = state.get("sp500_thrust_mcc", pd.DataFrame())
    if not thrust_df.empty:
        t = float(thrust_df["breadth_thrust"].iloc[-1])
        signals["thrust"] = {
            "raw": f"{t:.3f}",
            "score": score_signal(t, CFG.thrust_signal, 0.40, 0.70, 0.30),
            "label": "THRUST" if t >= CFG.thrust_signal else "WEAK" if t < 0.40 else "NORMAL",
            "detail": f"Breadth thrust (10D EMA of adv ratio): {t:.3f}",
        }
        m = float(thrust_df["mcclellan"].iloc[-1])
        signals["mcclellan"] = {
            "raw": f"{m:+.1f}",
            "score": score_signal(m, CFG.mcc_bullish, CFG.mcc_bearish, 150, -150),
            "label": "BULLISH" if m > CFG.mcc_bullish else "BEARISH" if m < CFG.mcc_bearish else "NEUTRAL",
            "detail": f"McClellan Oscillator: {m:+.1f}",
        }

    total_score = 0.0
    max_possible = 0.0
    for key, sig in signals.items():
        w = CFG.score_weights.get(key, 1.0)
        total_score += sig["score"] * w
        max_possible += abs(w)

    composite = (total_score / max_possible * 10) if max_possible > 0 else 0.0
    return signals, composite

## 9) Visualization Engine

Professional dark-theme Plotly charts with consistent styling, annotation helpers, and signal overlays.


In [9]:
# ============================================================
# 9) VISUALIZATION ENGINE
# ============================================================
PLOT_TEMPLATE = "plotly_dark"
COL = {
    "cyan": "#00e5ff",
    "orange": "#ff9100",
    "red": "#ff1744",
    "green": "#00e676",
    "white": "#ffffff",
    "gray": "#78909c",
    "yellow": "#ffea00",
    "purple": "#e040fb",
    "blue": "#448aff",
    "bg_card": "#0d1117",
    "bg_dark": "#010409",
    "border": "#21262d",
    "text_muted": "#8b949e",
    "bull": "#00e676",
    "bear": "#ff1744",
    "neutral": "#78909c",
}

def apply_layout(fig, title="", height=420, show_legend=True):
    fig.update_layout(
        template=PLOT_TEMPLATE,
        title=dict(text=title, x=0.02, xanchor="left", font=dict(size=16, color=COL["white"])),
        height=height,
        margin=dict(l=50, r=30, t=80, b=50),
        font=dict(size=12, color=COL["text_muted"]),
        hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0.0, xanchor="left",
                    bgcolor="rgba(0,0,0,0)", font=dict(size=11)),
        plot_bgcolor=COL["bg_dark"],
        paper_bgcolor=COL["bg_dark"],
    )
    if not show_legend:
        fig.update_layout(showlegend=False)
    fig.update_xaxes(showgrid=True, gridcolor="rgba(148,163,184,0.08)", zeroline=False,
                     linecolor=COL["border"])
    fig.update_yaxes(showgrid=True, gridcolor="rgba(148,163,184,0.08)", zeroline=False,
                     linecolor=COL["border"])
    return fig

def display_fig(fig):
    if fig is not None:
        fig.show(config={"displayModeBar": False, "responsive": True})

def fig_table(df, title="", precision=2, max_rows=50):
    d = df.copy()
    if len(d) > max_rows:
        d = d.head(max_rows)
    def fmt(v):
        if isinstance(v, (float, np.floating)):
            if np.isnan(v):
                return ""
            return f"{v:.{precision}f}"
        return str(v)
    cells = [[fmt(v) for v in d[col].tolist()] for col in d.columns]

    # Color-code numeric cells
    fill_colors = []
    for col_vals in cells:
        col_colors = []
        for v in col_vals:
            try:
                fv = float(v)
                if "%" in str(d.columns[cells.index([v for v2 in col_vals])]) if False else "":
                    pass
            except:
                pass
            col_colors.append(COL["bg_card"])
        fill_colors.append(col_colors)

    header = dict(
        values=[f"<b>{c}</b>" for c in d.columns],
        fill_color="#161b22",
        align="left",
        font=dict(color=COL["white"], size=12),
        line_color=COL["border"],
        height=32,
    )
    cell = dict(
        values=cells,
        fill_color=COL["bg_card"],
        align="left",
        font=dict(color="#c9d1d9", size=11),
        line_color=COL["border"],
        height=28,
    )
    fig = go.Figure(data=[go.Table(header=header, cells=cell)])
    fig.update_layout(
        template=PLOT_TEMPLATE,
        title=dict(text=title, x=0.02, font=dict(size=14, color=COL["white"])),
        height=min(380, 100 + len(d) * 30),
        margin=dict(l=20, r=20, t=65, b=20),
        plot_bgcolor=COL["bg_dark"],
        paper_bgcolor=COL["bg_dark"],
    )
    return fig

def signal_color(score):
    if score > 0.3:
        return COL["bull"]
    elif score < -0.3:
        return COL["bear"]
    return COL["neutral"]

def signal_emoji(label):
    mapping = {
        "BULLISH": "\u2B06", "ABOVE": "\u2B06", "STRONG": "\u2B06", "RISING": "\u2B06",
        "IMPROVING": "\u2B06", "THRUST": "\u26A1", "BREAKOUT": ">>",
        "BEARISH": "\u2B07", "BELOW": "\u2B07", "WEAK": "\u2B07", "FALLING": "\u2B07",
        "DETERIORATING": "\u2B07", "BREAKDOWN": "\u2B07",
        "OVERBOUGHT": "\u26A0", "OVERSOLD": "\u26A0", "ELEVATED": "\u26A0",
        "SEVERE": "!!", "BACKWARDATION": "\u26A0",
        "COMPRESSED": "\u2705", "CONTANGO": "\u2705", "NONE": "\u2705",
    }
    return mapping.get(label, "\u2796")


## 10) Conditions Menu — Filter Historical Dates for Outlook

Each condition isolates a subset of historical dates. The outlook engine then shows you what forward returns looked like on those dates. Combine two conditions with AND for more specific setups.


In [10]:
# ============================================================
# 10) CONDITIONS MENU
# ============================================================
def trend_stack_count(df: pd.DataFrame) -> pd.Series:
    cols = [f"dist_ma{w}" for w in CFG.ma_windows]
    present = [c for c in cols if c in df.columns]
    if not present:
        return pd.Series(index=df.index, dtype=float)
    return (df[present] > 0).sum(axis=1)

def build_conditions(state: Dict) -> Dict[str, pd.Series]:
    sig = state.get("index_feat", pd.DataFrame())
    if sig is None or sig.empty:
        return {}
    stack = trend_stack_count(sig)
    rv_pct = sig.get("rv20_pct", pd.Series(index=sig.index, dtype=float))
    rp = sig.get("range_pos", pd.Series(index=sig.index, dtype=float))
    r14 = sig.get("rsi14", pd.Series(index=sig.index, dtype=float))
    dd = sig.get("drawdown", pd.Series(index=sig.index, dtype=float))
    vt = sig.get("vol_term_structure", pd.Series(index=sig.index, dtype=float))

    cond = {}
    cond["All days"] = pd.Series(True, index=sig.index)

    # Trend conditions
    cond[f"Trend stack >= {CFG.trend_stack_bullish}/{len(CFG.ma_windows)} (strong uptrend)"] = (stack >= CFG.trend_stack_bullish)
    cond[f"Trend stack <= {CFG.trend_stack_bearish}/{len(CFG.ma_windows)} (strong downtrend)"] = (stack <= CFG.trend_stack_bearish)
    cond["Trend stack = 5/5 (perfect alignment)"] = (stack == len(CFG.ma_windows))
    cond["Trend stack = 0/5 (all below)"] = (stack == 0)

    # Range conditions
    cond[f"Near 52w high (>={CFG.range_pos_breakout:.0%})"] = (rp >= CFG.range_pos_breakout)
    cond[f"Near 52w low (<={CFG.range_pos_breakdown:.0%})"] = (rp <= CFG.range_pos_breakdown)
    cond["Mid-range (0.30–0.70)"] = ((rp >= 0.30) & (rp <= 0.70))

    # Vol conditions
    cond[f"High vol (RV >= {CFG.rv_high_pct:.0%} pctile)"] = (rv_pct >= CFG.rv_high_pct)
    cond[f"Low vol (RV <= {CFG.rv_low_pct:.0%} pctile)"] = (rv_pct <= CFG.rv_low_pct)
    cond["Vol spike (term structure > 1.3)"] = (vt > 1.3)

    # RSI conditions
    cond[f"RSI >= {CFG.rsi_overbought} (overbought)"] = (r14 >= CFG.rsi_overbought)
    cond[f"RSI <= {CFG.rsi_oversold} (oversold)"] = (r14 <= CFG.rsi_oversold)

    # Drawdown conditions
    cond["Drawdown > -5% (pullback)"] = (dd < -0.05)
    cond["Drawdown > -10% (correction)"] = (dd < -0.10)
    cond["Drawdown > -20% (bear market)"] = (dd < -0.20)

    # Combined conditions (pre-built for common setups)
    cond["SETUP: Bullish trend + low vol"] = ((stack >= CFG.trend_stack_bullish) & (rv_pct <= CFG.rv_low_pct))
    cond["SETUP: Oversold + high vol (bounce?)"] = ((r14 <= CFG.rsi_oversold) & (rv_pct >= CFG.rv_high_pct))
    cond["SETUP: Near highs + compressed vol (breakout?)"] = ((rp >= 0.85) & (rv_pct <= 0.30))
    cond["SETUP: Near lows + vol spike (capitulation?)"] = ((rp <= 0.15) & (vt > 1.3))

    # Breadth conditions
    memb = state.get("members_feat", {})
    if memb:
        bm = memb.get("breadth_mom_200")
        bm_pct = memb.get("breadth_mom_pct")
        disp_pct = memb.get("dispersion_pct")
        pct200 = memb.get("pct_above_200")
        if bm is not None and not bm.empty:
            cond["Breadth improving (mom >= +5pp)"] = (bm >= 0.05)
            cond["Breadth deteriorating (mom <= -5pp)"] = (bm <= -0.05)
        if pct200 is not None and not pct200.empty:
            cond[f"Broad participation (>={CFG.breadth_strong:.0%} > 200DMA)"] = (pct200 >= CFG.breadth_strong)
            cond[f"Narrow market (<={CFG.breadth_weak:.0%} > 200DMA)"] = (pct200 <= CFG.breadth_weak)
            cond["SETUP: Weak breadth + trend looks ok"] = ((pct200 <= CFG.breadth_weak) & (stack >= 3))
            cond["SETUP: Strong breadth + strong trend"] = ((pct200 >= CFG.breadth_strong) & (stack >= CFG.trend_stack_bullish))
        if disp_pct is not None and not disp_pct.empty:
            cond["High dispersion (>= 80th pct)"] = (disp_pct >= 0.80)
            cond["Low dispersion (<= 20th pct)"] = (disp_pct <= 0.20)

    for k in list(cond.keys()):
        cond[k] = cond[k].reindex(sig.index).fillna(False)
    return cond


## 11) State Builder — Loads Data & Computes Everything


In [11]:
# ============================================================



# 11) STATE BUILDER



# ============================================================



DASHBOARD_STATUS_HOOK = None











def _emit_status(message: str):



    hook = globals().get("DASHBOARD_STATUS_HOOK")



    if callable(hook):



        try:



            hook(message, "warn")



        except Exception:



            pass



    print(message)











def build_state(market_name: str) -> Dict:
    ticker = CFG.indices[market_name]
    _emit_status(f"{market_name}: loading index history")
    idx_close = load_or_download_index_close(ticker, CFG.lookback)
    _emit_status(f"{market_name}: computing index features")
    idx_feat = compute_index_features(idx_close)
    state = {
        "market_name": market_name,
        "ticker": ticker,
        "index_close": idx_close,
        "index_feat": idx_feat,
        "members_close": pd.DataFrame(),
        "members_feat": {},
        "constituents_df": pd.DataFrame(),
        "breadth_constituents_df": pd.DataFrame(),
        "sp500_thrust_mcc": pd.DataFrame(),
        "sp500_rrg": pd.DataFrame(),
        "sector_map": {},
        "conditions": {},
        "quality": {"index_rows": int(len(idx_close)), "index_empty": bool(len(idx_close) == 0)},
    }

    _emit_status(f"{market_name}: loading constituents / holdings list")
    state["constituents_df"] = load_market_constituents(market_name)
    state["quality"]["constituent_rows"] = int(len(state["constituents_df"]))
    if not state["constituents_df"].empty:
        state["quality"]["constituent_type"] = str(state["constituents_df"].get("UniverseType", pd.Series([""])).iloc[0])

    if market_universe_download_enabled(market_name) and not state["constituents_df"].empty:
        _emit_status(f"{market_name}: selecting constituent subset for breadth")
        state["breadth_constituents_df"] = select_constituent_subset(state["constituents_df"], market_name, CFG.breadth_member_limits)
        state["quality"]["breadth_constituent_rows"] = int(len(state["breadth_constituents_df"]))
        _emit_status(f"{market_name}: preparing member universe tickers for breadth")
        universe_symbols = prepare_market_universe_symbols(state["breadth_constituents_df"], market_name)
        universe_key = _safe_name(f"{market_name.lower()}_universe")
        if universe_symbols:
            _emit_status(f"{market_name}: downloading / loading member universe prices")
            uni_close = load_or_download_universe_close(universe_symbols, CFG.lookback, universe_key=universe_key)
            miss = uni_close.isna().mean()
            keep = miss[miss <= CFG.max_missing_pct].index.tolist()
            dropped = miss[miss > CFG.max_missing_pct].index.tolist()
            uni_close = uni_close[keep].ffill()
            state["members_close"] = uni_close
            state["quality"].update({
                "members_universe": market_name,
                "members_cols": int(uni_close.shape[1]),
                "dropped_members": int(len(dropped)),
                "coverage_today": float(uni_close.notna().iloc[-1].mean()) if len(uni_close) else np.nan,
                "dropped_sample": dropped[:25],
            })

            _emit_status(f"{market_name}: breadth step 1/6 - percent above 50DMA and 200DMA")
            _emit_status(f"{market_name}: breadth step 2/6 - 20-day breadth momentum")
            _emit_status(f"{market_name}: breadth step 3/6 - percent near 52-week highs and lows")
            _emit_status(f"{market_name}: breadth step 4/6 - dispersion and rolling dispersion")
            _emit_status(f"{market_name}: breadth step 5/6 - average cross-sectional correlation")
            _emit_status(f"{market_name}: breadth step 6/6 - percentile context for internals")
            state["members_feat"] = compute_universe_breadth_dispersion(uni_close)

            if market_name == "S&P 500":
                sector_map = dict(zip(
                    sp500_df["Symbol"],
                    sp500_df["Sector"] if "Sector" in sp500_df.columns else sp500_df.get("GICS Sector", "")
                ))
                _emit_status("S&P 500: thrust step 1/4 - advance / decline counts")
                _emit_status("S&P 500: thrust step 2/4 - breadth thrust EMA")
                _emit_status("S&P 500: thrust step 3/4 - McClellan oscillator")
                _emit_status("S&P 500: thrust step 4/4 - AD line and percentile context")
                state["sp500_thrust_mcc"] = compute_sp500_breadth_thrust_mcclellan(uni_close)
                _emit_status("S&P 500: RRG step 1/5 - aligning benchmark series")
                bench = idx_close.reindex(uni_close.index).ffill()
                _emit_status("S&P 500: RRG step 2/5 - grouping members into sectors")
                _emit_status("S&P 500: RRG step 3/5 - building sector return streams")
                _emit_status("S&P 500: RRG step 4/5 - computing RS-Ratio and RS-Momentum")
                _emit_status("S&P 500: RRG step 5/5 - assigning quadrants and tail paths")
                state["sp500_rrg"] = compute_sp500_rrg(uni_close, sector_map, bench)
                _emit_status("S&P 500: storing sector map and quality stats")
                state["sector_map"] = sector_map

    _emit_status(f"{market_name}: building condition library")
    state["conditions"] = build_conditions(state)
    return state

DEFAULT_MARKET = "S&P 500"



state = {}



print(f"Default market configured: {DEFAULT_MARKET}")

Default market configured: S&P 500


## 12) Plot Builders — Every Chart Explained


In [12]:
# ============================================================

# 12) PLOT BUILDERS

# ============================================================



# ----- TAB 1: SIGNAL SCORECARD -----

def fig_signal_scorecard(state):

    """The master signal dashboard — single screen showing all signals firing today."""

    df = state["index_feat"]

    if df.empty:

        return go.Figure(), go.Figure()



    signals, composite = compute_signal_scores(state)



    # Build the scorecard table

    rows = []

    for key, sig in signals.items():

        rows.append({

            "Signal": key.replace("_", " ").title(),

            "Value": sig["raw"],

            "Label": sig["label"],

            "Score": f"{sig['score']:+.2f}",

            "Weight": f"{CFG.score_weights.get(key, 1.0):.1f}",

            "Detail": sig["detail"],

        })

    rows.append({

        "Signal": "═══ COMPOSITE ═══",

        "Value": f"{composite:+.1f}/10",

        "Label": "BULLISH" if composite > 3 else "BEARISH" if composite < -3 else "NEUTRAL",

        "Score": f"{composite:+.1f}",

        "Weight": "",

        "Detail": "Weighted sum of all signals, scaled to ±10",

    })



    tbl_df = pd.DataFrame(rows)



    # Color the cells based on label

    label_colors = []

    for _, r in tbl_df.iterrows():

        lbl = r["Label"]

        if lbl in ("BULLISH", "ABOVE", "STRONG", "RISING", "IMPROVING", "THRUST", "BREAKOUT", "COMPRESSED", "CONTANGO", "NONE"):

            label_colors.append(COL["bull"])

        elif lbl in ("BEARISH", "BELOW", "WEAK", "FALLING", "DETERIORATING", "BREAKDOWN", "SEVERE"):

            label_colors.append(COL["bear"])

        elif lbl in ("OVERBOUGHT", "OVERSOLD", "ELEVATED", "BACKWARDATION"):

            label_colors.append(COL["yellow"])

        else:

            label_colors.append(COL["neutral"])



    header = dict(

        values=[f"<b>{c}</b>" for c in tbl_df.columns],

        fill_color="#161b22", align="left", font=dict(color=COL["white"], size=12),

        line_color=COL["border"], height=32,

    )

    cell_vals = [tbl_df[col].tolist() for col in tbl_df.columns]



    # Color the Label column

    fill_2d = []

    for i, col in enumerate(tbl_df.columns):

        if col == "Label":

            fill_2d.append([f"rgba({int(c[1:3],16)},{int(c[3:5],16)},{int(c[5:7],16)},0.15)" for c in label_colors])

        elif col == "Score":

            sc_colors = []

            for _, r in tbl_df.iterrows():

                try:

                    sv = float(r["Score"])

                    if sv > 0.3:

                        sc_colors.append("rgba(0,230,118,0.12)")

                    elif sv < -0.3:

                        sc_colors.append("rgba(255,23,68,0.12)")

                    else:

                        sc_colors.append(COL["bg_card"])

                except:

                    sc_colors.append(COL["bg_card"])

            fill_2d.append(sc_colors)

        else:

            fill_2d.append([COL["bg_card"]] * len(tbl_df))



    # Font colors for Label column

    font_2d = []

    for i, col in enumerate(tbl_df.columns):

        if col == "Label":

            font_2d.append(label_colors)

        else:

            font_2d.append(["#c9d1d9"] * len(tbl_df))



    cell = dict(

        values=cell_vals, fill_color=fill_2d, align="left",

        font=dict(color=font_2d, size=11), line_color=COL["border"], height=28,

    )

    scorecard_fig = go.Figure(data=[go.Table(header=header, cells=cell)])

    scorecard_fig.update_layout(

        template=PLOT_TEMPLATE,

        title=dict(text=f"SIGNAL SCORECARD — {state['market_name']} | Composite: {composite:+.1f}/10",

                   x=0.02, font=dict(size=16, color=COL["white"])),

        height=min(600, 120 + len(tbl_df) * 32),

        margin=dict(l=20, r=20, t=70, b=20),

        plot_bgcolor=COL["bg_dark"], paper_bgcolor=COL["bg_dark"],

    )



    # Composite gauge

    gauge_fig = go.Figure(go.Indicator(

        mode="gauge+number+delta",

        value=composite,

        title={"text": "Composite Signal Score", "font": {"size": 16, "color": COL["white"]}},

        number={"font": {"size": 36, "color": COL["white"]}, "suffix": "/10"},

        gauge={

            "axis": {"range": [-10, 10], "tickwidth": 1, "tickcolor": COL["gray"]},

            "bar": {"color": COL["bull"] if composite > 0 else COL["bear"]},

            "bgcolor": COL["bg_card"],

            "borderwidth": 1,

            "bordercolor": COL["border"],

            "steps": [

                {"range": [-10, -6], "color": "rgba(255,23,68,0.3)"},

                {"range": [-6, -3], "color": "rgba(255,23,68,0.15)"},

                {"range": [-3, 3], "color": "rgba(120,144,156,0.15)"},

                {"range": [3, 6], "color": "rgba(0,230,118,0.15)"},

                {"range": [6, 10], "color": "rgba(0,230,118,0.3)"},

            ],

            "threshold": {

                "line": {"color": COL["white"], "width": 3},

                "thickness": 0.8,

                "value": composite,

            },

        },

    ))

    gauge_fig.update_layout(

        template=PLOT_TEMPLATE, height=280,

        margin=dict(l=40, r=40, t=50, b=20),

        plot_bgcolor=COL["bg_dark"], paper_bgcolor=COL["bg_dark"],

    )



    return scorecard_fig, gauge_fig



# ----- TAB 2: TREND SIGNALS -----

def fig_trend_signals(state):

    df = state["index_feat"]

    if df.empty:

        return go.Figure(), go.Figure(), go.Figure()



    # Price + MAs with signal highlights

    fig1 = go.Figure()

    fig1.add_trace(go.Scatter(x=df.index, y=df["close"], name="Close",

                              line=dict(color=COL["white"], width=2.5)))

    ma_colors = {10: COL["purple"], 20: COL["cyan"], 50: COL["yellow"],

                 100: COL["orange"], 200: COL["red"]}

    for w in CFG.ma_windows:

        col = f"ma{w}"

        if col in df.columns:

            fig1.add_trace(go.Scatter(x=df.index, y=df[col], name=f"MA{w}",

                                      line=dict(color=ma_colors.get(w, COL["gray"]), width=1.5,

                                               dash="dash" if w >= 100 else "solid"),

                                      opacity=0.8))

    apply_layout(fig1, title="Price + Moving Averages — Look for MA alignment (all rising = strong trend)", height=550)



    # Trend stack distances as heatmap-style

    fig2 = go.Figure()

    for w in CFG.ma_windows:

        col = f"dist_ma{w}"

        if col in df.columns:

            fig2.add_trace(go.Scatter(x=df.index, y=df[col]*100, name=f"vs MA{w}",

                                      line=dict(width=2.2)))

    fig2.add_hline(y=0, line_color=COL["gray"], opacity=0.6, line_dash="dash")

    apply_layout(fig2, title="Distance to Each MA (%) — Above 0 = bullish for that timeframe, all above = trend-aligned", height=480)

    fig2.update_yaxes(title="Distance (%)")



    # Range position + momentum

    fig3 = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.08,

                         subplot_titles=(

                             "52-Week Range Position (>0.90 = breakout zone, <0.10 = breakdown zone)",

                             "20-Day Rate of Change (%) — Momentum confirmation",

                             "MA Slopes — Rising MAs confirm trend direction",

                         ))

    fig3.add_trace(go.Scatter(x=df.index, y=df["range_pos"], name="RangePos",

                              line=dict(color=COL["cyan"], width=2.2)), row=1, col=1)

    fig3.add_hline(y=CFG.range_pos_breakout, line_color=COL["green"], opacity=0.5, line_dash="dot", row=1, col=1)

    fig3.add_hline(y=CFG.range_pos_breakdown, line_color=COL["red"], opacity=0.5, line_dash="dot", row=1, col=1)



    roc20 = df.get("roc_20d")

    if roc20 is not None:

        colors = [COL["bull"] if v > 0 else COL["bear"] for v in roc20.values]

        fig3.add_trace(go.Bar(x=df.index, y=roc20*100, name="ROC 20D",

                              marker_color=colors, opacity=0.7), row=2, col=1)



    for w in [50, 200]:

        scol = f"slope_ma{w}"

        if scol in df.columns:

            fig3.add_trace(go.Scatter(x=df.index, y=df[scol]*100, name=f"Slope MA{w}",

                                      line=dict(width=1.8)), row=3, col=1)

    fig3.add_hline(y=0, line_color=COL["gray"], opacity=0.4, line_dash="dash", row=3, col=1)



    apply_layout(fig3, title="Breakout State + Momentum Confirmation", height=850)

    return fig1, fig2, fig3



# ----- TAB 3: RISK & VOL SIGNALS -----

def fig_risk_signals(state):

    df = state["index_feat"]

    if df.empty:

        return go.Figure(), go.Figure()



    fig1 = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.06,

                         subplot_titles=(

                             "Drawdown from Peak — Severity zones: >-5% pullback, >-10% correction, >-20% bear",

                             "Realized Vol (20D ann) — High vol = wider stops, smaller positions",

                             "Vol Term Structure (Short/Long) — >1.2 = stress spike, <0.85 = calm",

                             "RSI(14) — >70 overbought (caution), <30 oversold (bounce potential)",

                         ))



    # Drawdown with severity bands

    fig1.add_trace(go.Scatter(x=df.index, y=df["drawdown"]*100, name="Drawdown",

                              fill="tozeroy", fillcolor="rgba(255,23,68,0.15)",

                              line=dict(color=COL["red"], width=1.8)), row=1, col=1)

    for level, label in [(-5, "-5%"), (-10, "-10%"), (-20, "-20%")]:

        fig1.add_hline(y=level, line_color=COL["gray"], opacity=0.3, line_dash="dot", row=1, col=1)



    # RV

    fig1.add_trace(go.Scatter(x=df.index, y=df["rv20_ann"]*100, name="RV20",

                              line=dict(color=COL["orange"], width=2)), row=2, col=1)

    if "rv60_ann" in df.columns:

        fig1.add_trace(go.Scatter(x=df.index, y=df["rv60_ann"]*100, name="RV60",

                                  line=dict(color=COL["gray"], width=1.5, dash="dash")), row=2, col=1)



    # Vol term structure

    if "vol_term_structure" in df.columns:

        vt = df["vol_term_structure"]

        fig1.add_trace(go.Scatter(x=df.index, y=vt, name="Vol Term",

                                  line=dict(color=COL["purple"], width=2)), row=3, col=1)

        fig1.add_hline(y=1.2, line_color=COL["red"], opacity=0.4, line_dash="dot", row=3, col=1)

        fig1.add_hline(y=0.85, line_color=COL["green"], opacity=0.4, line_dash="dot", row=3, col=1)

        fig1.add_hline(y=1.0, line_color=COL["gray"], opacity=0.3, line_dash="dash", row=3, col=1)



    # RSI

    fig1.add_trace(go.Scatter(x=df.index, y=df["rsi14"], name="RSI(14)",

                              line=dict(color=COL["cyan"], width=2)), row=4, col=1)

    fig1.add_hline(y=CFG.rsi_overbought, line_color=COL["red"], opacity=0.4, line_dash="dot", row=4, col=1)

    fig1.add_hline(y=CFG.rsi_oversold, line_color=COL["green"], opacity=0.4, line_dash="dot", row=4, col=1)

    fig1.add_hline(y=50, line_color=COL["gray"], opacity=0.3, line_dash="dash", row=4, col=1)



    apply_layout(fig1, title="Risk & Volatility Dashboard", height=1100)



    # ATR for position sizing

    fig2 = go.Figure()

    if "atr14" in df.columns:

        atr_pct = (df["atr14"] / df["close"]) * 100

        fig2.add_trace(go.Scatter(x=df.index, y=atr_pct, name="ATR(14) as % of price",

                                  line=dict(color=COL["yellow"], width=2)))

        last_atr_pct = atr_pct.dropna().iloc[-1] if len(atr_pct.dropna()) else 0

        fig2.add_annotation(x=df.index[-1], y=last_atr_pct,

                           text=f"Current: {last_atr_pct:.2f}%<br>→ 2x ATR stop = {last_atr_pct*2:.2f}%",

                           showarrow=True, arrowhead=2, font=dict(size=12, color=COL["yellow"]))

    apply_layout(fig2, title="ATR(14) as % of Price — Use for stop placement (common: 1.5–2x ATR)", height=400)



    return fig1, fig2



# ----- TAB 4: BREADTH & INTERNALS -----

def fig_breadth(state):

    memb = state.get("members_feat", {})

    if not memb:

        return None, None, None



    pct50 = memb.get("pct_above_50")

    pct200 = memb.get("pct_above_200")

    bm = memb.get("breadth_mom_200")

    disp = memb.get("dispersion_roll")

    pct_hi = memb.get("pct_near_52w_high")

    pct_lo = memb.get("pct_near_52w_low")



    fig1 = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.07,

                         subplot_titles=(

                             "% Above 200DMA — >70% = healthy breadth, <40% = narrow/weak market",

                             "% Above 50DMA — Shorter-term participation",

                             "Breadth Momentum (Δ %>200DMA, 20D) — Rising = improving, falling = deteriorating",

                         ))

    if pct200 is not None:

        fig1.add_trace(go.Scatter(x=pct200.index, y=pct200*100, name="%>200DMA",

                                  line=dict(color=COL["orange"], width=2.2)), row=1, col=1)

        fig1.add_hline(y=70, line_color=COL["green"], opacity=0.4, line_dash="dot", row=1, col=1)

        fig1.add_hline(y=40, line_color=COL["red"], opacity=0.4, line_dash="dot", row=1, col=1)

    if pct50 is not None:

        fig1.add_trace(go.Scatter(x=pct50.index, y=pct50*100, name="%>50DMA",

                                  line=dict(color=COL["cyan"], width=2)), row=2, col=1)

    if bm is not None:

        colors = [COL["bull"] if v > 0 else COL["bear"] for v in bm.values]

        fig1.add_trace(go.Bar(x=bm.index, y=bm*100, name="Breadth Mom (pp)",

                              marker_color=colors, opacity=0.7), row=3, col=1)

        fig1.add_hline(y=0, line_color=COL["gray"], opacity=0.4, row=3, col=1)



    apply_layout(fig1, title=f"{state.get('quality',{}).get('members_universe','Members')} Breadth Dashboard", height=850)



    # New highs / new lows

    fig2 = None

    if pct_hi is not None and pct_lo is not None:

        fig2 = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.10,

                             subplot_titles=(

                                 "% Near 52-Week Highs (within 2%) — Broad highs = healthy rally",

                                 "% Near 52-Week Lows (within 2%) — Broad lows = widespread selling",

                             ))

        fig2.add_trace(go.Scatter(x=pct_hi.index, y=pct_hi*100, name="Near 52w Highs",

                                  line=dict(color=COL["green"], width=2)), row=1, col=1)

        fig2.add_trace(go.Scatter(x=pct_lo.index, y=pct_lo*100, name="Near 52w Lows",

                                  line=dict(color=COL["red"], width=2)), row=2, col=1)

        apply_layout(fig2, title="New High / New Low Breadth", height=550)



    # Dispersion

    fig3 = go.Figure()

    if disp is not None:

        fig3.add_trace(go.Scatter(x=disp.index, y=disp*100, name="Dispersion (20D avg)",

                                  line=dict(color=COL["purple"], width=2.2)))

    avg_corr = memb.get("avg_corr")

    if avg_corr is not None:

        fig3.add_trace(go.Scatter(x=avg_corr.dropna().index, y=avg_corr.dropna(),

                                  name="Avg Pairwise Corr (60D)", yaxis="y2",

                                  line=dict(color=COL["yellow"], width=1.8, dash="dash")))

        fig3.update_layout(yaxis2=dict(title="Correlation", overlaying="y", side="right",

                                        showgrid=False, range=[0, 1]))

    apply_layout(fig3, title="Dispersion + Correlation — High disp = stock picking matters, high corr = macro-driven", height=450)



    return fig1, fig2, fig3



# ----- TAB 4b: THRUST & McCLELLAN -----

def fig_thrust_mcclellan(state):

    thrust_df = state.get("sp500_thrust_mcc", pd.DataFrame())

    if thrust_df.empty:

        return None



    fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.07,

                        subplot_titles=(

                            f"Breadth Thrust (10D EMA) — >{CFG.thrust_signal:.3f} = rare bullish thrust signal",

                            "McClellan Oscillator — >+50 = breadth expanding, <-50 = contracting",

                            "Advance-Decline Line (cumulative) — Should confirm price trends",

                        ))

    fig.add_trace(go.Scatter(x=thrust_df.index, y=thrust_df["breadth_thrust"],

                             name="Breadth Thrust", line=dict(color=COL["cyan"], width=2.2)), row=1, col=1)

    fig.add_hline(y=CFG.thrust_signal, line_color=COL["green"], opacity=0.6, line_dash="dot", row=1, col=1)



    mcc = thrust_df["mcclellan"]

    colors_mcc = [COL["bull"] if v > 0 else COL["bear"] for v in mcc.values]

    fig.add_trace(go.Bar(x=thrust_df.index, y=mcc, name="McClellan",

                         marker_color=colors_mcc, opacity=0.7), row=2, col=1)

    fig.add_hline(y=CFG.mcc_bullish, line_color=COL["green"], opacity=0.3, line_dash="dot", row=2, col=1)

    fig.add_hline(y=CFG.mcc_bearish, line_color=COL["red"], opacity=0.3, line_dash="dot", row=2, col=1)



    fig.add_trace(go.Scatter(x=thrust_df.index, y=thrust_df["ad_line"],

                             name="A/D Line", line=dict(color=COL["orange"], width=2)), row=3, col=1)



    apply_layout(fig, title="S&P 500 Breadth Thrust & McClellan Oscillator", height=850)

    return fig



# ----- TAB 5: RRG -----

def fig_rrg(state):

    if state["market_name"] != "S&P 500":

        return None, None

    rrg = state.get("sp500_rrg", pd.DataFrame())

    if rrg is None or rrg.empty:

        return None, None



    sectors = rrg["Sector"].tolist()

    palette = px.colors.qualitative.Dark24

    colors = {sec: palette[i % len(palette)] for i, sec in enumerate(sectors)}



    fig = go.Figure()

    # Quadrant shading

    fig.add_shape(type="rect", x0=100, y0=100, x1=105, y1=105,

                  fillcolor="rgba(0,230,118,0.06)", line_width=0)

    fig.add_shape(type="rect", x0=95, y0=100, x1=100, y1=105,

                  fillcolor="rgba(68,138,255,0.06)", line_width=0)

    fig.add_shape(type="rect", x0=100, y0=95, x1=105, y1=100,

                  fillcolor="rgba(255,145,0,0.06)", line_width=0)

    fig.add_shape(type="rect", x0=95, y0=95, x1=100, y1=100,

                  fillcolor="rgba(255,23,68,0.06)", line_width=0)



    fig.add_vline(x=100, line_dash="dash", line_color=COL["gray"], opacity=0.6)

    fig.add_hline(y=100, line_dash="dash", line_color=COL["gray"], opacity=0.6)



    # Quadrant labels

    for txt, x, y in [("LEADING", 102, 103), ("IMPROVING", 98, 103),

                       ("WEAKENING", 102, 97), ("LAGGING", 98, 97)]:

        fig.add_annotation(x=x, y=y, text=f"<b>{txt}</b>", showarrow=False,

                          font=dict(size=10, color="rgba(255,255,255,0.25)"))



    for _, row in rrg.iterrows():

        sec = row["Sector"]

        c = colors[sec]

        fig.add_trace(go.Scatter(x=row["Tail_X"], y=row["Tail_Y"], mode="lines",

                                 line=dict(color=c, width=3), showlegend=False, opacity=0.5))

        fig.add_trace(go.Scatter(x=[row["RS_Ratio"]], y=[row["RS_Momentum"]], mode="markers+text",

                                 marker=dict(size=14, color=c, line=dict(color="white", width=1)),

                                 text=[sec], textposition="top center", name=sec,

                                 textfont=dict(size=10)))

    apply_layout(fig, title="S&P 500 Sector Relative Rotation Graph (RRG)", height=700)



    # Ranks table with trade implications

    ranks = rrg[["Sector","Members","RS_Ratio","RS_Momentum","Quadrant","Ret_20D"]].copy()

    ranks["Score"] = (ranks["RS_Ratio"]-100) + 0.7*(ranks["RS_Momentum"]-100)

    ranks["Ret_20D"] = (ranks["Ret_20D"] * 100).round(2)

    ranks = ranks.sort_values("Score", ascending=False).reset_index(drop=True)

    ranks.columns = ["Sector", "Members", "RS Ratio", "RS Mom", "Quadrant", "20D Ret%", "RRG Score"]

    ranks_fig = fig_table(ranks, title="Sector Ranks — Trade: Long Leading sectors, Short/Underweight Lagging", precision=2)



    return fig, ranks_fig



# ----- TAB 6: OUTLOOK -----

def fig_outlook(state, cond1, cond2):

    df = state["index_feat"]

    if df.empty:

        return go.Figure(), go.Figure(), go.Figure()



    cond = state["conditions"]

    m1 = cond.get(cond1, pd.Series(False, index=df.index))

    if cond2 and cond2 in cond and cond2 != "None":

        mask = (m1 & cond[cond2])

        cond_name = f"{cond1} AND {cond2}"

    else:

        mask = m1

        cond_name = cond1



    close = df["close"]

    fan = quantile_fan(close, mask, CFG.forward_horizons)

    tbl = outlook_table(close, mask, CFG.forward_horizons)



    # Enhanced fan chart

    f = go.Figure()

    # P5-P95 band

    f.add_trace(go.Scatter(x=fan["horizon"], y=fan.get("p95", fan["p90"])*100, mode="lines",

                           name="P95", line=dict(color=COL["gray"], width=1, dash="dot"),

                           showlegend=True))

    f.add_trace(go.Scatter(x=fan["horizon"], y=fan.get("p5", fan["p10"])*100, mode="lines",

                           name="P5", line=dict(color=COL["gray"], width=1, dash="dot"),

                           fill="tonexty", fillcolor="rgba(120,144,156,0.08)", showlegend=True))

    # P10-P90 band

    f.add_trace(go.Scatter(x=fan["horizon"], y=fan["p90"]*100, mode="lines",

                           name="P90", line=dict(color=COL["orange"], width=1.5)))

    f.add_trace(go.Scatter(x=fan["horizon"], y=fan["p10"]*100, mode="lines",

                           name="P10", line=dict(color=COL["orange"], width=1.5),

                           fill="tonexty", fillcolor="rgba(255,145,0,0.12)"))

    # P25-P75 band

    f.add_trace(go.Scatter(x=fan["horizon"], y=fan["p75"]*100, mode="lines",

                           name="P75", line=dict(color=COL["cyan"], width=1.8)))

    f.add_trace(go.Scatter(x=fan["horizon"], y=fan["p25"]*100, mode="lines",

                           name="P25", line=dict(color=COL["cyan"], width=1.8),

                           fill="tonexty", fillcolor="rgba(0,229,255,0.15)"))

    # Median

    f.add_trace(go.Scatter(x=fan["horizon"], y=fan["p50"]*100, mode="lines+markers",

                           name="Median", line=dict(color=COL["white"], width=3),

                           marker=dict(size=10, color=COL["white"])))

    # Zero line

    f.add_hline(y=0, line_color=COL["gray"], opacity=0.4, line_dash="dash")



    apply_layout(f, title=f"Forward Return Fan — Condition: {cond_name}", height=550)

    f.update_xaxes(title="Horizon (trading days)")

    f.update_yaxes(title="Forward return (%)")



    # Outlook table

    tfig = fig_table(tbl, title="Detailed Outlook Statistics — Returns %, Win rates %, MAE/MFE %", precision=2)



    # Forward return histogram for 20D horizon

    fwd20 = forward_returns(close, 20)[mask].dropna() * 100

    hist_fig = go.Figure()

    if len(fwd20) > 10:

        hist_fig.add_trace(go.Histogram(x=fwd20, nbinsx=40, name="20D Forward Returns",

                                        marker_color=COL["cyan"], opacity=0.7))

        hist_fig.add_vline(x=0, line_color=COL["gray"], opacity=0.6, line_dash="dash")

        hist_fig.add_vline(x=float(fwd20.median()), line_color=COL["white"], opacity=0.8, line_width=2,

                          annotation_text=f"Median: {fwd20.median():.1f}%")

        # Mark the tails

        p5 = float(fwd20.quantile(0.05))

        p95 = float(fwd20.quantile(0.95))

        hist_fig.add_vrect(x0=fwd20.min(), x1=p5, fillcolor="rgba(255,23,68,0.15)", line_width=0)

        hist_fig.add_vrect(x0=p95, x1=fwd20.max(), fillcolor="rgba(0,230,118,0.15)", line_width=0)

    apply_layout(hist_fig, title=f"20-Day Forward Return Distribution (N={len(fwd20)}) — Red/green tails = 5th/95th percentile", height=400)



    return f, tfig, hist_fig



# ----- TAB 7: CROSS-MARKET HEATMAP -----

def cross_market_snapshot():

    rows = []

    for name, ticker in CFG.indices.items():

        s = load_or_download_index_close(ticker, CFG.lookback)

        feat = compute_index_features(s)

        if feat.empty:

            continue

        last = feat.iloc[-1]

        stack = int(trend_stack_count(feat).iloc[-1])

        rows.append({

            "Market": name,

            "Close": float(last["close"]),

            "1D%": float(last.get("ret1d", np.nan))*100,

            "20D%": float(last.get("roc_20d", np.nan))*100,

            "Stack": f"{stack}/{len(CFG.ma_windows)}",

            "vs200DMA%": float(last.get("dist_ma200", np.nan))*100,

            "RangePos": float(last.get("range_pos", np.nan)),

            "DD%": float(last.get("drawdown", np.nan))*100,

            "RV20%": float(last.get("rv20_ann", np.nan))*100,

            "RSI": float(last.get("rsi14", np.nan)),

            "RV_Pct": float(last.get("rv20_pct", np.nan)),

        })

    df = pd.DataFrame(rows)

    if df.empty:

        return go.Figure()

    df = df.sort_values("RangePos", ascending=False).reset_index(drop=True)

    return fig_table(df.round(2), title="Cross-Market Snapshot — Sorted by 52w range position (strongest at top)", precision=2)



# ----- TAB 8: TRADE IDEAS -----

def fig_trade_ideas(state):

    """Generate mechanical trade ideas from current signal state."""

    df = state["index_feat"]

    if df.empty:

        return go.Figure()



    signals, composite = compute_signal_scores(state)

    last = df.iloc[-1]

    name = state["market_name"]

    atr_pct = (last.get("atr14", 0) / last.get("close", 1)) * 100



    ideas = []



    # Idea 1: Trend-following

    stack_sig = signals.get("trend_stack", {})

    if stack_sig.get("score", 0) >= 0.5:

        ideas.append({

            "Type": "TREND-FOLLOW LONG",

            "Signal": f"Strong uptrend ({stack_sig['raw']} MAs above)",

            "Entry": f"Buy on pullback to 20DMA ({last.get('ma20', 0):.0f})",

            "Stop": f"Below 50DMA ({last.get('ma50', 0):.0f}) or 2x ATR ({atr_pct*2:.1f}%)",

            "Target": f"52w high ({last['close'] / (1 + last.get('dist_52w_high', 0)):.0f}) or trailing 20DMA",

            "Confidence": "HIGH" if composite > 5 else "MODERATE",

        })

    elif stack_sig.get("score", 0) <= -0.5:

        ideas.append({

            "Type": "TREND-FOLLOW SHORT / HEDGE",

            "Signal": f"Strong downtrend ({stack_sig['raw']} MAs above)",

            "Entry": f"Short on rally to 20DMA ({last.get('ma20', 0):.0f})",

            "Stop": f"Above 50DMA ({last.get('ma50', 0):.0f}) or 2x ATR ({atr_pct*2:.1f}%)",

            "Target": f"52w low ({last.get('close', 0) / (1 + last.get('dist_52w_low', 0)):.0f})",

            "Confidence": "HIGH" if composite < -5 else "MODERATE",

        })



    # Idea 2: Mean reversion

    rsi_sig = signals.get("rsi_signal", {})

    if rsi_sig.get("label") == "OVERSOLD":

        ideas.append({

            "Type": "MEAN-REVERSION LONG",

            "Signal": f"RSI oversold ({rsi_sig['raw']})",

            "Entry": f"Buy at current ({last['close']:.0f}) or scale in",

            "Stop": f"Below recent low or 2x ATR ({atr_pct*2:.1f}%)",

            "Target": f"Mean RSI (50) / 20DMA ({last.get('ma20', 0):.0f})",

            "Confidence": "MODERATE — confirm with breadth" if composite > -3 else "LOW — fighting trend",

        })

    elif rsi_sig.get("label") == "OVERBOUGHT":

        ideas.append({

            "Type": "TAKE PROFIT / REDUCE",

            "Signal": f"RSI overbought ({rsi_sig['raw']})",

            "Entry": "Reduce long exposure or tighten stops",

            "Stop": "N/A — risk management action",

            "Target": "Lock in gains, re-enter on pullback",

            "Confidence": "MODERATE — trend may override",

        })



    # Idea 3: Vol compression breakout

    vol_sig = signals.get("vol_regime", {})

    range_sig = signals.get("range_position", {})

    if vol_sig.get("label") == "COMPRESSED" and range_sig.get("score", 0) > 0:

        ideas.append({

            "Type": "BREAKOUT WATCH",

            "Signal": f"Low vol ({vol_sig['raw']}) + near highs (RangePos {range_sig['raw']})",

            "Entry": f"Buy on break above 52w high with vol expansion",

            "Stop": f"Below breakout level or 2x ATR ({atr_pct*2:.1f}%)",

            "Target": "Measured move = 52w range added to breakout",

            "Confidence": "HIGH — classic breakout setup",

        })



    # Idea 4: Capitulation / Bottom fishing

    dd_sig = signals.get("drawdown", {})

    if dd_sig.get("label") in ("SEVERE",):

        ideas.append({

            "Type": "CAPITULATION WATCH",

            "Signal": f"Severe drawdown ({dd_sig['raw']})",

            "Entry": "Scale in slowly — don't catch falling knife",

            "Stop": f"Below further -10% or defined dollar risk",

            "Target": f"200DMA ({last.get('ma200', 0):.0f}) as first target",

            "Confidence": "LOW — needs vol + breadth confirmation",

        })



    # Idea 5: Breadth divergence

    breadth_sig = signals.get("breadth", {})

    if breadth_sig:

        if breadth_sig.get("score", 0) < -0.3 and stack_sig.get("score", 0) > 0.3:

            ideas.append({

                "Type": "BREADTH DIVERGENCE WARNING",

                "Signal": f"Index looks ok but breadth weak ({breadth_sig['raw']} > 200DMA)",

                "Entry": "Reduce size / tighten stops on longs",

                "Stop": "N/A — risk management signal",

                "Target": "Wait for breadth to confirm or index to crack",

                "Confidence": "HIGH — breadth divergence is a leading indicator",

            })



    if not ideas:

        ideas.append({

            "Type": "NO CLEAR SETUP",

            "Signal": "Mixed signals — no high-conviction trade",

            "Entry": "Wait for clarity",

            "Stop": "N/A",

            "Target": "N/A",

            "Confidence": "N/A — patience is a position",

        })



    ideas_df = pd.DataFrame(ideas)

    return fig_table(ideas_df, title=f"Trade Ideas for {name} — Mechanical, based on current signal state", precision=2, max_rows=20)



# ----- CONSTITUENTS / HOLDINGS -----
def fig_constituents(state, max_rows: int = 200):
    df = state.get("constituents_df", pd.DataFrame()).copy()
    if df.empty:
        return fig_table(
            pd.DataFrame([{"Message": "No constituent or proxy holdings list available for this market."}]),
            title="Constituents / Holdings",
            precision=2,
            max_rows=10,
        )

    summary_rows = [
        ("Market", state.get("market_name", "N/A")),
        ("Rows", len(df)),
        ("Universe Type", df.get("UniverseType", pd.Series(["N/A"])).iloc[0]),
        ("Source", df.get("Source", pd.Series(["N/A"])).iloc[0]),
        ("Note", df.get("Note", pd.Series(["N/A"])).iloc[0]),
    ]
    summary_df = pd.DataFrame(summary_rows, columns=["Metric", "Value"])

    display_df = df.copy()
    if "Weight" in display_df.columns:
        display_df["Weight"] = display_df["Weight"].astype(str)
    display_df = display_df.head(max_rows)

    fig = make_subplots(
        rows=2,
        cols=1,
        specs=[[{"type": "table"}], [{"type": "table"}]],
        vertical_spacing=0.08,
        row_heights=[0.25, 0.75],
        subplot_titles=("Universe Summary", f"Constituents / Holdings (top {min(max_rows, len(df))} rows)"),
    )
    fig.add_trace(
        go.Table(
            header=dict(values=[f"<b>{c}</b>" for c in summary_df.columns], fill_color="#161b22", align="left", font=dict(color=COL["white"], size=12), line_color=COL["border"]),
            cells=dict(values=[summary_df[c].tolist() for c in summary_df.columns], fill_color=COL["bg_card"], align="left", font=dict(color="#c9d1d9", size=11), line_color=COL["border"]),
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Table(
            header=dict(values=[f"<b>{c}</b>" for c in display_df.columns], fill_color="#161b22", align="left", font=dict(color=COL["white"], size=12), line_color=COL["border"]),
            cells=dict(values=[display_df[c].tolist() for c in display_df.columns], fill_color=COL["bg_card"], align="left", font=dict(color="#c9d1d9", size=10), line_color=COL["border"], height=24),
        ),
        row=2,
        col=1,
    )
    apply_layout(fig, title=f"Constituents / Proxy Holdings ? {state['market_name']}", height=950, show_legend=False)
    return fig

# ----- DATA QUALITY -----

def fig_quality(state):

    q = state.get("quality", {})

    rows = []

    for k, v in q.items():

        if isinstance(v, (list, tuple)):

            rows.append((k, ", ".join(map(str, v[:15])) + (" ..." if len(v) > 15 else "")))

        else:

            rows.append((k, v))

    df = pd.DataFrame(rows, columns=["Metric", "Value"])

    return fig_table(df, title="Data Quality / Diagnostics", precision=4, max_rows=80)

## 13) Trade Intelligence Engine
This upgrade turns the notebook into a fuller tactical workflow:

- ranks current setups instead of listing a few generic ideas
- sizes conviction using signal alignment, historical condition edge, and reward-to-risk
- adds tactical regime context, invalidation levels, and trade management notes
- caches market states so cross-market comparisons and UI interactions stay responsive

In [13]:
# ============================================================

# 13) TRADE INTELLIGENCE ENGINE

# ============================================================

STATE_CACHE = {}

SETUP_EDGE_CACHE = {}



CFG.setup_preferred_horizon = getattr(CFG, "setup_preferred_horizon", 20)

CFG.setup_min_sample = getattr(CFG, "setup_min_sample", 25)

CFG.idea_limit = getattr(CFG, "idea_limit", 8)

CFG.cross_market_limit = getattr(CFG, "cross_market_limit", 10)





def _safe_float(value, default=np.nan):

    try:

        if pd.isna(value):

            return default

        return float(value)

    except Exception:

        return default





def _fmt_price(value):

    if pd.isna(value):

        return "N/A"

    return f"{float(value):,.2f}"





def _fmt_pct(value, digits=1):

    if pd.isna(value):

        return "N/A"

    return f"{float(value) * 100:.{digits}f}%"





def _risk_reward(entry, stop, target, direction):

    entry = _safe_float(entry)

    stop = _safe_float(stop)

    target = _safe_float(target)

    if any(pd.isna(v) for v in [entry, stop, target]):

        return np.nan

    if direction == "LONG":

        risk = entry - stop

        reward = target - entry

    else:

        risk = stop - entry

        reward = entry - target

    if risk <= 0:

        return np.nan

    return reward / risk





def _clamp(value, lower=0.0, upper=1.0):

    return max(lower, min(upper, value))





def _condition_mask(state, name):

    return state.get("conditions", {}).get(name, pd.Series(False, index=state["index_feat"].index))





def summarize_setup_edge(state, mask, horizon=20):

    close = state["index_feat"]["close"]

    if mask is None:

        mask = pd.Series(False, index=close.index)

    aligned_mask = mask.reindex(close.index).fillna(False)

    cache_key = (state.get("market_name", "unknown"), horizon, tuple(aligned_mask.astype(bool).tolist()))

    if cache_key in SETUP_EDGE_CACHE:

        return SETUP_EDGE_CACHE[cache_key]

    tbl = outlook_table(close, aligned_mask, (horizon,))

    if tbl.empty:

        return {

            "sample_size": 0,

            "win_rate": np.nan,

            "mean_return": np.nan,

            "median_return": np.nan,

            "p10": np.nan,

            "es5": np.nan,

            "edge_mean": np.nan,

            "edge_win": np.nan,

        }

    row = tbl.iloc[0]

    result = {

        "sample_size": int(row["N"]),

        "win_rate": _safe_float(row["Win%"]) / 100,

        "mean_return": _safe_float(row["Mean%"]) / 100,

        "median_return": _safe_float(row["Median%"]) / 100,

        "p10": _safe_float(row["P10%"]) / 100,

        "es5": _safe_float(row["ES(5%)"]) / 100,

        "edge_mean": _safe_float(row["Edge_Mean%"]) / 100,

        "edge_win": _safe_float(row["Edge_Win%"]) / 100,

    }

    SETUP_EDGE_CACHE[cache_key] = result

    return result





def build_market_regime(state):

    df = state["index_feat"]

    if df.empty:

        return {}

    signals, composite = compute_signal_scores(state)

    last = df.iloc[-1]

    stack = int(trend_stack_count(df).iloc[-1])

    breadth = state.get("members_feat", {}).get("pct_above_200")

    breadth_now = _safe_float(breadth.iloc[-1]) if breadth is not None and len(breadth.dropna()) else np.nan

    drawdown = _safe_float(last.get("drawdown"))

    rv_pct = _safe_float(last.get("rv20_pct"), 0.5)



    if composite >= 4:

        bias = "Risk-on bullish"

    elif composite <= -4:

        bias = "Risk-off bearish"

    else:

        bias = "Mixed / tactical"



    if rv_pct >= 0.8:

        vol_bucket = "High volatility"

    elif rv_pct <= 0.2:

        vol_bucket = "Compressed volatility"

    else:

        vol_bucket = "Normal volatility"



    if drawdown <= -0.2:

        tape = "Deep drawdown"

    elif drawdown <= -0.1:

        tape = "Correction"

    elif drawdown <= -0.05:

        tape = "Pullback"

    else:

        tape = "Near highs / stable"



    participation = "N/A"

    if not pd.isna(breadth_now):

        if breadth_now >= 0.7:

            participation = "Broad participation"

        elif breadth_now <= 0.4:

            participation = "Narrow participation"

        else:

            participation = "Mixed participation"



    return {

        "bias": bias,

        "composite": composite,

        "trend_stack": f"{stack}/{len(CFG.ma_windows)}",

        "vol_bucket": vol_bucket,

        "tape": tape,

        "participation": participation,

    }





def generate_trade_ideas(state):

    df = state["index_feat"]

    if df.empty:

        return pd.DataFrame()



    signals, composite = compute_signal_scores(state)

    last = df.iloc[-1]

    members = state.get("members_feat", {})

    regime = build_market_regime(state)



    price = _safe_float(last["close"])

    atr = _safe_float(last.get("atr14"), 0.0)

    atr = 0.0 if pd.isna(atr) else atr

    ma20 = _safe_float(last.get("ma20"), price)

    ma50 = _safe_float(last.get("ma50"), price)

    ma200 = _safe_float(last.get("ma200"), price)

    high20 = _safe_float(df["close"].rolling(20).max().iloc[-1], price)

    low20 = _safe_float(df["close"].rolling(20).min().iloc[-1], price)

    high60 = _safe_float(df["close"].rolling(60).max().iloc[-1], high20)

    low60 = _safe_float(df["close"].rolling(60).min().iloc[-1], low20)

    range_pos = _safe_float(last.get("range_pos"), 0.5)

    rv_pct = _safe_float(last.get("rv20_pct"), 0.5)

    rsi_now = _safe_float(last.get("rsi14"), 50.0)

    drawdown = _safe_float(last.get("drawdown"), 0.0)

    trend_stack = int(trend_stack_count(df).iloc[-1])

    slope50 = _safe_float(last.get("slope_ma50"), 0.0)

    slope200 = _safe_float(last.get("slope_ma200"), 0.0)

    breadth = members.get("pct_above_200")

    breadth_now = _safe_float(breadth.iloc[-1]) if breadth is not None and len(breadth.dropna()) else np.nan

    breadth_mom = members.get("breadth_mom_200")

    breadth_mom_now = _safe_float(breadth_mom.iloc[-1]) if breadth_mom is not None and len(breadth_mom.dropna()) else np.nan



    def add_candidate(name, direction, suitability, condition_name, entry_value, stop_value, target1, target2, why, invalidation):

        mask = _condition_mask(state, condition_name) if condition_name else pd.Series(True, index=df.index)

        edge = summarize_setup_edge(state, mask, CFG.setup_preferred_horizon)

        rr = _risk_reward(entry_value, stop_value, target1, direction) if direction in ("LONG", "SHORT") else np.nan

        suitability = float(np.clip(suitability, 0.0, 1.0))

        sample_score = _clamp(edge["sample_size"] / 120) if edge["sample_size"] else 0.0

        edge_score = _clamp((edge["edge_mean"] + 0.04) / 0.08) if not pd.isna(edge["edge_mean"]) else 0.4

        win_score = _clamp((edge["win_rate"] - 0.40) / 0.25) if not pd.isna(edge["win_rate"]) else 0.4

        rr_score = _clamp(rr / 3.0) if not pd.isna(rr) else 0.35

        conviction = 100 * (0.45 * suitability + 0.20 * edge_score + 0.15 * win_score + 0.10 * rr_score + 0.10 * sample_score)

        status = "ACTIVE" if conviction >= 60 else "WATCHLIST" if conviction >= 42 else "LOW PRIORITY"

        plan = (

            "Risk management / hedge posture." if direction == "RISK"

            else f"Entry near {_fmt_price(entry_value)}, stop {_fmt_price(stop_value)}, first scale {_fmt_price(target1)}, stretch {_fmt_price(target2)}."

        )

        return {

            "Setup": name,

            "Status": status,

            "Direction": direction,

            "Conviction": round(conviction, 1),

            "Bias": regime.get("bias", "Mixed"),

            "Entry": _fmt_price(entry_value) if direction in ("LONG", "SHORT") else "Portfolio action",

            "Stop": _fmt_price(stop_value) if direction in ("LONG", "SHORT") else "Tighten risk",

            "Target 1": _fmt_price(target1) if direction in ("LONG", "SHORT") else "Protect capital",

            "Target 2": _fmt_price(target2) if direction in ("LONG", "SHORT") else "Wait for repair",

            "R/R": "N/A" if pd.isna(rr) else f"{rr:.2f}x",

            "Win Rate": _fmt_pct(edge["win_rate"]),

            "Avg 20D": _fmt_pct(edge["mean_return"]),

            "Edge vs Base": _fmt_pct(edge["edge_mean"]),

            "Samples": edge["sample_size"],

            "Plan": plan,

            "Why It Works": why,

            "Invalidation": invalidation,

        }



    candidates = []



    long_pullback_suit = _clamp(0.35 * ((trend_stack / max(len(CFG.ma_windows), 1))) + 0.25 * _clamp((range_pos - 0.45) / 0.35) + 0.20 * _clamp((composite + 4) / 8) + 0.20 * _clamp((0.02 - abs(price - ma20) / max(price, 1)) / 0.02 + 0.5))

    candidates.append(add_candidate(

        "Trend Pullback Long", "LONG", long_pullback_suit,

        f"Trend stack >= {CFG.trend_stack_bullish}/{len(CFG.ma_windows)} (strong uptrend)",

        max(ma20, price - 0.5 * atr), min(ma50, price - 2.0 * atr), max(high20, price + 1.8 * atr), max(high60, price + 3.5 * atr),

        "Best when price is still in the upper half of its range and the broader trend structure is intact.",

        "Price loses the 50DMA or medium-term momentum continues to worsen."

    ))



    breakout_suit = _clamp(0.40 * _clamp((range_pos - 0.70) / 0.25) + 0.30 * _clamp((0.45 - rv_pct) / 0.25) + 0.30 * _clamp((composite + 3) / 8))

    candidates.append(add_candidate(

        "Breakout Continuation Long", "LONG", breakout_suit,

        "SETUP: Near highs + compressed vol (breakout?)",

        max(high20, price * 1.002), max(ma20, price - 1.5 * atr), max(high60, price + 2.5 * atr), price + max(4.5 * atr, high20 - low20),

        "Compression near highs can resolve into a trend continuation move when volatility re-expands.",

        "Breakout fails back below the 20DMA / trigger zone."

    ))



    bounce_suit = _clamp(0.35 * _clamp((45 - rsi_now) / 20) + 0.25 * _clamp((rv_pct - 0.45) / 0.35) + 0.20 * _clamp((0.12 - abs(drawdown)) / 0.12 + 0.5) + 0.20 * _clamp(((breadth_mom_now if not pd.isna(breadth_mom_now) else -0.02) + 0.08) / 0.16))

    candidates.append(add_candidate(

        "Oversold Bounce Long", "LONG", bounce_suit,

        "SETUP: Oversold + high vol (bounce?)",

        price, min(low20, price - 1.5 * atr), min(ma20 if ma20 > 0 else price + 1.5 * atr, price + 1.8 * atr), ma50 if ma50 > 0 else price + 3.0 * atr,

        "Useful when downside extension is stretched and internals stop worsening.",

        "Fresh lows keep printing and breadth fails to stabilize."

    ))



    short_trend_suit = _clamp(0.35 * _clamp((2 - trend_stack) / 2) + 0.25 * _clamp((-composite - 1) / 6) + 0.20 * _clamp((0.55 - range_pos) / 0.35) + 0.20 * _clamp((price < ma20) + (price < ma50)) / 2)

    candidates.append(add_candidate(

        "Trend Failure Short", "SHORT", short_trend_suit,

        f"Trend stack <= {CFG.trend_stack_bearish}/{len(CFG.ma_windows)} (strong downtrend)",

        min(ma20, price + 0.5 * atr), max(ma50, price + 2.0 * atr), min(low20, price - 1.8 * atr), min(low60, price - 3.5 * atr),

        "Works best when rallies are failing beneath declining moving averages.",

        "Price reclaims the 20DMA and 50DMA with improving momentum."

    ))



    reclaim_suit = _clamp(0.30 * _clamp((slope200 + 0.002) / 0.01) + 0.30 * _clamp((0.10 + drawdown) / 0.10) + 0.20 * _clamp((42 - rsi_now) / 12) + 0.20 * _clamp((composite + 5) / 8))

    candidates.append(add_candidate(

        "200DMA Reclaim Long", "LONG", reclaim_suit,

        "All days",

        price, min(ma200, price - 1.5 * atr), max(ma20, price + 1.5 * atr), max(ma50, price + 3.0 * atr),

        "A rising long-term average with a pullback near the 200DMA often creates an asymmetric re-entry zone.",

        "Price loses the 200DMA decisively and long-term slope turns down."

    ))



    if not pd.isna(breadth_now):

        hedge_suit = _clamp(0.50 * _clamp((0.55 - breadth_now) / 0.20) + 0.30 * _clamp((trend_stack - 2) / 3) + 0.20 * _clamp((-breadth_mom_now if not pd.isna(breadth_mom_now) else 0.0) / 0.10))

        candidates.append(add_candidate(

            "Breadth Divergence Hedge", "RISK", hedge_suit,

            "SETUP: Weak breadth + trend looks ok",

            price, np.nan, np.nan, np.nan,

            "If the index is holding up while participation narrows, the right trade can be reducing exposure rather than pressing longs.",

            "Breadth expands back above trend-confirming levels."

        ))



    ideas = pd.DataFrame(candidates)

    ideas = ideas.sort_values(["Conviction", "Samples"], ascending=[False, False]).reset_index(drop=True)

    ideas.insert(0, "Rank", np.arange(1, len(ideas) + 1))

    return ideas.head(max(CFG.idea_limit, 6))




def generate_member_trade_ideas(state):
    prices = state.get("members_close", pd.DataFrame())
    if prices is None or prices.empty:
        return pd.DataFrame()

    market_name = state.get("market_name", "Unknown")
    constituent_meta = state.get("constituents_df", pd.DataFrame()).copy()
    if not constituent_meta.empty:
        constituent_meta = select_constituent_subset(constituent_meta, market_name, CFG.member_idea_limits)
        meta = constituent_meta.copy()
        meta["YahooSymbol"] = meta["Symbol"].apply(lambda s: yahoo_symbol_for_market(s, market_name))
        meta = meta.drop_duplicates(subset=["YahooSymbol"])
        candidate_cols = [c for c in meta["YahooSymbol"].tolist() if c in prices.columns]
    else:
        candidate_cols = list(prices.columns)
        meta = pd.DataFrame({"YahooSymbol": candidate_cols, "Symbol": candidate_cols, "Name": candidate_cols})

    if not candidate_cols:
        return pd.DataFrame()

    prices = prices[candidate_cols].copy()
    close = prices.ffill()
    ret1 = close.pct_change()
    ma20 = close.rolling(20, min_periods=20).mean()
    ma50 = close.rolling(50, min_periods=50).mean()
    ma200 = close.rolling(200, min_periods=200).mean()
    roc20 = close.pct_change(20)
    high252 = close.rolling(252, min_periods=252).max()
    low252 = close.rolling(252, min_periods=252).min()
    range_pos = (close - low252) / (high252 - low252).replace(0, np.nan)
    drawdown = close / close.cummax() - 1.0
    delta = close.diff()
    up = delta.clip(lower=0)
    down = -delta.clip(upper=0)
    rs = up.ewm(alpha=1 / CFG.rsi_period, adjust=False).mean() / down.ewm(alpha=1 / CFG.rsi_period, adjust=False).mean().replace(0, np.nan)
    rsi_df = 100 - (100 / (1 + rs))
    atr_proxy = close.pct_change().abs().rolling(14, min_periods=14).mean() * close

    latest = pd.DataFrame({
        "YahooSymbol": candidate_cols,
        "Close": close.iloc[-1].values,
        "MA20": ma20.iloc[-1].values,
        "MA50": ma50.iloc[-1].values,
        "MA200": ma200.iloc[-1].values,
        "ROC20": roc20.iloc[-1].values,
        "RangePos": range_pos.iloc[-1].values,
        "RSI14": rsi_df.iloc[-1].values,
        "Drawdown": drawdown.iloc[-1].values,
        "ATRProxy": atr_proxy.iloc[-1].values,
    })
    latest["TrendStack"] = (
        (latest["Close"] > latest["MA20"]).astype(int)
        + (latest["Close"] > latest["MA50"]).astype(int)
        + (latest["Close"] > latest["MA200"]).astype(int)
    )
    latest["Dist200"] = latest["Close"] / latest["MA200"] - 1.0
    latest["WeightNum"] = np.nan

    if not meta.empty:
        latest = latest.merge(meta[[c for c in meta.columns if c in ["YahooSymbol", "Symbol", "Name", "Weight"]]], on="YahooSymbol", how="left")
        if "Weight" in latest.columns:
            latest["WeightNum"] = latest["Weight"].apply(parse_weight_value)

    latest["LongScore"] = (
        0.35 * (latest["TrendStack"] / 3.0)
        + 0.20 * latest["RangePos"].clip(0, 1)
        + 0.20 * ((latest["ROC20"] + 0.10) / 0.20).clip(0, 1)
        + 0.15 * ((latest["Dist200"] + 0.10) / 0.20).clip(0, 1)
        + 0.10 * (1 - ((latest["RSI14"] - 55).abs() / 45).clip(0, 1))
    )
    latest["ShortScore"] = (
        0.35 * ((3 - latest["TrendStack"]) / 3.0)
        + 0.20 * (1 - latest["RangePos"].clip(0, 1))
        + 0.20 * ((-latest["ROC20"] + 0.10) / 0.20).clip(0, 1)
        + 0.15 * ((-latest["Dist200"] + 0.10) / 0.20).clip(0, 1)
        + 0.10 * ((latest["RSI14"] - 35).clip(0, 65) / 65)
    )
    latest["BounceScore"] = (
        0.40 * ((45 - latest["RSI14"]) / 25).clip(0, 1)
        + 0.30 * ((0.15 + latest["Drawdown"]) / 0.15).clip(0, 1)
        + 0.30 * ((0.85 - latest["RangePos"]) / 0.85).clip(0, 1)
    )

    def build_rows(df, direction, score_col, setup_label, n=10):
        rows = []
        ranked = df.sort_values([score_col, "WeightNum"], ascending=[False, False], na_position='last').head(n)
        for _, r in ranked.iterrows():
            atr = r["ATRProxy"] if pd.notna(r["ATRProxy"]) else r["Close"] * 0.03
            if direction == "LONG":
                stop = r["Close"] - 2 * atr
                target = max(r["MA20"] if pd.notna(r["MA20"]) else r["Close"] + 1.5 * atr, r["Close"] + 2 * atr)
                rr = _risk_reward(r["Close"], stop, target, "LONG")
            else:
                stop = r["Close"] + 2 * atr
                target = min(r["MA20"] if pd.notna(r["MA20"]) else r["Close"] - 1.5 * atr, r["Close"] - 2 * atr)
                rr = _risk_reward(r["Close"], stop, target, "SHORT")
            conviction = float(np.clip(r[score_col], 0, 1) * 100)
            rows.append({
                "Ticker": r.get("Symbol", r["YahooSymbol"]),
                "Name": r.get("Name", r["YahooSymbol"]),
                "Direction": direction,
                "Setup": setup_label,
                "Conviction": round(conviction, 1),
                "Close": _fmt_price(r["Close"]),
                "20D": _fmt_pct(r["ROC20"]),
                "RSI14": f"{r['RSI14']:.1f}" if pd.notna(r['RSI14']) else "N/A",
                "RangePos": f"{r['RangePos']:.2f}" if pd.notna(r['RangePos']) else "N/A",
                "Weight": r.get("Weight", "N/A"),
                "Stop": _fmt_price(stop),
                "Target": _fmt_price(target),
                "R/R": "N/A" if pd.isna(rr) else f"{rr:.2f}x",
            })
        return rows

    longs = build_rows(latest.dropna(subset=["Close"]), "LONG", "LongScore", "Constituent trend long", n=8)
    shorts = build_rows(latest.dropna(subset=["Close"]), "SHORT", "ShortScore", "Constituent weak / hedge short", n=6)
    bounces = build_rows(latest.dropna(subset=["Close"]), "LONG", "BounceScore", "Constituent oversold bounce watch", n=6)
    ideas = pd.DataFrame(longs + shorts + bounces)
    if ideas.empty:
        return ideas
    ideas = ideas.sort_values(["Conviction", "Direction"], ascending=[False, True]).drop_duplicates(subset=["Ticker", "Setup"]).reset_index(drop=True)
    ideas.insert(0, "Rank", np.arange(1, len(ideas) + 1))
    return ideas.head(20)


def fig_member_trade_ideas(state):
    ideas = state.get("member_trade_ideas_df", pd.DataFrame())
    if ideas is None or ideas.empty:
        return fig_table(
            pd.DataFrame([{"Message": "No constituent-level ideas available for this market."}]),
            title=f"Constituent Trade Ideas - {state['market_name']}",
            precision=2,
            max_rows=10,
        )
    return fig_table(
        ideas,
        title=f"Constituent Trade Ideas - {state['market_name']}",
        precision=2,
        max_rows=25,
    )


_build_state_base = build_state





def build_state(market_name: str, force_refresh: bool = False) -> Dict:

    cache_key = (market_name, CFG.lookback)

    if not force_refresh and cache_key in STATE_CACHE:

        _emit_status(f"{market_name}: using cached dashboard state")

        return STATE_CACHE[cache_key]



    _emit_status(f"{market_name}: assembling dashboard state")

    SETUP_EDGE_CACHE.clear()

    state = _build_state_base(market_name)

    _emit_status(f"{market_name}: scoring regime and trade ideas")

    state["regime"] = build_market_regime(state)

    state["trade_ideas_df"] = generate_trade_ideas(state)

    _emit_status(f"{market_name}: scanning constituent trade ideas")

    state["member_trade_ideas_df"] = generate_member_trade_ideas(state)

    state["quality"]["trade_ideas"] = int(len(state["trade_ideas_df"]))

    state["quality"]["member_trade_ideas"] = int(len(state["member_trade_ideas_df"]))

    STATE_CACHE[cache_key] = state

    _emit_status(f"{market_name}: ready")

    return state





def fig_exec_summary(state):

    regime = state.get("regime", {})

    df = state["index_feat"]

    if df.empty:

        return go.Figure()



    last = df.iloc[-1]

    ideas = state.get("trade_ideas_df", pd.DataFrame())

    best = ideas.iloc[0] if not ideas.empty else None

    rows = [

        ("Market", state["market_name"]),

        ("Ticker", state["ticker"]),

        ("Bias", regime.get("bias", "Mixed")),

        ("Composite", f"{regime.get('composite', 0):+.1f}/10"),

        ("Trend Stack", regime.get("trend_stack", "N/A")),

        ("Volatility", regime.get("vol_bucket", "N/A")),

        ("Tape", regime.get("tape", "N/A")),

        ("Participation", regime.get("participation", "N/A")),

        ("Last Close", _fmt_price(last.get("close"))),

        ("20D Return", _fmt_pct(last.get("roc_20d"))),

        ("Drawdown", _fmt_pct(last.get("drawdown"))),

        ("Best Setup", best["Setup"] if best is not None else "N/A"),

        ("Best Conviction", f"{best['Conviction']:.1f}" if best is not None else "N/A"),

    ]

    summary_df = pd.DataFrame(rows, columns=["Metric", "Value"])

    return fig_table(summary_df, title=f"Executive Summary ? {state['market_name']}", precision=2, max_rows=30)





def fig_trade_ideas(state, direction_filter="All", limit=None):

    ideas = state.get("trade_ideas_df", pd.DataFrame()).copy()

    if ideas.empty:

        return go.Figure()

    if direction_filter != "All":

        ideas = ideas[ideas["Direction"] == direction_filter]

    if limit is None:

        limit = CFG.idea_limit

    ideas = ideas.head(limit)

    return fig_table(

        ideas,

        title=f"Trade Ideas ? {state['market_name']} ranked by conviction, edge, and reward-to-risk",

        precision=2,

        max_rows=max(limit, 8),

    )





def fig_trade_conviction(state, direction_filter="All", limit=None):

    ideas = state.get("trade_ideas_df", pd.DataFrame()).copy()

    if ideas.empty:

        return go.Figure()

    if direction_filter != "All":

        ideas = ideas[ideas["Direction"] == direction_filter]

    if limit is None:

        limit = CFG.idea_limit

    ideas = ideas.head(limit)

    fig = go.Figure()

    colors = [COL["bull"] if d == "LONG" else COL["bear"] if d == "SHORT" else COL["yellow"] for d in ideas["Direction"]]

    fig.add_trace(

        go.Bar(

            x=ideas["Conviction"],

            y=ideas["Setup"],

            orientation="h",

            marker_color=colors,

            text=[f"{v:.1f}" for v in ideas["Conviction"]],

            textposition="outside",

        )

    )

    apply_layout(fig, title="Trade Setup Conviction Score", height=420 + 30 * len(ideas), show_legend=False)

    fig.update_xaxes(title="Conviction (0-100)")

    fig.update_yaxes(autorange="reversed")

    return fig





def cross_market_snapshot():

    rows = []

    for market_name, ticker in CFG.indices.items():

        try:

            close = load_or_download_index_close(ticker, CFG.lookback)

            feat = compute_index_features(close)

            if feat.empty:

                continue

            light_state = {

                "market_name": market_name,

                "ticker": ticker,

                "index_close": close,

                "index_feat": feat,

                "members_feat": {},

                "conditions": {},

                "quality": {},

            }

            regime = build_market_regime(light_state)

            ideas = generate_trade_ideas(light_state)

            last = feat.iloc[-1]

            best = ideas.iloc[0] if not ideas.empty else None

            rows.append(

                {

                    "Market": market_name,

                    "Bias": regime.get("bias", "Mixed"),

                    "Composite": regime.get("composite", np.nan),

                    "Best Setup": best["Setup"] if best is not None else "N/A",

                    "Best Conviction": best["Conviction"] if best is not None else np.nan,

                    "20D Return": _safe_float(last.get("roc_20d")) * 100,

                    "Drawdown": _safe_float(last.get("drawdown")) * 100,

                }

            )

        except Exception as exc:

            rows.append(

                {

                    "Market": market_name,

                    "Bias": "Load failed",

                    "Composite": np.nan,

                    "Best Setup": str(exc)[:60],

                    "Best Conviction": np.nan,

                    "20D Return": np.nan,

                    "Drawdown": np.nan,

                }

            )

    df = pd.DataFrame(rows)

    if df.empty:

        return go.Figure()

    df = df.sort_values(["Best Conviction", "Composite"], ascending=[False, False]).head(CFG.cross_market_limit)

    return fig_table(df, title="Cross-Market Opportunity Board", precision=2, max_rows=CFG.cross_market_limit + 2)





def fig_exec_summary(state):

    regime = state.get("regime", {})

    df = state["index_feat"]

    if df.empty:

        return go.Figure()



    last = df.iloc[-1]

    ideas = state.get("trade_ideas_df", pd.DataFrame())

    best = ideas.iloc[0] if not ideas.empty else None

    rows = [

        ("Market", state["market_name"]),

        ("Ticker", state["ticker"]),

        ("Bias", regime.get("bias", "Mixed")),

        ("Composite", f"{regime.get('composite', 0):+.1f}/10"),

        ("Trend Stack", regime.get("trend_stack", "N/A")),

        ("Volatility", regime.get("vol_bucket", "N/A")),

        ("Tape", regime.get("tape", "N/A")),

        ("Participation", regime.get("participation", "N/A")),

        ("Last Close", _fmt_price(last.get("close"))),

        ("20D Return", _fmt_pct(last.get("roc_20d"))),

        ("Drawdown", _fmt_pct(last.get("drawdown"))),

        ("Best Setup", best["Setup"] if best is not None else "N/A"),

        ("Best Conviction", f"{best['Conviction']:.1f}" if best is not None else "N/A"),

    ]

    summary_df = pd.DataFrame(rows, columns=["Metric", "Value"])

    return fig_table(summary_df, title=f"Executive Summary ? {state['market_name']}", precision=2, max_rows=30)





def fig_trade_ideas(state, direction_filter="All", limit=None):

    ideas = state.get("trade_ideas_df", pd.DataFrame()).copy()

    if ideas.empty:

        return go.Figure()

    if direction_filter != "All":

        ideas = ideas[ideas["Direction"] == direction_filter]

    if limit is None:

        limit = CFG.idea_limit

    ideas = ideas.head(limit)

    return fig_table(

        ideas,

        title=f"Trade Ideas ? {state['market_name']} ranked by conviction, edge, and reward-to-risk",

        precision=2,

        max_rows=max(limit, 8),

    )





def fig_trade_conviction(state, direction_filter="All", limit=None):

    ideas = state.get("trade_ideas_df", pd.DataFrame()).copy()

    if ideas.empty:

        return go.Figure()

    if direction_filter != "All":

        ideas = ideas[ideas["Direction"] == direction_filter]

    if limit is None:

        limit = CFG.idea_limit

    ideas = ideas.head(limit)

    fig = go.Figure()

    colors = [COL["bull"] if d == "LONG" else COL["bear"] if d == "SHORT" else COL["yellow"] for d in ideas["Direction"]]

    fig.add_trace(

        go.Bar(

            x=ideas["Conviction"],

            y=ideas["Setup"],

            orientation="h",

            marker_color=colors,

            text=[f"{v:.1f}" for v in ideas["Conviction"]],

            textposition="outside",

        )

    )

    apply_layout(fig, title="Trade Setup Conviction Score", height=420 + 30 * len(ideas), show_legend=False)

    fig.update_xaxes(title="Conviction (0-100)")

    fig.update_yaxes(autorange="reversed")

    return fig





def cross_market_snapshot():

    rows = []

    for market_name in CFG.indices.keys():

        try:

            s = build_state(market_name)

        except Exception as exc:

            rows.append(

                {

                    "Market": market_name,

                    "Bias": "Load failed",

                    "Composite": np.nan,

                    "Best Setup": str(exc)[:60],

                    "Best Conviction": np.nan,

                    "20D Return": np.nan,

                    "Drawdown": np.nan,

                }

            )

            continue



        regime = s.get("regime", {})

        last = s["index_feat"].iloc[-1] if not s["index_feat"].empty else pd.Series(dtype=float)

        ideas = s.get("trade_ideas_df", pd.DataFrame())

        best = ideas.iloc[0] if not ideas.empty else None

        rows.append(

            {

                "Market": market_name,

                "Bias": regime.get("bias", "Mixed"),

                "Composite": regime.get("composite", np.nan),

                "Best Setup": best["Setup"] if best is not None else "N/A",

                "Best Conviction": best["Conviction"] if best is not None else np.nan,

                "20D Return": _safe_float(last.get("roc_20d")) * 100,

                "Drawdown": _safe_float(last.get("drawdown")) * 100,

            }

        )



    df = pd.DataFrame(rows)

    if df.empty:

        return go.Figure()

    df = df.sort_values(["Best Conviction", "Composite"], ascending=[False, False]).head(CFG.cross_market_limit)

    return fig_table(df, title="Cross-Market Opportunity Board", precision=2, max_rows=CFG.cross_market_limit + 2)

In [ ]:
# ============================================================
# 13B) TACTICAL LAYERS AND EXECUTION FRAMEWORK
# ============================================================
import re

EXTENDED_STATE_CACHE = {}
_trade_engine_build_state_base = build_state


def _trade_engine_force_base_state(market_name, force_refresh=False):
    if force_refresh:
        try:
            STATE_CACHE.pop((market_name, CFG.lookback), None)
        except Exception:
            pass
    try:
        return _trade_engine_build_state_base(market_name, force_refresh=force_refresh)
    except TypeError:
        return _trade_engine_build_state_base(market_name)


def build_signal_drift(state):
    df = state.get("index_feat", pd.DataFrame())
    if df is None or df.empty or len(df) < 30:
        return pd.DataFrame()

    current_signals, composite_now = compute_signal_scores(state)
    checkpoints = {"5D": 5, "20D": 20}
    rows = []
    for key, sig in current_signals.items():
        row = {
            "Signal": key.replace("_", " ").title(),
            "Current": round(float(sig.get("score", 0.0)), 3),
            "Label": sig.get("label", "N/A"),
        }
        drift_values = []
        for label, lag in checkpoints.items():
            if len(df) <= lag:
                row[f"{label} Ago"] = np.nan
                row[f"Delta{label}"] = np.nan
                continue
            hist_state = dict(state)
            hist_state["index_feat"] = df.iloc[:-lag].copy()
            if state.get("members_feat"):
                hist_state["members_feat"] = {
                    k: (v.iloc[:-lag].copy() if hasattr(v, "iloc") else v)
                    for k, v in state.get("members_feat", {}).items()
                }
            if not state.get("sp500_thrust_mcc", pd.DataFrame()).empty:
                hist_state["sp500_thrust_mcc"] = state["sp500_thrust_mcc"].iloc[:-lag].copy()
            hist_signals, _ = compute_signal_scores(hist_state)
            past_score = float(hist_signals.get(key, {}).get("score", np.nan))
            row[f"{label} Ago"] = round(past_score, 3) if not pd.isna(past_score) else np.nan
            delta = float(sig.get("score", 0.0)) - past_score if not pd.isna(past_score) else np.nan
            row[f"Delta{label}"] = round(delta, 3) if not pd.isna(delta) else np.nan
            if not pd.isna(delta):
                drift_values.append(delta)
        drift_score = np.nanmean(drift_values) if drift_values else np.nan
        row["Drift Score"] = round(float(drift_score), 3) if not pd.isna(drift_score) else np.nan
        if pd.isna(drift_score):
            row["Drift"] = "Stable"
        elif drift_score >= 0.20:
            row["Drift"] = "Improving fast"
        elif drift_score >= 0.05:
            row["Drift"] = "Improving"
        elif drift_score <= -0.20:
            row["Drift"] = "Deteriorating fast"
        elif drift_score <= -0.05:
            row["Drift"] = "Deteriorating"
        else:
            row["Drift"] = "Stable"
        rows.append(row)
    drift_df = pd.DataFrame(rows)
    if drift_df.empty:
        return drift_df
    drift_df = drift_df.sort_values(["Drift Score", "Current"], ascending=[True, False], na_position="last").reset_index(drop=True)
    drift_df.attrs["summary"] = {
        "Composite now": round(float(composite_now), 2),
        "Improving signals": int((pd.to_numeric(drift_df["Drift Score"], errors="coerce") > 0.05).sum()),
        "Deteriorating signals": int((pd.to_numeric(drift_df["Drift Score"], errors="coerce") < -0.05).sum()),
    }
    return drift_df


def build_trade_playbook(state):
    ideas = state.get("trade_ideas_df", pd.DataFrame())
    member_ideas = state.get("member_action_board_df", state.get("member_trade_ideas_df", pd.DataFrame()))
    regime = state.get("regime", {})
    drift_df = state.get("signal_drift_df", pd.DataFrame())
    signals, composite = compute_signal_scores(state)
    playbook = []

    best_index = ideas.iloc[0] if ideas is not None and len(ideas) else None
    if best_index is not None:
        playbook.append({
            "Priority": 1,
            "Focus": "Primary Index Setup",
            "Action": f"{best_index['Setup']} ({best_index.get('Status', 'N/A')})",
            "Why": best_index.get("Why It Works", best_index.get("Plan", "")),
            "Trigger": best_index.get("Entry", "N/A"),
            "Risk": best_index.get("Stop", "N/A"),
            "Status": best_index.get("Direction", "N/A"),
        })

    risk_action = "Keep a balanced book and avoid oversized directional bets"
    risk_trigger = "Demand cleaner entries"
    risk_text = f"Composite backdrop is mixed at {composite:+.1f}/10."
    if composite <= -1.5:
        risk_action = "Trade smaller and prefer quicker profit-taking"
        risk_trigger = "Only add on confirmation"
        risk_text = f"Composite backdrop is still cautious at {composite:+.1f}/10."
    elif composite >= 1.5:
        risk_action = "Lean into trend-following ideas and buy pullbacks"
        risk_trigger = "Prefer continuation entries"
        risk_text = f"Composite backdrop supports risk-taking at {composite:+.1f}/10."
    playbook.append({
        "Priority": 2,
        "Focus": "Risk Posture",
        "Action": risk_action,
        "Why": risk_text,
        "Trigger": risk_trigger,
        "Risk": regime.get("vol_bucket", "Normal volatility"),
        "Status": regime.get("bias", "Mixed"),
    })

    if member_ideas is not None and len(member_ideas):
        top_stock = member_ideas.iloc[0]
        playbook.append({
            "Priority": 3,
            "Focus": "Best Constituent",
            "Action": f"{top_stock['Ticker']} - {top_stock['Setup']}",
            "Why": f"Top stock-level conviction at {top_stock['Conviction']:.1f}.",
            "Trigger": top_stock.get("Close", "N/A"),
            "Risk": top_stock.get("Stop", "N/A"),
            "Status": top_stock.get("Direction", "N/A"),
        })
        trade_now = int((member_ideas.get("Action", pd.Series(dtype=str)) == "Trade now").sum()) if "Action" in member_ideas.columns else 0
        playbook.append({
            "Priority": 4,
            "Focus": "Opportunity Breadth",
            "Action": f"{trade_now} constituents currently flagged as Trade now",
            "Why": "Use this as a gauge of whether opportunity is concentrated or broad across the member universe.",
            "Trigger": top_stock.get("Bucket", "N/A"),
            "Risk": "If only one bucket is working, stay selective",
            "Status": f"{len(member_ideas)} names ranked",
        })

    if drift_df is not None and len(drift_df):
        best = drift_df.sort_values("Drift Score", ascending=False).iloc[0]
        worst = drift_df.sort_values("Drift Score", ascending=True).iloc[0]
        playbook.append({
            "Priority": 5,
            "Focus": "Signal Drift",
            "Action": f"Best improving: {best['Signal']} | Weakest: {worst['Signal']}",
            "Why": "Recent signal change often matters as much as the absolute level.",
            "Trigger": f"{best['Drift']} / {worst['Drift']}",
            "Risk": "If deterioration spreads, reduce aggression",
            "Status": "Monitor change",
        })

    if "breadth_momentum" in signals and signals["breadth_momentum"].get("score", 0) <= -0.3:
        playbook.append({
            "Priority": 6,
            "Focus": "Breadth Warning",
            "Action": "Do not trust index strength blindly",
            "Why": signals["breadth_momentum"].get("detail", "Breadth is deteriorating."),
            "Trigger": signals["breadth_momentum"].get("raw", "N/A"),
            "Risk": "Favor selective longs and hedges",
            "Status": signals["breadth_momentum"].get("label", "Watch"),
        })

    return pd.DataFrame(playbook).sort_values("Priority").reset_index(drop=True)


def build_execution_checklist(state):
    ideas = state.get("trade_ideas_df", pd.DataFrame())
    member_ideas = state.get("member_action_board_df", state.get("member_trade_ideas_df", pd.DataFrame()))
    checklist = []

    if ideas is not None and len(ideas):
        top = ideas.iloc[0]
        stage = str(top.get("Status", "WATCHLIST"))
        if stage == "ACTIVE":
            timing = "Actionable now"
        elif stage == "WATCHLIST":
            timing = "Close watch"
        else:
            timing = "Wait for better alignment"
        checklist.append({
            "Layer": "Index",
            "Item": top.get("Setup", "Primary setup"),
            "Timing": timing,
            "Risk Level": "Controlled" if str(top.get("R/R", "")).startswith(("2", "3")) else "Moderate",
            "What To Do": top.get("Plan", "N/A"),
            "Why": top.get("Why It Works", "N/A"),
        })

    if member_ideas is not None and len(member_ideas):
        top_long = member_ideas[member_ideas["Direction"] == "LONG"].head(1)
        top_short = member_ideas[member_ideas["Direction"] == "SHORT"].head(1)
        if len(top_long):
            r = top_long.iloc[0]
            checklist.append({
                "Layer": "Stock Long",
                "Item": r["Ticker"],
                "Timing": r.get("Action", "Watch"),
                "Risk Level": "Moderate",
                "What To Do": f"Use stop {r.get('Stop', 'N/A')} and target {r.get('Target', 'N/A')}",
                "Why": r.get("Reason", "N/A"),
            })
        if len(top_short):
            r = top_short.iloc[0]
            checklist.append({
                "Layer": "Stock Short/Hedge",
                "Item": r["Ticker"],
                "Timing": r.get("Action", "Watch"),
                "Risk Level": "Moderate",
                "What To Do": f"Use stop {r.get('Stop', 'N/A')} and target {r.get('Target', 'N/A')}",
                "Why": r.get("Reason", "N/A"),
            })

    return pd.DataFrame(checklist)


def _parse_price_text(value):
    if value is None:
        return np.nan
    text = str(value).strip()
    if not text or text.upper() == "N/A":
        return np.nan
    match = re.search(r"[-+]?\d+(?:\.\d+)?", text.replace(",", ""))
    return float(match.group(0)) if match else np.nan


def build_position_sizing(state):
    rows = []
    ideas = state.get("trade_ideas_df", pd.DataFrame())
    member_ideas = state.get("member_action_board_df", state.get("member_trade_ideas_df", pd.DataFrame()))
    drift = state.get("signal_drift_df", pd.DataFrame())
    positive_drift = int((pd.to_numeric(drift.get("Delta5D"), errors="coerce") > 0.03).sum()) if drift is not None and len(drift) else 0
    negative_drift = int((pd.to_numeric(drift.get("Delta5D"), errors="coerce") < -0.03).sum()) if drift is not None and len(drift) else 0

    def timing_label(direction):
        direction = str(direction).upper()
        if direction == "LONG":
            if positive_drift >= max(2, negative_drift + 1):
                return "Scale on strength"
            if negative_drift >= max(2, positive_drift + 1):
                return "Start smaller"
            return "Build selectively"
        if negative_drift >= max(2, positive_drift + 1):
            return "Press on weakness"
        if positive_drift >= max(2, negative_drift + 1):
            return "Keep hedge smaller"
        return "Scale selectively"

    def add_row(layer, item, direction, conviction, entry_text, stop_text):
        entry = _parse_price_text(entry_text)
        stop = _parse_price_text(stop_text)
        if pd.isna(entry) or pd.isna(stop) or entry == 0:
            risk_pct = np.nan
        else:
            risk_pct = abs(entry - stop) / entry
        if conviction >= 80:
            risk_unit = 1.00
        elif conviction >= 65:
            risk_unit = 0.75
        elif conviction >= 50:
            risk_unit = 0.50
        else:
            risk_unit = 0.25
        if not pd.isna(risk_pct) and risk_pct > 0.08:
            risk_unit *= 0.60
        elif not pd.isna(risk_pct) and risk_pct > 0.05:
            risk_unit *= 0.80
        expression = (
            "Full starter" if risk_unit >= 0.90 else
            "Three-quarter starter" if risk_unit >= 0.70 else
            "Half-size probe" if risk_unit >= 0.45 else
            "Quarter-size tracker"
        )
        rows.append({
            "Layer": layer,
            "Item": item,
            "Direction": direction,
            "Conviction": round(float(conviction), 1),
            "Entry": entry_text,
            "Stop": stop_text,
            "RiskPct": round(float(risk_pct) * 100, 2) if not pd.isna(risk_pct) else np.nan,
            "SizeUnit": round(float(risk_unit), 2),
            "Expression": expression,
            "Timing": timing_label(direction),
        })

    if ideas is not None and len(ideas):
        for _, row in ideas.head(3).iterrows():
            add_row("Index", row.get("Setup", "Primary setup"), row.get("Direction", "LONG"), float(row.get("Conviction", 0)), row.get("Entry", "N/A"), row.get("Stop", "N/A"))
    if member_ideas is not None and len(member_ideas):
        for _, row in member_ideas.head(5).iterrows():
            add_row("Constituent", row.get("Ticker", "Member"), row.get("Direction", "LONG"), float(row.get("Conviction", 0)), row.get("Close", row.get("Entry", "N/A")), row.get("Stop", "N/A"))
    return pd.DataFrame(rows)


def fig_signal_drift(state):
    drift_df = state.get("signal_drift_df", pd.DataFrame())
    if drift_df is None or drift_df.empty:
        return fig_table(pd.DataFrame([{"Message": "Not enough history for signal drift analysis."}]), title="Signal Drift", precision=2, max_rows=10)
    summary = drift_df.attrs.get("summary", {})
    title = f"Signal Drift - {state['market_name']} | Composite {summary.get('Composite now', 'N/A')} | Improving {summary.get('Improving signals', 0)} | Deteriorating {summary.get('Deteriorating signals', 0)}"
    return fig_table(drift_df, title=title, precision=3, max_rows=15)


def fig_trade_playbook(state):
    playbook = state.get("trade_playbook_df", pd.DataFrame())
    if playbook is None or playbook.empty:
        return fig_table(pd.DataFrame([{"Message": "No playbook actions generated."}]), title="Trade Playbook", precision=2, max_rows=10)
    return fig_table(playbook, title=f"Trade Playbook - {state['market_name']}", precision=2, max_rows=12)


def fig_execution_checklist(state):
    checklist = state.get("execution_checklist_df", pd.DataFrame())
    if checklist is None or checklist.empty:
        return fig_table(pd.DataFrame([{"Message": "No execution checklist available."}]), title="Execution Checklist", precision=2, max_rows=10)
    return fig_table(checklist, title=f"Execution Checklist - {state['market_name']}", precision=2, max_rows=10)


def fig_position_sizing(state):
    sizing = state.get("position_sizing_df", pd.DataFrame())
    if sizing is None or sizing.empty:
        return fig_table(pd.DataFrame([{"Message": "No position-sizing guidance available."}]), title="Position Sizing", precision=2, max_rows=10)
    return fig_table(sizing, title=f"Position Sizing / Expression - {state['market_name']}", precision=2, max_rows=10)




def build_member_action_board(state):
    ideas = state.get("member_trade_ideas_df", pd.DataFrame())
    if ideas is None or ideas.empty:
        return pd.DataFrame()

    board = ideas.copy()
    sector_map = state.get("sector_map", {}) or {}
    if not sector_map and isinstance(state.get("constituents_df"), pd.DataFrame) and len(state.get("constituents_df")):
        tmp = state["constituents_df"].copy()
        if "Sector" in tmp.columns and "Symbol" in tmp.columns:
            sector_map = dict(zip(tmp["Symbol"].astype(str), tmp["Sector"].astype(str)))

    board["Sector"] = board["Ticker"].astype(str).map(sector_map).fillna("Unknown")
    board["BaseConviction"] = pd.to_numeric(board.get("Conviction"), errors="coerce").fillna(0.0)
    board["SectorRank"] = board.groupby(["Direction", "Sector"]).cumcount()
    board["BucketRank"] = board.groupby(["Direction", "Setup"]).cumcount()
    board["CrowdingPenalty"] = board["SectorRank"] * 8.0 + board["BucketRank"] * 3.0
    board["DiversifiedScore"] = (board["BaseConviction"] - board["CrowdingPenalty"]).round(1)

    selected = []
    sector_caps = {"LONG": 2, "SHORT": 1}
    sector_counts = {}
    for _, row in board.sort_values(["DiversifiedScore", "BaseConviction"], ascending=[False, False]).iterrows():
        direction = str(row.get("Direction", "LONG")).upper()
        sector = row.get("Sector", "Unknown")
        key = (direction, sector)
        cap = sector_caps.get(direction, 1)
        if sector_counts.get(key, 0) >= cap:
            continue
        sector_counts[key] = sector_counts.get(key, 0) + 1
        selected.append(row)
        if len(selected) >= 12:
            break

    if not selected:
        return pd.DataFrame()

    action_board = pd.DataFrame(selected).reset_index(drop=True)
    action_board["ActionTier"] = np.where(
        action_board["DiversifiedScore"] >= 85,
        "Best in sector",
        np.where(
            action_board["DiversifiedScore"] >= 70,
            "Add selectively",
            np.where(action_board["Direction"].eq("SHORT"), "Hedge candidate", "Crowded watch"),
        ),
    )
    action_board["WhyNow"] = action_board.apply(
        lambda r: f"{r['Sector']} | {r.get('Setup', 'Setup')} | score {r['DiversifiedScore']:.1f}",
        axis=1,
    )
    preferred = [
        "Ticker", "Sector", "Direction", "ActionTier", "DiversifiedScore", "Conviction", "Setup",
        "WhyNow", "Close", "20D", "RSI14", "Stop", "Target", "R/R"
    ]
    cols = [c for c in preferred if c in action_board.columns] + [c for c in action_board.columns if c not in preferred]
    action_board = action_board[cols]
    if "Rank" in action_board.columns:
        action_board = action_board.drop(columns=["Rank"])
    action_board.insert(0, "Rank", np.arange(1, len(action_board) + 1))
    action_board.attrs["summary"] = {
        "Unique sectors": int(action_board["Sector"].nunique()),
        "Longs": int((action_board["Direction"] == "LONG").sum()),
        "Shorts": int((action_board["Direction"] == "SHORT").sum()),
    }
    return action_board


def fig_member_action_board(state):
    board = state.get("member_action_board_df", pd.DataFrame())
    if board is None or board.empty:
        return fig_table(pd.DataFrame([{"Message": "No diversified action board available."}]), title="Action Board", precision=2, max_rows=10)
    summary = board.attrs.get("summary", {})
    title = f"Action Board - {state['market_name']} | Sectors {summary.get('Unique sectors', 0)} | Longs {summary.get('Longs', 0)} | Shorts {summary.get('Shorts', 0)}"
    return fig_table(board, title=title, precision=2, max_rows=15)

def build_trade_book(state):
    ideas = state.get("trade_ideas_df", pd.DataFrame())
    member_ideas = state.get("member_action_board_df", state.get("member_trade_ideas_df", pd.DataFrame()))
    regime = state.get("regime", {})
    drift = state.get("signal_drift_df", pd.DataFrame())
    rows = []

    if ideas is None or ideas.empty or member_ideas is None or member_ideas.empty:
        return pd.DataFrame()

    positive_drift = int((pd.to_numeric(drift.get("Delta5D"), errors="coerce") > 0.03).sum()) if drift is not None and len(drift) else 0
    negative_drift = int((pd.to_numeric(drift.get("Delta5D"), errors="coerce") < -0.03).sum()) if drift is not None and len(drift) else 0
    composite = float(regime.get("composite", 0.0) if regime else 0.0)

    top_index = ideas.iloc[0]
    rows.append({
        "Sleeve": "Index Core",
        "Instrument": state.get("ticker", state.get("market_name", "Index")),
        "Theme": top_index.get("Setup", "Primary setup"),
        "Direction": top_index.get("Direction", "LONG"),
        "Conviction": round(float(top_index.get("Conviction", 0.0)), 1),
        "BookWeight": 35.0,
        "Role": "Anchor exposure",
        "Why": top_index.get("Why It Works", top_index.get("Plan", "N/A")),
    })

    long_names = member_ideas[member_ideas["Direction"] == "LONG"].copy() if "Direction" in member_ideas.columns else pd.DataFrame()
    short_names = member_ideas[member_ideas["Direction"] == "SHORT"].copy() if "Direction" in member_ideas.columns else pd.DataFrame()

    def add_sleeve(df, sleeve, base_weight, limit):
        if df is None or df.empty:
            return 0.0
        picks = df.sort_values(["Conviction"], ascending=[False]).head(limit).copy()
        total = pd.to_numeric(picks["Conviction"], errors="coerce").fillna(0).sum()
        used = 0.0
        for _, row in picks.iterrows():
            share = (float(row.get("Conviction", 0.0)) / total) if total > 0 else (1.0 / max(len(picks), 1))
            weight = round(base_weight * share, 1)
            used += weight
            rows.append({
                "Sleeve": sleeve,
                "Instrument": row.get("Ticker", row.get("Name", "Name")),
                "Theme": row.get("Setup", row.get("Bucket", "Constituent idea")),
                "Direction": row.get("Direction", "LONG"),
                "Conviction": round(float(row.get("Conviction", 0.0)), 1),
                "BookWeight": weight,
                "Role": row.get("Action", "Watch"),
                "Why": row.get("Reason", "N/A"),
            })
        return used

    long_budget = 45.0 if composite >= -1.0 else 35.0
    short_budget = 10.0 if composite >= 1.5 else 20.0 if composite >= -1.0 else 30.0
    if positive_drift >= negative_drift + 2:
        long_budget += 5.0
        short_budget -= 5.0
    elif negative_drift >= positive_drift + 2:
        long_budget -= 5.0
        short_budget += 5.0
    long_budget = max(20.0, min(55.0, long_budget))
    short_budget = max(5.0, min(35.0, short_budget))

    long_used = add_sleeve(long_names, "Long Ideas", long_budget, 4)
    short_used = add_sleeve(short_names, "Short / Hedge", short_budget, 2)

    cash_weight = round(max(0.0, 100.0 - 35.0 - long_used - short_used), 1)
    rows.append({
        "Sleeve": "Cash Buffer",
        "Instrument": "Cash / dry powder",
        "Theme": "Optionality reserve",
        "Direction": "NEUTRAL",
        "Conviction": 0.0,
        "BookWeight": cash_weight,
        "Role": "Wait for better entries" if cash_weight >= 10 else "Mostly deployed",
        "Why": f"Composite {composite:+.1f}, improving drift {positive_drift}, deteriorating drift {negative_drift}.",
    })

    book = pd.DataFrame(rows)
    if not book.empty:
        book["BookWeight"] = pd.to_numeric(book["BookWeight"], errors="coerce").round(1)
        book = book.sort_values(["Sleeve", "BookWeight", "Conviction"], ascending=[True, False, False]).reset_index(drop=True)
        gross_long = float(book.loc[book["Direction"] == "LONG", "BookWeight"].sum())
        gross_short = float(book.loc[book["Direction"] == "SHORT", "BookWeight"].sum())
        net = gross_long - gross_short
        book.attrs["summary"] = {
            "Gross Long": round(gross_long, 1),
            "Gross Short": round(gross_short, 1),
            "Net": round(net, 1),
            "Cash": round(float(book.loc[book["Direction"] == "NEUTRAL", "BookWeight"].sum()), 1),
        }
    return book


def fig_trade_book(state):
    book = state.get("trade_book_df", pd.DataFrame())
    if book is None or book.empty:
        return fig_table(pd.DataFrame([{"Message": "No trade book available."}]), title="Trade Book", precision=2, max_rows=10)
    summary = book.attrs.get("summary", {})
    title = f"Trade Book - {state['market_name']} | Gross Long {summary.get('Gross Long', 0):.1f}% | Gross Short {summary.get('Gross Short', 0):.1f}% | Net {summary.get('Net', 0):+.1f}% | Cash {summary.get('Cash', 0):.1f}%"
    return fig_table(book, title=title, precision=2, max_rows=12)



def build_scenario_deck(state):
    ideas = state.get("trade_ideas_df", pd.DataFrame())
    action_board = state.get("member_action_board_df", pd.DataFrame())
    drift = state.get("signal_drift_df", pd.DataFrame())
    regime = state.get("regime", {}) or {}
    df = state.get("index_feat", pd.DataFrame())
    if df is None or df.empty:
        return pd.DataFrame()

    last = df.iloc[-1]
    close = _safe_float(last.get("close"))
    ma20 = _safe_float(last.get("ma20"), close)
    ma50 = _safe_float(last.get("ma50"), close)
    high20 = _safe_float(df["close"].rolling(20).max().iloc[-1], close)
    low20 = _safe_float(df["close"].rolling(20).min().iloc[-1], close)
    breadth = state.get("members_feat", {}).get("pct_above_200")
    breadth_now = _safe_float(breadth.iloc[-1]) if breadth is not None and len(breadth.dropna()) else np.nan
    improving = int((pd.to_numeric(drift.get("Delta5D"), errors="coerce") > 0.03).sum()) if drift is not None and len(drift) else 0
    deteriorating = int((pd.to_numeric(drift.get("Delta5D"), errors="coerce") < -0.03).sum()) if drift is not None and len(drift) else 0

    best_long = action_board[action_board.get("Direction", pd.Series(dtype=str)) == "LONG"].head(2) if len(action_board) else pd.DataFrame()
    best_short = action_board[action_board.get("Direction", pd.Series(dtype=str)) == "SHORT"].head(2) if len(action_board) else pd.DataFrame()
    top_index = ideas.iloc[0] if ideas is not None and len(ideas) else None
    bearish_index = ideas[ideas.get("Direction", pd.Series(dtype=str)) == "SHORT"].head(1) if ideas is not None and len(ideas) else pd.DataFrame()

    rows = []
    rows.append({
        "Scenario": "Bull continuation",
        "Trigger": f"Hold above {_fmt_price(max(ma20, ma50))} and clear {_fmt_price(high20)}",
        "Posture": "Add risk and let leaders work",
        "Index Expression": top_index.get("Setup", "Breakout continuation") if top_index is not None else "N/A",
        "Constituent Expression": ", ".join(best_long["Ticker"].head(2).tolist()) if len(best_long) else "Use strongest longs",
        "Hedge": "Keep hedges lighter and trail stops higher",
        "Why": f"Improving drift {improving} vs deteriorating {deteriorating}; composite {regime.get('composite', 0):+.1f}.",
    })
    rows.append({
        "Scenario": "Range / chop",
        "Trigger": f"Stay between {_fmt_price(low20)} and {_fmt_price(high20)}",
        "Posture": "Trade smaller and rotate selectively",
        "Index Expression": "Favor pullbacks over breakouts",
        "Constituent Expression": ", ".join(action_board["Ticker"].head(3).tolist()) if len(action_board) else "Use diversified action board",
        "Hedge": "Keep partial hedges on if breadth remains mixed",
        "Why": f"Tape likely remains tactical when price chops around the 20DMA at {_fmt_price(ma20)}.",
    })
    rows.append({
        "Scenario": "Bear acceleration",
        "Trigger": f"Lose {_fmt_price(ma50)} and press below {_fmt_price(low20)}",
        "Posture": "Cut gross, raise cash, lean on hedges",
        "Index Expression": bearish_index.iloc[0]["Setup"] if len(bearish_index) else "Trend failure short / hedge posture",
        "Constituent Expression": ", ".join(best_short["Ticker"].head(2).tolist()) if len(best_short) else "Use weakest names / hedges",
        "Hedge": "Increase hedge sleeve and stop forcing longs",
        "Why": f"Breadth {breadth_now:.0%} above 200DMA" if not pd.isna(breadth_now) else f"Deteriorating drift {deteriorating} argues for defense if support breaks.",
    })
    deck = pd.DataFrame(rows)
    deck.attrs["summary"] = {
        "Composite": round(float(regime.get("composite", 0.0)), 1),
        "Improving": improving,
        "Deteriorating": deteriorating,
    }
    return deck


def fig_scenario_deck(state):
    deck = state.get("scenario_deck_df", pd.DataFrame())
    if deck is None or deck.empty:
        return fig_table(pd.DataFrame([{"Message": "No scenario deck available."}]), title="Scenario Deck", precision=2, max_rows=10)
    summary = deck.attrs.get("summary", {})
    title = f"Scenario Deck - {state['market_name']} | Composite {summary.get('Composite', 0):+.1f} | Improving {summary.get('Improving', 0)} | Deteriorating {summary.get('Deteriorating', 0)}"
    return fig_table(deck, title=title, precision=2, max_rows=10)



def build_regime_radar(state):
    regime = state.get("regime", {}) or {}
    drift = state.get("signal_drift_df", pd.DataFrame())
    ideas = state.get("trade_ideas_df", pd.DataFrame())
    df = state.get("index_feat", pd.DataFrame())
    members = state.get("members_feat", {}) or {}
    if df is None or df.empty:
        return pd.DataFrame()

    last = df.iloc[-1]
    composite = float(regime.get("composite", 0.0))
    trend_stack_text = str(regime.get("trend_stack", "0/5"))
    try:
        trend_num = int(trend_stack_text.split('/')[0])
        trend_den = max(1, int(trend_stack_text.split('/')[1]))
    except Exception:
        trend_num, trend_den = 0, 5
    trend_score = trend_num / trend_den
    breadth = members.get("pct_above_200")
    breadth_now = _safe_float(breadth.iloc[-1]) if breadth is not None and len(breadth.dropna()) else 0.5
    rv_pct = _safe_float(last.get("rv20_pct"), 0.5)
    drawdown = abs(_safe_float(last.get("drawdown"), 0.0))
    improving = int((pd.to_numeric(drift.get("Delta5D"), errors="coerce") > 0.03).sum()) if drift is not None and len(drift) else 0
    deteriorating = int((pd.to_numeric(drift.get("Delta5D"), errors="coerce") < -0.03).sum()) if drift is not None and len(drift) else 0
    top_setup = ideas.iloc[0]["Setup"] if ideas is not None and len(ideas) else "N/A"

    bull_raw = 0.38 * ((composite + 6) / 12) + 0.27 * trend_score + 0.20 * breadth_now + 0.15 * min(1.0, improving / 6.0)
    bear_raw = 0.34 * ((6 - composite) / 12) + 0.24 * (1 - trend_score) + 0.20 * (1 - breadth_now) + 0.12 * min(1.0, deteriorating / 6.0) + 0.10 * min(1.0, drawdown / 0.20)
    bull_raw = max(0.05, bull_raw * (1.05 if rv_pct < 0.45 else 0.92 if rv_pct > 0.80 else 1.0))
    bear_raw = max(0.05, bear_raw * (1.08 if rv_pct > 0.80 else 0.95 if rv_pct < 0.35 else 1.0))
    base_raw = max(0.05, 1.0 - 0.55 * abs(composite) / 6.0 - 0.20 * abs(breadth_now - 0.5) - 0.10 * abs(improving - deteriorating) / 8.0)
    total = bull_raw + bear_raw + base_raw
    bull = 100 * bull_raw / total
    base = 100 * base_raw / total
    bear = 100 * bear_raw / total

    if bull >= max(base, bear):
        posture = "Pro-risk"
        action = "Bias toward longs, keep hedges tactical, and use pullbacks to add."
    elif bear >= max(bull, base):
        posture = "Defensive"
        action = "Run smaller gross, prioritize hedges, and avoid forcing breakout longs."
    else:
        posture = "Tactical / balanced"
        action = "Trade selectively, recycle risk quickly, and keep dry powder for resolution."

    radar = pd.DataFrame([
        {"Regime": "Bull", "Probability": round(bull, 1), "Drivers": f"Composite {composite:+.1f}, trend {trend_num}/{trend_den}, breadth {breadth_now:.0%}"},
        {"Regime": "Base", "Probability": round(base, 1), "Drivers": f"Vol pctile {rv_pct:.0%}, mixed drift {improving}/{deteriorating}"},
        {"Regime": "Bear", "Probability": round(bear, 1), "Drivers": f"Drawdown {drawdown:.1%}, deteriorating drift {deteriorating}"},
    ])
    radar.attrs["summary"] = {
        "Posture": posture,
        "Action": action,
        "Top setup": top_setup,
    }
    return radar


def fig_regime_radar(state):
    radar = state.get("regime_radar_df", pd.DataFrame())
    if radar is None or radar.empty:
        return fig_table(pd.DataFrame([{"Message": "No regime radar available."}]), title="Regime Radar", precision=2, max_rows=10)
    summary = radar.attrs.get("summary", {})
    title = f"Regime Radar - {state['market_name']} | {summary.get('Posture', 'Mixed')} | {summary.get('Top setup', 'N/A')}"
    table = radar.copy()
    table["Probability"] = table["Probability"].map(lambda x: f"{x:.1f}%")
    table["Action"] = [summary.get("Action", "")] + ["", ""]
    return fig_table(table, title=title, precision=2, max_rows=10)



def build_leadership_map(state):
    action_board = state.get("member_action_board_df", pd.DataFrame())
    member_ideas = state.get("member_trade_ideas_df", pd.DataFrame())
    constituents = state.get("constituents_df", pd.DataFrame())
    sector_map = state.get("sector_map", {}) or {}
    if action_board is None or action_board.empty:
        return pd.DataFrame()

    if not sector_map and isinstance(constituents, pd.DataFrame) and len(constituents):
        if "Symbol" in constituents.columns and "Sector" in constituents.columns:
            sector_map = dict(zip(constituents["Symbol"].astype(str), constituents["Sector"].astype(str)))

    df = action_board.copy()
    if "Sector" not in df.columns:
        df["Sector"] = df["Ticker"].astype(str).map(sector_map).fillna("Unknown")
    df["ConvictionNum"] = pd.to_numeric(df.get("Conviction"), errors="coerce").fillna(0.0)
    df["DiversifiedScoreNum"] = pd.to_numeric(df.get("DiversifiedScore"), errors="coerce").fillna(df["ConvictionNum"])

    full = member_ideas.copy() if isinstance(member_ideas, pd.DataFrame) else pd.DataFrame()
    if len(full):
        if "Sector" not in full.columns:
            full["Sector"] = full["Ticker"].astype(str).map(sector_map).fillna("Unknown")
        full["ConvictionNum"] = pd.to_numeric(full.get("Conviction"), errors="coerce").fillna(0.0)
    else:
        full = df.copy()

    summary = full.groupby("Sector", dropna=False).agg(
        Names=("Ticker", "count"),
        AvgConviction=("ConvictionNum", "mean"),
    ).reset_index()

    board_summary = df.groupby("Sector", dropna=False).agg(
        ActionNames=("Ticker", "count"),
        Longs=("Direction", lambda s: int((s.astype(str).str.upper() == "LONG").sum())),
        Shorts=("Direction", lambda s: int((s.astype(str).str.upper() == "SHORT").sum())),
        BestTicker=("Ticker", "first"),
        BestScore=("DiversifiedScoreNum", "max"),
    ).reset_index()

    leadership = board_summary.merge(summary, on="Sector", how="left")
    leadership["AvgConviction"] = leadership["AvgConviction"].fillna(0).round(1)
    leadership["BestScore"] = leadership["BestScore"].fillna(0).round(1)
    leadership["LeadershipScore"] = (
        0.45 * leadership["BestScore"]
        + 0.30 * leadership["AvgConviction"]
        + 12.0 * leadership["Longs"]
        - 8.0 * leadership["Shorts"]
    ).round(1)
    leadership["Stance"] = np.where(
        leadership["LeadershipScore"] >= 70,
        "Leader",
        np.where(leadership["LeadershipScore"] >= 45, "Supportive", np.where(leadership["Shorts"] > leadership["Longs"], "Lagging", "Mixed")),
    )
    leadership = leadership.sort_values(["LeadershipScore", "BestScore"], ascending=[False, False]).reset_index(drop=True)
    leadership.insert(0, "Rank", np.arange(1, len(leadership) + 1))
    leadership.attrs["summary"] = {
        "Leaders": int((leadership["Stance"] == "Leader").sum()),
        "Lagging": int((leadership["Stance"] == "Lagging").sum()),
        "Top sector": leadership.iloc[0]["Sector"] if len(leadership) else "N/A",
    }
    return leadership


def fig_leadership_map(state):
    leadership = state.get("leadership_map_df", pd.DataFrame())
    if leadership is None or leadership.empty:
        return fig_table(pd.DataFrame([{"Message": "No leadership map available."}]), title="Leadership Map", precision=2, max_rows=10)
    summary = leadership.attrs.get("summary", {})
    title = f"Leadership Map - {state['market_name']} | Leaders {summary.get('Leaders', 0)} | Lagging {summary.get('Lagging', 0)} | Top {summary.get('Top sector', 'N/A')}"
    return fig_table(leadership, title=title, precision=2, max_rows=15)



def build_tripwire_monitor(state):
    df = state.get("index_feat", pd.DataFrame())
    regime = state.get("regime", {}) or {}
    drift = state.get("signal_drift_df", pd.DataFrame())
    members = state.get("members_feat", {}) or {}
    if df is None or df.empty:
        return pd.DataFrame()

    last = df.iloc[-1]
    close = _safe_float(last.get("close"))
    ma20 = _safe_float(last.get("ma20"), close)
    ma50 = _safe_float(last.get("ma50"), close)
    ma200 = _safe_float(last.get("ma200"), close)
    rv_pct = _safe_float(last.get("rv20_pct"), 0.5)
    drawdown = _safe_float(last.get("drawdown"), 0.0)
    breadth = members.get("pct_above_200")
    breadth_now = _safe_float(breadth.iloc[-1]) if breadth is not None and len(breadth.dropna()) else np.nan
    deteriorating = int((pd.to_numeric(drift.get("Delta5D"), errors="coerce") < -0.03).sum()) if drift is not None and len(drift) else 0
    composite = float(regime.get("composite", 0.0))

    def classify(triggered, near):
        if triggered:
            return "TRIGGERED"
        if near:
            return "NEAR"
        return "CLEAR"

    rows = []
    rows.append({
        "Tripwire": "Lose 20DMA",
        "Now": _fmt_price(close),
        "Threshold": _fmt_price(ma20),
        "Status": classify(close < ma20, close < ma20 * 1.01),
        "Risk": "Short-term trend damage",
        "Response": "Reduce chase risk and demand better entries.",
    })
    rows.append({
        "Tripwire": "Lose 50DMA",
        "Now": _fmt_price(close),
        "Threshold": _fmt_price(ma50),
        "Status": classify(close < ma50, close < ma50 * 1.01),
        "Risk": "Intermediate trend failure",
        "Response": "Cut gross exposure and raise hedge ratio.",
    })
    rows.append({
        "Tripwire": "Lose 200DMA",
        "Now": _fmt_price(close),
        "Threshold": _fmt_price(ma200),
        "Status": classify(close < ma200, close < ma200 * 1.01),
        "Risk": "Long-term regime deterioration",
        "Response": "Shift from pro-risk to defense and favor cash.",
    })
    rows.append({
        "Tripwire": "Volatility shock",
        "Now": f"{rv_pct:.0%}",
        "Threshold": ">= 80% pctile",
        "Status": classify(rv_pct >= 0.80, rv_pct >= 0.70),
        "Risk": "Breakout failure / unstable tape",
        "Response": "Trade smaller and widen selectivity.",
    })
    if not pd.isna(breadth_now):
        rows.append({
            "Tripwire": "Breadth breakdown",
            "Now": f"{breadth_now:.0%}",
            "Threshold": "<= 40% above 200DMA",
            "Status": classify(breadth_now <= 0.40, breadth_now <= 0.45),
            "Risk": "Index masking internal weakness",
            "Response": "Trust fewer longs and keep hedges active.",
        })
    rows.append({
        "Tripwire": "Signal deterioration cluster",
        "Now": str(deteriorating),
        "Threshold": ">= 6 signals worsening",
        "Status": classify(deteriorating >= 6, deteriorating >= 4),
        "Risk": "Regime quality eroding under the surface",
        "Response": "Stop adding exposure until drift stabilizes.",
    })
    rows.append({
        "Tripwire": "Deep drawdown",
        "Now": _fmt_pct(drawdown),
        "Threshold": "<= -10%",
        "Status": classify(drawdown <= -0.10, drawdown <= -0.07),
        "Risk": "Tape entering correction behavior",
        "Response": "Prefer defensive setups and shorter holding periods.",
    })
    rows.append({
        "Tripwire": "Composite collapse",
        "Now": f"{composite:+.1f}",
        "Threshold": "<= -4.0",
        "Status": classify(composite <= -4.0, composite <= -2.5),
        "Risk": "Broad bearish alignment",
        "Response": "Lean defensive and treat longs as tactical only.",
    })

    trip = pd.DataFrame(rows)
    severity_rank = {"TRIGGERED": 0, "NEAR": 1, "CLEAR": 2}
    trip["_rank"] = trip["Status"].map(severity_rank).fillna(9)
    trip = trip.sort_values(["_rank", "Tripwire"]).drop(columns=["_rank"]).reset_index(drop=True)
    trip.attrs["summary"] = {
        "Triggered": int((trip["Status"] == "TRIGGERED").sum()),
        "Near": int((trip["Status"] == "NEAR").sum()),
        "Clear": int((trip["Status"] == "CLEAR").sum()),
    }
    return trip


def fig_tripwire_monitor(state):
    trip = state.get("tripwire_monitor_df", pd.DataFrame())
    if trip is None or trip.empty:
        return fig_table(pd.DataFrame([{"Message": "No tripwire monitor available."}]), title="Tripwires", precision=2, max_rows=10)
    summary = trip.attrs.get("summary", {})
    title = f"Tripwires - {state['market_name']} | Triggered {summary.get('Triggered', 0)} | Near {summary.get('Near', 0)} | Clear {summary.get('Clear', 0)}"
    return fig_table(trip, title=title, precision=2, max_rows=15)



def build_concentration_risk(state):
    action_board = state.get("member_action_board_df", pd.DataFrame())
    leadership = state.get("leadership_map_df", pd.DataFrame())
    trade_book = state.get("trade_book_df", pd.DataFrame())
    if action_board is None or action_board.empty:
        return pd.DataFrame()

    board = action_board.copy()
    longs = int((board.get("Direction", pd.Series(dtype=str)).astype(str).str.upper() == "LONG").sum())
    shorts = int((board.get("Direction", pd.Series(dtype=str)).astype(str).str.upper() == "SHORT").sum())
    total = max(len(board), 1)
    sector_counts = board.groupby("Sector", dropna=False).size().sort_values(ascending=False)
    top_sector = sector_counts.index[0] if len(sector_counts) else "Unknown"
    top_sector_share = float(sector_counts.iloc[0] / total) if len(sector_counts) else 0.0
    top2_share = float(sector_counts.head(2).sum() / total) if len(sector_counts) else 0.0
    unique_sectors = int(board["Sector"].nunique()) if "Sector" in board.columns else 0

    if trade_book is not None and len(trade_book):
        tb = trade_book.copy()
        tb["BookWeight"] = pd.to_numeric(tb.get("BookWeight"), errors="coerce").fillna(0.0)
        gross_long = float(tb.loc[tb.get("Direction", pd.Series(dtype=str)).astype(str).str.upper() == "LONG", "BookWeight"].sum())
        gross_short = float(tb.loc[tb.get("Direction", pd.Series(dtype=str)).astype(str).str.upper() == "SHORT", "BookWeight"].sum())
        cash = float(tb.loc[tb.get("Direction", pd.Series(dtype=str)).astype(str).str.upper() == "NEUTRAL", "BookWeight"].sum())
    else:
        gross_long = gross_short = cash = 0.0

    if leadership is not None and len(leadership):
        leader_count = int((leadership.get("Stance", pd.Series(dtype=str)) == "Leader").sum())
    else:
        leader_count = 0

    rows = []
    def add_row(metric, value, threshold, status, why, response):
        rows.append({
            "Metric": metric,
            "Value": value,
            "Threshold": threshold,
            "Status": status,
            "Why": why,
            "Response": response,
        })

    sector_status = "HIGH" if top_sector_share >= 0.45 else "ELEVATED" if top_sector_share >= 0.30 else "OK"
    add_row(
        "Top sector share",
        f"{top_sector_share:.0%} ({top_sector})",
        "< 30% ideal",
        sector_status,
        "Too much concentration in one sector can turn the action board into a disguised macro bet.",
        "Prefer the action board's second-tier sectors before adding more to the leader.",
    )

    cluster_status = "HIGH" if top2_share >= 0.65 else "ELEVATED" if top2_share >= 0.50 else "OK"
    add_row(
        "Top 2 sector share",
        f"{top2_share:.0%}",
        "< 50% ideal",
        cluster_status,
        "A narrow opportunity set often means the trade book is less resilient if leadership rotates.",
        "Spread risk across more sectors or keep more cash if breadth is narrow.",
    )

    side_bias = abs(longs - shorts) / total
    side_status = "HIGH" if side_bias >= 0.60 else "ELEVATED" if side_bias >= 0.35 else "OK"
    add_row(
        "Directional skew",
        f"{longs} long / {shorts} short",
        "Balanced or intentional",
        side_status,
        "When almost everything leans one way, the notebook is really making a single macro call.",
        "Use the tripwires and regime radar to decide whether that skew is intentional or too aggressive.",
    )

    diversity_status = "HIGH" if unique_sectors <= 3 else "ELEVATED" if unique_sectors <= 5 else "OK"
    add_row(
        "Sector diversity",
        str(unique_sectors),
        ">= 6 sectors preferred",
        diversity_status,
        "A broader sector mix usually means the opportunity set is healthier and less fragile.",
        "If diversity is low, treat the book as tactical and avoid oversized adds.",
    )

    book_skew = gross_long - gross_short
    book_status = "HIGH" if abs(book_skew) >= 45 else "ELEVATED" if abs(book_skew) >= 25 else "OK"
    add_row(
        "Book net exposure",
        f"{book_skew:+.1f}%",
        "Within +/-25% unless conviction is high",
        book_status,
        "A heavily skewed book needs strong regime confirmation to be worth carrying.",
        "If regime radar is base-heavy, keep the net closer to flat or carry more cash.",
    )

    leader_status = "HIGH" if leader_count <= 1 else "ELEVATED" if leader_count <= 3 else "OK"
    add_row(
        "Leadership breadth",
        str(leader_count),
        ">= 4 leading sectors preferred",
        leader_status,
        "Few leading sectors means upside participation is still narrow under the surface.",
        "Treat breakouts with more skepticism until more sectors join the move.",
    )

    cash_status = "OK" if cash >= 10 else "ELEVATED" if cash >= 5 else "HIGH"
    add_row(
        "Cash buffer",
        f"{cash:.1f}%",
        ">= 10% when backdrop is mixed",
        cash_status,
        "Dry powder matters more when the regime is tactical and opportunities are clustered.",
        "Raise cash by trimming crowded winners instead of adding into already concentrated sectors.",
    )

    conc = pd.DataFrame(rows)
    severity_rank = {"HIGH": 0, "ELEVATED": 1, "OK": 2}
    conc["_rank"] = conc["Status"].map(severity_rank).fillna(9)
    conc = conc.sort_values(["_rank", "Metric"]).drop(columns=["_rank"]).reset_index(drop=True)
    conc.attrs["summary"] = {
        "High": int((conc["Status"] == "HIGH").sum()),
        "Elevated": int((conc["Status"] == "ELEVATED").sum()),
        "OK": int((conc["Status"] == "OK").sum()),
    }
    return conc


def fig_concentration_risk(state):
    conc = state.get("concentration_risk_df", pd.DataFrame())
    if conc is None or conc.empty:
        return fig_table(pd.DataFrame([{"Message": "No concentration risk view available."}]), title="Concentration Risk", precision=2, max_rows=10)
    summary = conc.attrs.get("summary", {})
    title = f"Concentration Risk - {state['market_name']} | High {summary.get('High', 0)} | Elevated {summary.get('Elevated', 0)} | OK {summary.get('OK', 0)}"
    return fig_table(conc, title=title, precision=2, max_rows=15)



def build_catalyst_grid(state):
    ideas = state.get("trade_ideas_df", pd.DataFrame())
    action_board = state.get("member_action_board_df", pd.DataFrame())
    regime = state.get("regime", {}) or {}
    if (ideas is None or ideas.empty) and (action_board is None or action_board.empty):
        return pd.DataFrame()

    rows = []

    def classify_catalyst(text, direction):
        text = str(text).lower()
        direction = str(direction).upper()
        if "reclaim" in text or "pullback" in text:
            return "Trend Repair"
        if "breakout" in text or "near highs" in text:
            return "Momentum Continuation"
        if "oversold" in text or "bounce" in text:
            return "Mean Reversion"
        if "short" in text or "weak" in text or "hedge" in text:
            return "Hedge / Protection"
        if direction == "SHORT":
            return "Hedge / Protection"
        return "Trend Continuation"

    if ideas is not None and len(ideas):
        for _, row in ideas.head(5).iterrows():
            catalyst = classify_catalyst(row.get("Setup", ""), row.get("Direction", "LONG"))
            rows.append({
                "Layer": "Index",
                "Catalyst": catalyst,
                "Instrument": row.get("Setup", "Index setup"),
                "Direction": row.get("Direction", "LONG"),
                "Conviction": round(float(row.get("Conviction", 0.0)), 1),
                "Action": row.get("Status", "WATCHLIST"),
                "WhyNow": row.get("Why It Works", row.get("Plan", "N/A")),
            })

    if action_board is not None and len(action_board):
        for _, row in action_board.head(10).iterrows():
            catalyst = classify_catalyst(row.get("Setup", ""), row.get("Direction", "LONG"))
            rows.append({
                "Layer": "Constituent",
                "Catalyst": catalyst,
                "Instrument": row.get("Ticker", row.get("Instrument", "Name")),
                "Direction": row.get("Direction", "LONG"),
                "Conviction": round(float(row.get("DiversifiedScore", row.get("Conviction", 0.0))), 1),
                "Action": row.get("ActionTier", "Watch"),
                "WhyNow": row.get("WhyNow", row.get("Reason", "N/A")),
            })

    grid = pd.DataFrame(rows)
    if grid.empty:
        return grid

    summary = grid.groupby(["Catalyst", "Direction"], dropna=False).agg(
        Names=("Instrument", "count"),
        AvgConviction=("Conviction", "mean"),
        BestIdea=("Instrument", "first"),
        PrimaryAction=("Action", "first"),
    ).reset_index()
    summary["AvgConviction"] = summary["AvgConviction"].round(1)
    summary = summary.sort_values(["AvgConviction", "Names"], ascending=[False, False]).reset_index(drop=True)
    summary.attrs["summary"] = {
        "Catalysts": int(summary["Catalyst"].nunique()),
        "Long buckets": int((summary["Direction"] == "LONG").sum()),
        "Short buckets": int((summary["Direction"] == "SHORT").sum()),
        "Top catalyst": summary.iloc[0]["Catalyst"] if len(summary) else "N/A",
    }
    return summary


def fig_catalyst_grid(state):
    grid = state.get("catalyst_grid_df", pd.DataFrame())
    if grid is None or grid.empty:
        return fig_table(pd.DataFrame([{"Message": "No catalyst grid available."}]), title="Catalyst Grid", precision=2, max_rows=10)
    summary = grid.attrs.get("summary", {})
    title = f"Catalyst Grid - {state['market_name']} | Catalysts {summary.get('Catalysts', 0)} | Long buckets {summary.get('Long buckets', 0)} | Short buckets {summary.get('Short buckets', 0)} | Top {summary.get('Top catalyst', 'N/A')}"
    return fig_table(grid, title=title, precision=2, max_rows=15)



def build_entry_planner(state):
    ideas = state.get("trade_ideas_df", pd.DataFrame())
    action_board = state.get("member_action_board_df", pd.DataFrame())
    if (ideas is None or ideas.empty) and (action_board is None or action_board.empty):
        return pd.DataFrame()

    rows = []

    def parse_num(text):
        return _parse_price_text(text) if '_parse_price_text' in globals() else np.nan

    def add_plan(layer, item, direction, conviction, entry_text, stop_text, target_text, action_text):
        entry = parse_num(entry_text)
        stop = parse_num(stop_text)
        target = parse_num(target_text)
        trigger = entry_text if isinstance(entry_text, str) and entry_text != 'N/A' else _fmt_price(entry)
        if pd.notna(entry) and pd.notna(stop):
            risk_gap = abs(entry - stop)
            if str(direction).upper() == 'LONG':
                pullback = _fmt_price(max(stop, entry - 0.33 * risk_gap))
                failure = _fmt_price(stop)
            else:
                pullback = _fmt_price(min(stop, entry + 0.33 * risk_gap))
                failure = _fmt_price(stop)
        else:
            pullback = 'N/A'
            failure = stop_text if isinstance(stop_text, str) else 'N/A'
        rows.append({
            'Layer': layer,
            'Item': item,
            'Direction': direction,
            'Conviction': round(float(conviction), 1),
            'Trigger Entry': trigger,
            'Pullback Entry': pullback,
            'Failure Level': failure,
            'Target': target_text if isinstance(target_text, str) else _fmt_price(target),
            'Execution': action_text,
        })

    if ideas is not None and len(ideas):
        for _, row in ideas.head(4).iterrows():
            action_text = 'Use breakout trigger' if 'Breakout' in str(row.get('Setup', '')) else 'Prefer pullback fill' if 'Pullback' in str(row.get('Setup', '')) or 'Reclaim' in str(row.get('Setup', '')) else 'Scale only if tape confirms'
            add_plan('Index', row.get('Setup', 'Index idea'), row.get('Direction', 'LONG'), row.get('Conviction', 0), row.get('Entry', 'N/A'), row.get('Stop', 'N/A'), row.get('Target 1', row.get('Target', 'N/A')), action_text)

    if action_board is not None and len(action_board):
        for _, row in action_board.head(6).iterrows():
            action_text = 'Best in sector' if str(row.get('ActionTier', '')) == 'Best in sector' else 'Add selectively' if 'Add' in str(row.get('ActionTier', '')) else 'Treat as hedge'
            add_plan('Constituent', row.get('Ticker', 'Name'), row.get('Direction', 'LONG'), row.get('DiversifiedScore', row.get('Conviction', 0)), row.get('Close', 'N/A'), row.get('Stop', 'N/A'), row.get('Target', 'N/A'), action_text)

    planner = pd.DataFrame(rows)
    planner = planner.sort_values(['Layer', 'Conviction'], ascending=[True, False]).reset_index(drop=True)
    planner.attrs['summary'] = {
        'Index plans': int((planner['Layer'] == 'Index').sum()) if len(planner) else 0,
        'Constituent plans': int((planner['Layer'] == 'Constituent').sum()) if len(planner) else 0,
    }
    return planner


def fig_entry_planner(state):
    planner = state.get('entry_planner_df', pd.DataFrame())
    if planner is None or planner.empty:
        return fig_table(pd.DataFrame([{'Message': 'No entry planner available.'}]), title='Entry Planner', precision=2, max_rows=10)
    summary = planner.attrs.get('summary', {})
    title = f"Entry Planner - {state['market_name']} | Index plans {summary.get('Index plans', 0)} | Constituent plans {summary.get('Constituent plans', 0)}"
    return fig_table(planner, title=title, precision=2, max_rows=15)



def build_risk_budget(state):
    regime = state.get('regime', {}) or {}
    radar = state.get('regime_radar_df', pd.DataFrame())
    trip = state.get('tripwire_monitor_df', pd.DataFrame())
    conc = state.get('concentration_risk_df', pd.DataFrame())
    book = state.get('trade_book_df', pd.DataFrame())

    bull = base = bear = 0.0
    if radar is not None and len(radar):
        probs = dict(zip(radar['Regime'], pd.to_numeric(radar['Probability'], errors='coerce').fillna(0.0)))
        bull = float(probs.get('Bull', 0.0))
        base = float(probs.get('Base', 0.0))
        bear = float(probs.get('Bear', 0.0))

    triggered = int((trip.get('Status', pd.Series(dtype=str)) == 'TRIGGERED').sum()) if trip is not None and len(trip) else 0
    near = int((trip.get('Status', pd.Series(dtype=str)) == 'NEAR').sum()) if trip is not None and len(trip) else 0
    high_conc = int((conc.get('Status', pd.Series(dtype=str)) == 'HIGH').sum()) if conc is not None and len(conc) else 0
    elevated_conc = int((conc.get('Status', pd.Series(dtype=str)) == 'ELEVATED').sum()) if conc is not None and len(conc) else 0
    composite = float(regime.get('composite', 0.0))

    gross_target = 95.0
    net_target = 0.0
    cash_target = 5.0

    gross_target += min(10.0, max(-15.0, composite * 2.0))
    gross_target += (bull - bear) * 0.15
    net_target += (bull - bear) * 0.45 + composite * 3.0

    gross_target -= triggered * 7.5 + near * 2.5
    gross_target -= high_conc * 4.0 + elevated_conc * 1.5
    net_target -= triggered * 5.0

    gross_target = max(35.0, min(110.0, gross_target))
    net_target = max(-40.0, min(70.0, net_target))
    cash_target = max(0.0, round(100.0 - min(gross_target, 100.0), 1))

    long_target = max(0.0, min(100.0, (gross_target + net_target) / 2.0))
    short_target = max(0.0, min(100.0, gross_target - long_target))

    current_long = current_short = current_cash = 0.0
    if book is not None and len(book):
        tb = book.copy()
        tb['BookWeight'] = pd.to_numeric(tb.get('BookWeight'), errors='coerce').fillna(0.0)
        dirs = tb.get('Direction', pd.Series(dtype=str)).astype(str).str.upper()
        current_long = float(tb.loc[dirs == 'LONG', 'BookWeight'].sum())
        current_short = float(tb.loc[dirs == 'SHORT', 'BookWeight'].sum())
        current_cash = float(tb.loc[dirs == 'NEUTRAL', 'BookWeight'].sum())

    def guidance(target, current, label):
        gap = round(target - current, 1)
        if abs(gap) < 5:
            return f'{label} close to target'
        if gap > 0:
            return f'Add about {gap:.1f}% to {label.lower()}'
        return f'Reduce about {abs(gap):.1f}% from {label.lower()}'

    budget = pd.DataFrame([
        {
            'Bucket': 'Gross exposure',
            'Target': f'{gross_target:.1f}%',
            'Current': f'{current_long + current_short:.1f}%',
            'Adjustment': guidance(gross_target, current_long + current_short, 'Gross exposure'),
            'Why': f'Composite {composite:+.1f}, bull {bull:.1f}%, bear {bear:.1f}%, triggered tripwires {triggered}.',
        },
        {
            'Bucket': 'Net long bias',
            'Target': f'{net_target:+.1f}%',
            'Current': f'{(current_long - current_short):+.1f}%',
            'Adjustment': guidance(net_target, current_long - current_short, 'Net long bias'),
            'Why': 'Align directional bias with regime radar while respecting tripwires.',
        },
        {
            'Bucket': 'Long sleeve',
            'Target': f'{long_target:.1f}%',
            'Current': f'{current_long:.1f}%',
            'Adjustment': guidance(long_target, current_long, 'Long sleeve'),
            'Why': 'Main risk-on sleeve built from the index core plus diversified longs.',
        },
        {
            'Bucket': 'Short / hedge sleeve',
            'Target': f'{short_target:.1f}%',
            'Current': f'{current_short:.1f}%',
            'Adjustment': guidance(short_target, current_short, 'Short / hedge sleeve'),
            'Why': 'Keeps downside protection proportional to regime deterioration and concentration risk.',
        },
        {
            'Bucket': 'Cash buffer',
            'Target': f'{cash_target:.1f}%',
            'Current': f'{current_cash:.1f}%',
            'Adjustment': guidance(cash_target, current_cash, 'Cash buffer'),
            'Why': 'Dry powder should rise when tripwires trigger or leadership breadth narrows.',
        },
    ])
    budget.attrs['summary'] = {
        'Bull': round(bull, 1),
        'Base': round(base, 1),
        'Bear': round(bear, 1),
        'Triggered': triggered,
    }
    return budget


def fig_risk_budget(state):
    budget = state.get('risk_budget_df', pd.DataFrame())
    if budget is None or budget.empty:
        return fig_table(pd.DataFrame([{'Message': 'No risk budget available.'}]), title='Risk Budget', precision=2, max_rows=10)
    summary = budget.attrs.get('summary', {})
    title = f"Risk Budget - {state['market_name']} | Bull {summary.get('Bull', 0):.1f}% | Base {summary.get('Base', 0):.1f}% | Bear {summary.get('Bear', 0):.1f}% | Tripwires {summary.get('Triggered', 0)}"
    return fig_table(budget, title=title, precision=2, max_rows=12)

def build_state(market_name: str, force_refresh: bool = False) -> Dict:
    cache_key = (market_name, CFG.lookback, "extended")
    if not force_refresh and cache_key in EXTENDED_STATE_CACHE:
        _emit_status(f"{market_name}: using cached dashboard state")
        return EXTENDED_STATE_CACHE[cache_key]

    base_state = _trade_engine_force_base_state(market_name, force_refresh=force_refresh)
    state = dict(base_state)

    _emit_status(f"{market_name}: building signal drift")
    state["signal_drift_df"] = build_signal_drift(state)
    _emit_status(f"{market_name}: building tactical playbook")
    state["trade_playbook_df"] = build_trade_playbook(state)
    _emit_status(f"{market_name}: building diversified action board")
    state["member_action_board_df"] = build_member_action_board(state)
    _emit_status(f"{market_name}: assembling catalyst grid")
    state["catalyst_grid_df"] = build_catalyst_grid(state)
    _emit_status(f"{market_name}: mapping sector leadership")
    state["leadership_map_df"] = build_leadership_map(state)
    _emit_status(f"{market_name}: building tripwire monitor")
    state["tripwire_monitor_df"] = build_tripwire_monitor(state)
    _emit_status(f"{market_name}: building regime radar")
    state["regime_radar_df"] = build_regime_radar(state)
    _emit_status(f"{market_name}: building scenario deck")
    state["scenario_deck_df"] = build_scenario_deck(state)
    _emit_status(f"{market_name}: building entry planner")
    state["entry_planner_df"] = build_entry_planner(state)
    _emit_status(f"{market_name}: building execution checklist")
    state["execution_checklist_df"] = build_execution_checklist(state)
    _emit_status(f"{market_name}: building position sizing")
    state["position_sizing_df"] = build_position_sizing(state)
    _emit_status(f"{market_name}: assembling trade book")
    state["trade_book_df"] = build_trade_book(state)
    _emit_status(f"{market_name}: building risk budget")
    state["risk_budget_df"] = build_risk_budget(state)
    _emit_status(f"{market_name}: checking concentration risk")
    state["concentration_risk_df"] = build_concentration_risk(state)

    state.setdefault("quality", {})
    state["quality"]["signal_drift_rows"] = int(len(state.get("signal_drift_df", pd.DataFrame())))
    state["quality"]["playbook_actions"] = int(len(state.get("trade_playbook_df", pd.DataFrame())))
    state["quality"]["action_board_rows"] = int(len(state.get("member_action_board_df", pd.DataFrame())))
    state["quality"]["catalyst_rows"] = int(len(state.get("catalyst_grid_df", pd.DataFrame())))
    state["quality"]["leadership_rows"] = int(len(state.get("leadership_map_df", pd.DataFrame())))
    state["quality"]["tripwire_rows"] = int(len(state.get("tripwire_monitor_df", pd.DataFrame())))
    state["quality"]["regime_radar_rows"] = int(len(state.get("regime_radar_df", pd.DataFrame())))
    state["quality"]["scenario_rows"] = int(len(state.get("scenario_deck_df", pd.DataFrame())))
    state["quality"]["entry_planner_rows"] = int(len(state.get("entry_planner_df", pd.DataFrame())))
    state["quality"]["execution_items"] = int(len(state.get("execution_checklist_df", pd.DataFrame())))
    state["quality"]["sizing_rows"] = int(len(state.get("position_sizing_df", pd.DataFrame())))
    state["quality"]["trade_book_rows"] = int(len(state.get("trade_book_df", pd.DataFrame())))
    state["quality"]["risk_budget_rows"] = int(len(state.get("risk_budget_df", pd.DataFrame())))
    state["quality"]["concentration_rows"] = int(len(state.get("concentration_risk_df", pd.DataFrame())))
    state["quality"]["positive_signal_drift"] = int((pd.to_numeric(state["signal_drift_df"].get("Delta5D"), errors="coerce") > 0.03).sum()) if len(state["signal_drift_df"]) else 0
    state["quality"]["negative_signal_drift"] = int((pd.to_numeric(state["signal_drift_df"].get("Delta5D"), errors="coerce") < -0.03).sum()) if len(state["signal_drift_df"]) else 0

    EXTENDED_STATE_CACHE[cache_key] = state
    _emit_status(f"{market_name}: ready")
    return state


## 13) Dashboard UI — Interactive Tabs with Condition Filters


In [ ]:
# ============================================================
# 14) DASHBOARD UI
# ============================================================
market_dd = widgets.Dropdown(
    options=list(CFG.indices.keys()),
    value=DEFAULT_MARKET,
    description="Market:",
    style={"description_width": "60px"},
)
direction_dd = widgets.Dropdown(
    options=["All", "LONG", "SHORT", "RISK"],
    value="All",
    description="Ideas:",
    style={"description_width": "45px"},
)
idea_limit_slider = widgets.IntSlider(
    value=min(6, CFG.idea_limit),
    min=3,
    max=12,
    step=1,
    description="Top N:",
    style={"description_width": "45px"},
    continuous_update=False,
)
refresh_btn = widgets.Button(description="Refresh Market", button_style="info")
status_html = widgets.HTML()
cond1_dd = widgets.Dropdown(
    options=[],
    description="Condition:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="470px"),
)
cond2_dd = widgets.Dropdown(
    options=["None"],
    value="None",
    description="AND:",
    style={"description_width": "40px"},
    layout=widgets.Layout(width="420px"),
)

out_exec = widgets.Output()
out_ideas = widgets.Output()
out_stock_ideas = widgets.Output()
out_action_board = widgets.Output()
out_catalyst = widgets.Output()
out_leadership = widgets.Output()
out_playbook = widgets.Output()
out_tripwires = widgets.Output()
out_concentration = widgets.Output()
out_radar = widgets.Output()
out_scenarios = widgets.Output()
out_drift = widgets.Output()
out_entry = widgets.Output()
out_execution = widgets.Output()
out_budget = widgets.Output()
out_sizing = widgets.Output()
out_tradebook = widgets.Output()
out_conviction = widgets.Output()
out_trend = widgets.Output()
out_risk = widgets.Output()
out_breadth = widgets.Output()
out_rrg = widgets.Output()
out_outlook = widgets.Output()
out_cross = widgets.Output()
out_constituents = widgets.Output()
out_quality = widgets.Output()
cross_refresh_btn = widgets.Button(description="Load Cross-Market Board", button_style="warning")
UI_SYNCING = False

tabs = widgets.Tab(
    children=[
        out_exec,
        out_ideas,
        out_stock_ideas,
        out_action_board,
        out_catalyst,
        out_leadership,
        out_playbook,
        out_tripwires,
        out_concentration,
        out_radar,
        out_scenarios,
        out_drift,
        out_entry,
        out_execution,
        out_budget,
        out_sizing,
        out_tradebook,
        out_conviction,
        out_trend,
        out_risk,
        out_breadth,
        out_rrg,
        out_outlook,
        out_cross,
        out_constituents,
        out_quality,
    ]
)
tab_names = [
    "Executive Summary",
    "Trade Ideas",
    "Constituent Ideas",
    "Action Board",
    "Catalyst Grid",
    "Leadership Map",
    "Playbook",
    "Tripwires",
    "Concentration Risk",
    "Regime Radar",
    "Scenario Deck",
    "Signal Drift",
    "Entry Planner",
    "Execution",
    "Risk Budget",
    "Sizing",
    "Trade Book",
    "Setup Conviction",
    "Trend",
    "Risk",
    "Breadth",
    "RRG",
    "Outlook",
    "Cross-Market",
    "Constituents",
    "Data Quality",
]
for idx, name in enumerate(tab_names):
    tabs.set_title(idx, name)


def update_status(message, tone="muted"):
    colors = {
        "muted": "#8b949e",
        "info": "#7ee787",
        "warn": "#ffea7f",
        "error": "#ff7b72",
    }
    status_html.value = (
        "<div style='padding:8px 12px;border:1px solid #21262d;border-radius:8px;"
        f"background:#0d1117;color:{colors.get(tone, colors['muted'])}'>"
        f"{message}</div>"
    )


def sync_condition_dropdowns(state):
    global UI_SYNCING
    UI_SYNCING = True
    cond_keys = list(state.get("conditions", {}).keys())
    if not cond_keys:
        cond1_dd.options = ["All days"]
        cond2_dd.options = ["None"]
        cond1_dd.value = "All days"
        cond2_dd.value = "None"
        UI_SYNCING = False
        return

    cond1_dd.options = cond_keys
    if cond1_dd.value not in cond_keys:
        cond1_dd.value = "All days" if "All days" in cond_keys else cond_keys[0]

    cond2_options = ["None"] + [name for name in cond_keys if name != cond1_dd.value]
    cond2_dd.options = cond2_options
    if cond2_dd.value not in cond2_options:
        cond2_dd.value = "None"
    UI_SYNCING = False


def render_outlook(state):
    update_status(f"Rendering <b>{state['market_name']}</b>: outlook analysis", "warn")
    with out_outlook:
        out_outlook.clear_output()
        display(
            HTML(
                "<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'>"
                "<b style='color:#7ee787;font-size:14px'>CONDITIONAL FORWARD OUTLOOK</b><br>"
                "<span style='color:#8b949e'>Filter history using one or two conditions, then compare the current setup against its historical forward-return profile.</span>"
                "</div>"
            )
        )
        display(widgets.HBox([cond1_dd, cond2_dd]))
        fig_fan, fig_table_outlook, fig_hist = fig_outlook(state, cond1_dd.value, cond2_dd.value)
        display_fig(fig_fan)
        display_fig(fig_hist)
        display_fig(fig_table_outlook)


def render_cross_market(force=False):
    with out_cross:
        out_cross.clear_output()
        display(
            HTML(
                "<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'>"
                "<b style='color:#7ee787;font-size:14px'>CROSS-MARKET OPPORTUNITY BOARD</b><br>"
                "<span style='color:#8b949e'>This panel ranks markets by current setup quality. It is loaded on demand because it refreshes multiple indices.</span>"
                "</div>"
            )
        )
        display(cross_refresh_btn)
        if force:
            display_fig(cross_market_snapshot())


def show_loading_state(message):
    placeholders = [
        out_exec, out_ideas, out_stock_ideas, out_action_board, out_catalyst, out_leadership, out_playbook, out_tripwires, out_concentration, out_radar, out_scenarios, out_drift, out_entry, out_execution, out_budget, out_sizing, out_tradebook, out_conviction, out_trend, out_risk,
        out_breadth, out_rrg, out_outlook, out_cross, out_constituents, out_quality,
    ]
    for out in placeholders:
        with out:
            out.clear_output()
            display(HTML(f"<div style='color:#8b949e;padding:24px'>{message}</div>"))


def render_all(state):
    sync_condition_dropdowns(state)
    best_setup = state.get("trade_ideas_df", pd.DataFrame())

    update_status(f"Rendering <b>{state['market_name']}</b>: executive summary", "warn")
    with out_exec:
        out_exec.clear_output()
        display(
            HTML(
                "<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'>"
                "<b style='color:#7ee787;font-size:14px'>EXECUTIVE SUMMARY</b><br>"
                "<span style='color:#8b949e'>Use this as the top-down read: market bias, tape condition, participation quality, and the current highest-conviction trade setup.</span>"
                "</div>"
            )
        )
        display_fig(fig_exec_summary(state))

    update_status(f"Rendering <b>{state['market_name']}</b>: trade ideas", "warn")
    with out_ideas:
        out_ideas.clear_output()
        display(
            HTML(
                "<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'>"
                "<b style='color:#7ee787;font-size:14px'>TRADE IDEA BOARD</b><br>"
                "<span style='color:#8b949e'>Setups are ranked using live signal alignment, historical edge for similar conditions, and reward-to-risk. This is the main decision surface for action.</span>"
                "</div>"
            )
        )
        display_fig(fig_trade_ideas(state, direction_dd.value, idea_limit_slider.value))

    update_status(f"Rendering <b>{state['market_name']}</b>: constituent ideas", "warn")
    with out_stock_ideas:
        out_stock_ideas.clear_output()
        display(
            HTML(
                "<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'>"
                "<b style='color:#7ee787;font-size:14px'>CONSTITUENT IDEAS</b><br>"
                "<span style='color:#8b949e'>Top stock-level opportunities from the selected market's constituent or proxy universe.</span>"
                "</div>"
            )
        )
        display_fig(fig_member_trade_ideas(state))

    update_status(f"Rendering <b>{state['market_name']}</b>: diversified action board", "warn")
    with out_action_board:
        out_action_board.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>ACTION BOARD</b><br><span style='color:#8b949e'>A diversification-aware short list that avoids over-clustering in a single sector and promotes a cleaner actionable lineup.</span></div>"))
        display_fig(fig_member_action_board(state))

    update_status(f"Rendering <b>{state['market_name']}</b>: catalyst grid", "warn")
    with out_catalyst:
        out_catalyst.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>CATALYST GRID</b><br><span style='color:#8b949e'>Groups index and constituent ideas by what is actually driving them now: trend repair, continuation, mean reversion, or hedge demand.</span></div>"))
        display_fig(fig_catalyst_grid(state))

    update_status(f"Rendering <b>{state['market_name']}</b>: leadership map", "warn")
    with out_leadership:
        out_leadership.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>LEADERSHIP MAP</b><br><span style='color:#8b949e'>Shows which sectors are actually leading, supporting, or lagging based on diversified constituent opportunities.</span></div>"))
        display_fig(fig_leadership_map(state))

    update_status(f"Rendering <b>{state['market_name']}</b>: tactical playbook", "warn")
    with out_action_board:
        out_action_board.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>ACTION BOARD</b><br><span style='color:#8b949e'>Diversification-aware short list across sectors and directions.</span></div>"))
        display_fig(fig_member_action_board(state))
    with out_catalyst:
        out_catalyst.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>CATALYST GRID</b><br><span style='color:#8b949e'>Groups ideas by what is actually driving them now.</span></div>"))
        display_fig(fig_catalyst_grid(state))
    with out_leadership:
        out_leadership.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>LEADERSHIP MAP</b><br><span style='color:#8b949e'>Sector-level leadership derived from the diversified action board.</span></div>"))
        display_fig(fig_leadership_map(state))
    with out_playbook:
        out_playbook.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>TACTICAL PLAYBOOK</b><br><span style='color:#8b949e'>A concise top-down action layer combining index setup, risk posture, breadth context, and best constituent idea.</span></div>"))
        display_fig(fig_trade_playbook(state))

    update_status(f"Rendering <b>{state['market_name']}</b>: tripwires", "warn")
    with out_tripwires:
        out_tripwires.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>TRIPWIRES</b><br><span style='color:#8b949e'>Explicit invalidation levels and response rules so the current posture has clear failure conditions.</span></div>"))
        display_fig(fig_tripwire_monitor(state))

    update_status(f"Rendering <b>{state['market_name']}</b>: concentration risk", "warn")
    with out_concentration:
        out_concentration.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>CONCENTRATION RISK</b><br><span style='color:#8b949e'>Flags when the opportunity set or trade book is leaning too hard on one side, one sector cluster, or too little cash.</span></div>"))
        display_fig(fig_concentration_risk(state))

    update_status(f"Rendering <b>{state['market_name']}</b>: regime radar", "warn")
    with out_tripwires:
        out_tripwires.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>TRIPWIRES</b><br><span style='color:#8b949e'>Explicit invalidation levels and response rules.</span></div>"))
        display_fig(fig_tripwire_monitor(state))
    with out_concentration:
        out_concentration.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>CONCENTRATION RISK</b><br><span style='color:#8b949e'>Flags when the opportunity set or trade book is leaning too hard on one side, one sector cluster, or too little cash.</span></div>"))
        display_fig(fig_concentration_risk(state))
    with out_radar:
        out_radar.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>REGIME RADAR</b><br><span style='color:#8b949e'>Turns the signal stack into explicit bull, base, and bear odds with a recommended posture.</span></div>"))
        display_fig(fig_regime_radar(state))

    update_status(f"Rendering <b>{state['market_name']}</b>: scenario deck", "warn")
    with out_tripwires:
        out_tripwires.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>TRIPWIRES</b><br><span style='color:#8b949e'>Explicit invalidation levels and response rules.</span></div>"))
        display_fig(fig_tripwire_monitor(state))
    with out_concentration:
        out_concentration.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>CONCENTRATION RISK</b><br><span style='color:#8b949e'>Flags when the opportunity set or trade book is leaning too hard on one side, one sector cluster, or too little cash.</span></div>"))
        display_fig(fig_concentration_risk(state))
    with out_radar:
        out_radar.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>REGIME RADAR</b><br><span style='color:#8b949e'>Explicit bull, base, and bear odds with a recommended posture.</span></div>"))
        display_fig(fig_regime_radar(state))
    with out_scenarios:
        out_scenarios.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>SCENARIO DECK</b><br><span style='color:#8b949e'>Bull, base, and bear paths from here with trigger levels, posture changes, and suggested expressions.</span></div>"))
        display_fig(fig_scenario_deck(state))

    update_status(f"Rendering <b>{state['market_name']}</b>: signal drift", "warn")
    with out_tripwires:
        out_tripwires.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>TRIPWIRES</b><br><span style='color:#8b949e'>Explicit invalidation levels and response rules.</span></div>"))
        display_fig(fig_tripwire_monitor(state))
    with out_concentration:
        out_concentration.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>CONCENTRATION RISK</b><br><span style='color:#8b949e'>Flags when the opportunity set or trade book is leaning too hard on one side, one sector cluster, or too little cash.</span></div>"))
        display_fig(fig_concentration_risk(state))
    with out_radar:
        out_radar.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>REGIME RADAR</b><br><span style='color:#8b949e'>Explicit bull, base, and bear odds with a recommended posture.</span></div>"))
        display_fig(fig_regime_radar(state))
    with out_scenarios:
        out_scenarios.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>SCENARIO DECK</b><br><span style='color:#8b949e'>Bull, base, and bear paths from here with trigger levels and posture changes.</span></div>"))
        display_fig(fig_scenario_deck(state))
    with out_drift:
        out_drift.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>SIGNAL DRIFT</b><br><span style='color:#8b949e'>Shows which signals are improving or deteriorating versus recent checkpoints, so regime transitions become visible earlier.</span></div>"))
        display_fig(fig_signal_drift(state))

    update_status(f"Rendering <b>{state['market_name']}</b>: entry planner", "warn")
    with out_entry:
        out_entry.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>ENTRY PLANNER</b><br><span style='color:#8b949e'>Turns the strongest setups into trigger, pullback, and failure-entry levels so execution is less guessy.</span></div>"))
        display_fig(fig_entry_planner(state))

    update_status(f"Rendering <b>{state['market_name']}</b>: execution checklist", "warn")
    with out_execution:
        out_execution.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>EXECUTION CHECKLIST</b><br><span style='color:#8b949e'>Turns the highest-conviction setups into concrete next actions for the index and the most actionable constituents.</span></div>"))
        display_fig(fig_execution_checklist(state))

    update_status(f"Rendering <b>{state['market_name']}</b>: risk budget", "warn")
    with out_budget:
        out_budget.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>RISK BUDGET</b><br><span style='color:#8b949e'>Translates the regime, tripwires, and concentration into target gross, net, hedge, and cash budgets.</span></div>"))
        display_fig(fig_risk_budget(state))

    update_status(f"Rendering <b>{state['market_name']}</b>: position sizing", "warn")
    with out_sizing:
        out_sizing.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>POSITION SIZING</b><br><span style='color:#8b949e'>Translates conviction, stop distance, and signal drift into starter size and execution posture.</span></div>"))
        display_fig(fig_position_sizing(state))

    update_status(f"Rendering <b>{state['market_name']}</b>: trade book", "warn")
    with out_tradebook:
        out_tradebook.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>TRADE BOOK</b><br><span style='color:#8b949e'>Translates the idea engine into a suggested basket with core index exposure, best longs, hedges, and cash buffer.</span></div>"))
        display_fig(fig_trade_book(state))

    update_status(f"Rendering <b>{state['market_name']}</b>: conviction chart", "warn")
    with out_action_board:
        out_action_board.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>ACTION BOARD</b><br><span style='color:#8b949e'>Diversification-aware short list across sectors and directions.</span></div>"))
        display_fig(fig_member_action_board(state))
    with out_catalyst:
        out_catalyst.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>CATALYST GRID</b><br><span style='color:#8b949e'>Groups ideas by what is actually driving them now.</span></div>"))
        display_fig(fig_catalyst_grid(state))
    with out_leadership:
        out_leadership.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>LEADERSHIP MAP</b><br><span style='color:#8b949e'>Sector-level leadership derived from the diversified action board.</span></div>"))
        display_fig(fig_leadership_map(state))
    with out_playbook:
        out_playbook.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>TACTICAL PLAYBOOK</b><br><span style='color:#8b949e'>A concise top-down action layer combining index setup, risk posture, breadth context, and best constituent idea.</span></div>"))
        display_fig(fig_trade_playbook(state))
    with out_tripwires:
        out_tripwires.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>TRIPWIRES</b><br><span style='color:#8b949e'>Explicit invalidation levels and response rules.</span></div>"))
        display_fig(fig_tripwire_monitor(state))
    with out_concentration:
        out_concentration.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>CONCENTRATION RISK</b><br><span style='color:#8b949e'>Flags when the opportunity set or trade book is leaning too hard on one side, one sector cluster, or too little cash.</span></div>"))
        display_fig(fig_concentration_risk(state))
    with out_radar:
        out_radar.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>REGIME RADAR</b><br><span style='color:#8b949e'>Explicit bull, base, and bear odds with a recommended posture.</span></div>"))
        display_fig(fig_regime_radar(state))
    with out_scenarios:
        out_scenarios.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>SCENARIO DECK</b><br><span style='color:#8b949e'>Bull, base, and bear paths from here with trigger levels and posture changes.</span></div>"))
        display_fig(fig_scenario_deck(state))
    with out_drift:
        out_drift.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>SIGNAL DRIFT</b><br><span style='color:#8b949e'>Shows which signals are improving or deteriorating versus recent checkpoints.</span></div>"))
        display_fig(fig_signal_drift(state))
    with out_entry:
        out_entry.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>ENTRY PLANNER</b><br><span style='color:#8b949e'>Trigger, pullback, and failure-entry levels for the strongest setups.</span></div>"))
        display_fig(fig_entry_planner(state))
    with out_execution:
        out_execution.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>EXECUTION CHECKLIST</b><br><span style='color:#8b949e'>Concrete next actions for the index and top constituents.</span></div>"))
        display_fig(fig_execution_checklist(state))
    with out_budget:
        out_budget.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>RISK BUDGET</b><br><span style='color:#8b949e'>Target gross, net, hedge, and cash budgets from the current regime.</span></div>"))
        display_fig(fig_risk_budget(state))
    with out_sizing:
        out_sizing.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>POSITION SIZING</b><br><span style='color:#8b949e'>Starter size and execution posture from conviction, stop distance, and drift.</span></div>"))
        display_fig(fig_position_sizing(state))
    with out_tradebook:
        out_tradebook.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>TRADE BOOK</b><br><span style='color:#8b949e'>Suggested basket with core index exposure, best longs, hedges, and cash buffer.</span></div>"))
        display_fig(fig_trade_book(state))
    with out_conviction:
        out_conviction.clear_output()
        display(
            HTML(
                "<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'>"
                "<b style='color:#7ee787;font-size:14px'>SETUP CONVICTION</b><br>"
                "<span style='color:#8b949e'>Higher scores reflect stronger signal alignment, better historical edge, more sample support, and cleaner reward-to-risk.</span>"
                "</div>"
            )
        )
        display_fig(fig_trade_conviction(state, direction_dd.value, idea_limit_slider.value))

    update_status(f"Rendering <b>{state['market_name']}</b>: trend charts", "warn")
    with out_trend:
        out_trend.clear_output()
        f1, f2, f3 = fig_trend_signals(state)
        display_fig(f1)
        display_fig(f2)
        display_fig(f3)

    update_status(f"Rendering <b>{state['market_name']}</b>: risk and volatility", "warn")
    with out_risk:
        out_risk.clear_output()
        f1, f2 = fig_risk_signals(state)
        display_fig(f1)
        display_fig(f2)

    update_status(f"Rendering <b>{state['market_name']}</b>: breadth internals", "warn")
    with out_breadth:
        out_breadth.clear_output()
        if state.get("members_feat"):
            f1, f2, f3 = fig_breadth(state)
            display_fig(f1)
            if f2 is not None:
                display_fig(f2)
            display_fig(f3)
            tm = fig_thrust_mcclellan(state)
            if tm is not None:
                display_fig(tm)
        else:
            display(HTML("<div style='color:#8b949e;padding:20px'>Breadth internals are not available for the selected market.</div>"))

    update_status(f"Rendering <b>{state['market_name']}</b>: RRG / relative rotation", "warn")
    with out_rrg:
        out_rrg.clear_output()
        r1, r2 = fig_rrg(state) if state["market_name"] == "S&P 500" else (None, None)
        if r1 is None:
            display(HTML("<div style='color:#8b949e;padding:20px'>RRG is available for S&P 500 sector rotation.</div>"))
        else:
            display_fig(r1)
            display_fig(r2)

    render_outlook(state)

    update_status(f"Rendering <b>{state['market_name']}</b>: cross-market panel", "warn")
    render_cross_market(force=False)

    update_status(f"Rendering <b>{state['market_name']}</b>: constituents table", "warn")
    with out_constituents:
        out_constituents.clear_output()
        display(
            HTML(
                "<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'>"
                "<b style='color:#7ee787;font-size:14px'>CONSTITUENTS / HOLDINGS</b><br>"
                "<span style='color:#8b949e'>This tab shows either exact index members or a clearly labeled proxy holdings list when a full free public constituent list is not practical.</span>"
                "</div>"
            )
        )
        display_fig(fig_constituents(state))

    update_status(f"Rendering <b>{state['market_name']}</b>: diagnostics", "warn")
    with out_quality:
        out_quality.clear_output()
        display_fig(fig_quality(state))

    best_setup_name = best_setup.iloc[0]["Setup"] if not best_setup.empty else "None"
    update_status(
        f"{state['market_name']} ready. Best current setup: <b>{best_setup_name}</b>.",
        "info",
    )


def render_trade_views(state):
    update_status(f"Refreshing <b>{state['market_name']}</b>: trade idea views", "warn")
    with out_ideas:
        out_ideas.clear_output()
        display(
            HTML(
                "<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'>"
                "<b style='color:#7ee787;font-size:14px'>TRADE IDEA BOARD</b><br>"
                "<span style='color:#8b949e'>Setups are ranked using live signal alignment, historical edge for similar conditions, and reward-to-risk. This is the main decision surface for action.</span>"
                "</div>"
            )
        )
        display_fig(fig_trade_ideas(state, direction_dd.value, idea_limit_slider.value))
    with out_stock_ideas:
        out_stock_ideas.clear_output()
        display(
            HTML(
                "<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'>"
                "<b style='color:#7ee787;font-size:14px'>CONSTITUENT IDEAS</b><br>"
                "<span style='color:#8b949e'>Top stock-level opportunities from the selected market's constituent or proxy universe.</span>"
                "</div>"
            )
        )
        display_fig(fig_member_trade_ideas(state))
    with out_action_board:
        out_action_board.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>ACTION BOARD</b><br><span style='color:#8b949e'>Diversification-aware short list across sectors and directions.</span></div>"))
        display_fig(fig_member_action_board(state))
    with out_catalyst:
        out_catalyst.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>CATALYST GRID</b><br><span style='color:#8b949e'>Groups ideas by what is actually driving them now.</span></div>"))
        display_fig(fig_catalyst_grid(state))
    with out_leadership:
        out_leadership.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>LEADERSHIP MAP</b><br><span style='color:#8b949e'>Sector-level leadership derived from the diversified action board.</span></div>"))
        display_fig(fig_leadership_map(state))
    with out_playbook:
        out_playbook.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>TACTICAL PLAYBOOK</b><br><span style='color:#8b949e'>A concise top-down action layer combining index setup, risk posture, breadth context, and best constituent idea.</span></div>"))
        display_fig(fig_trade_playbook(state))
    with out_tripwires:
        out_tripwires.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>TRIPWIRES</b><br><span style='color:#8b949e'>Explicit invalidation levels and response rules.</span></div>"))
        display_fig(fig_tripwire_monitor(state))
    with out_concentration:
        out_concentration.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>CONCENTRATION RISK</b><br><span style='color:#8b949e'>Flags when the opportunity set or trade book is leaning too hard on one side, one sector cluster, or too little cash.</span></div>"))
        display_fig(fig_concentration_risk(state))
    with out_radar:
        out_radar.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>REGIME RADAR</b><br><span style='color:#8b949e'>Explicit bull, base, and bear odds with a recommended posture.</span></div>"))
        display_fig(fig_regime_radar(state))
    with out_scenarios:
        out_scenarios.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>SCENARIO DECK</b><br><span style='color:#8b949e'>Bull, base, and bear paths from here with trigger levels and posture changes.</span></div>"))
        display_fig(fig_scenario_deck(state))
    with out_drift:
        out_drift.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>SIGNAL DRIFT</b><br><span style='color:#8b949e'>Shows which signals are improving or deteriorating versus recent checkpoints.</span></div>"))
        display_fig(fig_signal_drift(state))
    with out_entry:
        out_entry.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>ENTRY PLANNER</b><br><span style='color:#8b949e'>Trigger, pullback, and failure-entry levels for the strongest setups.</span></div>"))
        display_fig(fig_entry_planner(state))
    with out_execution:
        out_execution.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>EXECUTION CHECKLIST</b><br><span style='color:#8b949e'>Concrete next actions for the index and top constituents.</span></div>"))
        display_fig(fig_execution_checklist(state))
    with out_budget:
        out_budget.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>RISK BUDGET</b><br><span style='color:#8b949e'>Target gross, net, hedge, and cash budgets from the current regime.</span></div>"))
        display_fig(fig_risk_budget(state))
    with out_sizing:
        out_sizing.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>POSITION SIZING</b><br><span style='color:#8b949e'>Starter size and execution posture from conviction, stop distance, and drift.</span></div>"))
        display_fig(fig_position_sizing(state))
    with out_tradebook:
        out_tradebook.clear_output()
        display(HTML("<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'><b style='color:#7ee787;font-size:14px'>TRADE BOOK</b><br><span style='color:#8b949e'>Suggested basket with core index exposure, best longs, hedges, and cash buffer.</span></div>"))
        display_fig(fig_trade_book(state))
    with out_conviction:
        out_conviction.clear_output()
        display(
            HTML(
                "<div style='background:#0d1117;padding:12px;border:1px solid #21262d;border-radius:8px;margin-bottom:10px'>"
                "<b style='color:#7ee787;font-size:14px'>SETUP CONVICTION</b><br>"
                "<span style='color:#8b949e'>Higher scores reflect stronger signal alignment, better historical edge, more sample support, and cleaner reward-to-risk.</span>"
                "</div>"
            )
        )
        display_fig(fig_trade_conviction(state, direction_dd.value, idea_limit_slider.value))
    best_setup_name = state['trade_ideas_df'].iloc[0]['Setup'] if len(state.get('trade_ideas_df', [])) else 'None'
    update_status(f"{state['market_name']} ready. Best current setup: <b>{best_setup_name}</b>.", "info")


def load_market(force_refresh=False):
    global state, DASHBOARD_STATUS_HOOK
    market_name = market_dd.value
    note = " First run may take a while for full member-universe data." if market_name in ("S&P 500", "NASDAQ 100") else ""
    update_status(f"Loading <b>{market_name}</b>...{note}", "warn")
    show_loading_state(f"Loading {market_name}... this may take longer on the first run while data is downloaded and cached.")
    DASHBOARD_STATUS_HOOK = update_status
    try:
        state = build_state(market_name, force_refresh=force_refresh)
        render_all(state)
    except Exception as exc:
        update_status(f"Load failed: {exc}", "error")
        raise


def on_market_change(change):
    if change["name"] == "value":
        load_market(force_refresh=False)


def on_control_change(change):
    if change["name"] == "value":
        render_trade_views(state)


def on_condition_change(change):
    if UI_SYNCING:
        return
    if change["name"] == "value":
        if cond1_dd.value == cond2_dd.value and cond2_dd.value != "None":
            cond2_dd.value = "None"
        render_outlook(state)


def on_refresh_click(_):
    load_market(force_refresh=True)


def on_cross_refresh_click(_):
    render_cross_market(force=True)


market_dd.observe(on_market_change, names="value")
direction_dd.observe(on_control_change, names="value")
idea_limit_slider.observe(on_control_change, names="value")
cond1_dd.observe(on_condition_change, names="value")
cond2_dd.observe(on_condition_change, names="value")
refresh_btn.on_click(on_refresh_click)
cross_refresh_btn.on_click(on_cross_refresh_click)

display(
    HTML(
        "<h2 style='color:#7ee787;margin-bottom:4px'>Global Market Signal Dashboard v25</h2>"
        "<p style='color:#8b949e;margin-top:0'>A notebook-native decision engine for market regime analysis, conditional edges, ranked index setups, tactical playbooks, signal drift, execution guidance, sizing, trade-book construction, diversification-aware action boards, scenario planning, regime odds, tripwires, concentration control, catalysts, sector leadership, entry planning, risk budgeting, and constituent-level ideas.</p>"
    )
)
display(widgets.HBox([market_dd, direction_dd, idea_limit_slider, refresh_btn]))
display(status_html)
display(tabs)

load_market(force_refresh=False)


HTML(value='')

S&P 500: assembling dashboard state
S&P 500: loading index history
S&P 500: computing index features
S&P 500: loading constituents / holdings list
S&P 500: selecting constituent subset for breadth
S&P 500: preparing member universe tickers for breadth
S&P 500: downloading / loading member universe prices
S&P 500: breadth step 1/6 - percent above 50DMA and 200DMA
S&P 500: breadth step 2/6 - 20-day breadth momentum
S&P 500: breadth step 3/6 - percent near 52-week highs and lows
S&P 500: breadth step 4/6 - dispersion and rolling dispersion
S&P 500: breadth step 5/6 - average cross-sectional correlation
S&P 500: breadth step 6/6 - percentile context for internals
S&P 500: thrust step 1/4 - advance / decline counts
S&P 500: thrust step 2/4 - breadth thrust EMA
S&P 500: thrust step 3/4 - McClellan oscillator
S&P 500: thrust step 4/4 - AD line and percentile context
S&P 500: RRG step 1/5 - aligning benchmark series
S&P 500: RRG step 2/5 - grouping members into sectors
S&P 500: RRG step 3/5 - 

HTTP Error 404: 
HTTP Error 404: 


---
## Extending This Dashboard

**Add new markets**: Add tickers to `CFG.indices` dict (any Yahoo Finance ticker works).

**Add member universes**: For any index where you have a CSV of constituents, plug into the universe pipeline:
1. Load tickers from CSV
2. Call `load_or_download_universe_close(...)`
3. Compute breadth/dispersion to unlock member-based conditions and breadth signals

**Add new signals**: Add to `compute_signal_scores()` with a weight in `CFG.score_weights`.

**Add new conditions**: Add to `build_conditions()` — any boolean Series indexed on the same dates.

**Customize thresholds**: All thresholds are in `CFG` — tune for your market/strategy.
